In [ ]:
# =============================================================================
# CELL 2b — DATA INGESTION: UPLOAD AND EXTRACT ZIP
# =============================================================================
# EXPLANATION:
# This cell handles the upload of one ZIP file into the Bronze layer.
# It saves the uploaded ZIP to one stable project path, clears any previous
# extracted image folder, extracts the ZIP, discovers image files, and builds
# a simple images_df manifest for later SAM3 inference.
# =============================================================================

from google.colab import files
import os
import shutil
import zipfile
import pandas as pd

# -----------------------------------------------------------------------------
# 0. Pipeline state validation
# -----------------------------------------------------------------------------
# Ensures that Cell 2 was executed successfully before attempting to use its variables.
if "BRONZE_INPUT_ZIPS" not in globals():
    raise NameError("BRONZE_INPUT_ZIPS is not defined. Please run Cell 2 first.")

if "XARM_EXTRACT_DIR" not in globals():
    raise NameError("XARM_EXTRACT_DIR is not defined. Please run Cell 2 first.")

if "IMAGE_EXTS" not in globals():
    raise NameError("IMAGE_EXTS is not defined. Please run Cell 2 first.")

if "LOCAL_ZIP_HINT" not in globals():
    LOCAL_ZIP_HINT = r"C:\Users\shary\Documents\Defect_Project\Xarm.zip"

if "PROJECT_ZIP_NAME" not in globals():
    PROJECT_ZIP_NAME = "uploaded_dataset.zip"

if "PROJECT_ZIP_PATH" not in globals():
    PROJECT_ZIP_PATH = os.path.join(BRONZE_INPUT_ZIPS, PROJECT_ZIP_NAME)

# -----------------------------------------------------------------------------
# 1. Upload invocation
# -----------------------------------------------------------------------------
# Triggers the interactive Colab upload widget.
print("Please choose any ZIP file from your computer.")
print(f"Local path hint: {LOCAL_ZIP_HINT}")

uploaded = files.upload()

# -----------------------------------------------------------------------------
# 2. Upload validation
# -----------------------------------------------------------------------------
# Enforces strict ingestion rules: exactly one file must be uploaded,
# and it must be a valid ZIP archive.
uploaded_names = list(uploaded.keys())

if len(uploaded_names) == 0:
    raise FileNotFoundError("No file was uploaded.")

if len(uploaded_names) > 1:
    raise ValueError(
        f"Please upload only ONE ZIP file.\n"
        f"Uploaded files were: {uploaded_names}"
    )

uploaded_name = uploaded_names[0]

if not uploaded_name.lower().endswith(".zip"):
    raise ValueError(f"Uploaded file is not a ZIP file: {uploaded_name}")

print(f"Uploaded ZIP detected: {uploaded_name}")

# -----------------------------------------------------------------------------
# 3. Save to one stable Bronze-layer ZIP path
# -----------------------------------------------------------------------------
# Writes the raw byte stream to a standardized path, preventing duplicate copies.
with open(PROJECT_ZIP_PATH, "wb") as f:
    f.write(uploaded[uploaded_name])

print(f"Saved project ZIP to: {PROJECT_ZIP_PATH}")

UPLOADED_ZIP_DISPLAY_NAME = uploaded_name

# -----------------------------------------------------------------------------
# 4. Validate ZIP archive before extraction
# -----------------------------------------------------------------------------
# Double checks file integrity to avoid crashing during extraction.
if not zipfile.is_zipfile(PROJECT_ZIP_PATH):
    raise ValueError(f"Not a valid ZIP archive: {PROJECT_ZIP_PATH}")

# -----------------------------------------------------------------------------
# 5. Clear previous extracted folder, then extract fresh
# -----------------------------------------------------------------------------
# Wipes the destination folder first to ensure no "ghost files" from previous runs.
if os.path.isdir(XARM_EXTRACT_DIR):
    shutil.rmtree(XARM_EXTRACT_DIR)

os.makedirs(XARM_EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(PROJECT_ZIP_PATH, "r") as zf:
    zf.extractall(XARM_EXTRACT_DIR)

# -----------------------------------------------------------------------------
# 6. Recursively discover extracted image files
# -----------------------------------------------------------------------------
# Scans the extracted folder tree for supported image extensions.
image_files = []

for root, _, dir_files in os.walk(XARM_EXTRACT_DIR):
    for fn in dir_files:
        if fn.endswith(IMAGE_EXTS):
            image_files.append(os.path.join(root, fn))

image_files = sorted(image_files)

print(f"Extracted image count: {len(image_files)}")

# Fails gracefully if the zip contained no valid images.
if len(image_files) == 0:
    raise ValueError(
        "No image files were found after extraction.\n"
        "Check whether the ZIP contains images in nested folders or unsupported formats."
    )

print("\nFirst few image paths:")
for fp in image_files[:10]:
    print(f"  - {fp}")

# -----------------------------------------------------------------------------
# 7. Build image manifest
# -----------------------------------------------------------------------------
# Constructs a pandas DataFrame to act as the primary tracking manifest
# for all images moving through the Silver and Gold pipeline phases.
images_df = pd.DataFrame({
    "image_path": image_files,
})

images_df["file_name"] = images_df["image_path"].map(os.path.basename)
images_df["stem"] = images_df["file_name"].map(lambda x: os.path.splitext(x)[0])
images_df["ext"] = images_df["file_name"].map(lambda x: os.path.splitext(x)[1])

print(f"\nPROJECT_ZIP_PATH : {PROJECT_ZIP_PATH}")
print(f"Uploaded ZIP name: {UPLOADED_ZIP_DISPLAY_NAME}")
print(f"images_df shape  : {images_df.shape}")

# Safe display handler in case IPython isn't loaded correctly
try:
    display(images_df.head())
except NameError:
    print(images_df.head())


In [ ]:
# =============================================================================
# CELL 1 — BASE ENV (RUN ONCE) then HARD RESTART
# - Fix NumPy/OpenCV ABI in Colab
# - Fix Pillow to avoid: ImportError: cannot import name '_Ink' from PIL._typing
# =============================================================================
!pip -q uninstall -y opencv-python opencv-python-headless numpy pillow
!pip -q install "numpy==1.26.4"
!pip -q install "opencv-python-headless==4.9.0.80"
!pip -q install -U transformers huggingface_hub safetensors accelerate "pillow<12" matplotlib
!pip -q install -U pyarrow
!pip -q install scikit-learn seaborn tqdm

import os
os.kill(os.getpid(), 9)

In [ ]:
# =============================================================================
# CELL 2 — IMPORTS + WORKSPACE
# =============================================================================
# EXPLANATION:
# This cell centralizes imports, defines the project workspace, creates the
# folder structure for the SAM3 crossarm pipeline, and initializes global
# constants used across the notebook.
#
# PIPELINE ALIGNMENT:
#   1) Upload and extract ZIP file to Bronze layer
#   2) Run SAM3 with multiple text prompts for crossarm detection
#   3) Save masks, crops, and prompt-run outputs to Silver layer
#   4) Export debug artifacts and visual summaries to Gold layer
# =============================================================================

# -----------------------------------------------------------------------------
# Standard library
# -----------------------------------------------------------------------------
# Core Python utilities for file system management (os, glob), text parsing (re),
# JSON handling, timing, and reproducibility.
import os
import gc
import re
import io
import json
import time
import glob
import math
import shutil
import random
import zipfile
import warnings
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

# -----------------------------------------------------------------------------
# Data science and visualization
# -----------------------------------------------------------------------------
# Essential libraries for array manipulation, tabular data, image processing,
# and plotting.
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import PIL
from PIL import Image, ImageOps

# -----------------------------------------------------------------------------
# Machine learning / utilities
# -----------------------------------------------------------------------------
# PyTorch for running the SAM3 model.
import torch

# -----------------------------------------------------------------------------
# Global warning behavior
# -----------------------------------------------------------------------------
# Keeps the notebook output clean from non-critical library warnings.
warnings.filterwarnings("ignore")

# =============================================================================
# WORKSPACE ROOT
# =============================================================================
# The top-level directory for the entire pipeline run.
WORK_DIR = "/content/sam3_crossarm_project"

# -----------------------------------------------------------------------------
# Core state / artifact folders
# -----------------------------------------------------------------------------
# STATE_DIR: Saves the notebook state (like pandas DataFrames) to avoid recomputing.
# ART_DIR: General folder for output artifacts.
STATE_DIR = os.path.join(WORK_DIR, "state")
DF_DIR    = os.path.join(STATE_DIR, "dataframes")
ART_DIR   = os.path.join(WORK_DIR, "artifacts")

# =============================================================================
# PROJECT FOLDER STRUCTURE
# =============================================================================

# -----------------------------------------------------------------------------
# Bronze: raw source data
# -----------------------------------------------------------------------------
# Where the untouched, uploaded ZIP and extracted images live.
BRONZE_ROOT       = os.path.join(WORK_DIR, "bronze")
BRONZE_INPUT_ZIPS = os.path.join(BRONZE_ROOT, "input_zips")
BRONZE_IMAGES     = os.path.join(BRONZE_ROOT, "images")
XARM_EXTRACT_DIR  = os.path.join(BRONZE_IMAGES, "xarm")

# -----------------------------------------------------------------------------
# Silver: processed intermediate artifacts
# -----------------------------------------------------------------------------
# Where SAM3 outputs (masks, crops) and intermediate tables/runs are stored.
SILVER_ROOT            = os.path.join(WORK_DIR, "silver")
SILVER_TABLES          = os.path.join(SILVER_ROOT, "tables")
SILVER_CROSSARM_MASKS  = os.path.join(SILVER_ROOT, "crossarm_masks")
SILVER_CROSSARM_CROPS  = os.path.join(SILVER_ROOT, "crossarm_crops")
SILVER_PROMPT_RUNS     = os.path.join(SILVER_ROOT, "prompt_runs")
SILVER_DEBUG           = os.path.join(SILVER_ROOT, "debug")

# -----------------------------------------------------------------------------
# Gold: final curated outputs
# -----------------------------------------------------------------------------
# Where the final evaluation metrics, visual summaries, and submission files end up.
GOLD_ROOT        = os.path.join(WORK_DIR, "gold")
GOLD_SUMMARIES   = os.path.join(GOLD_ROOT, "summaries")
GOLD_VISUALS     = os.path.join(GOLD_ROOT, "visuals")
GOLD_EXPORTS     = os.path.join(GOLD_ROOT, "exports")

# -----------------------------------------------------------------------------
# Config: reproducible pipeline settings
# -----------------------------------------------------------------------------
# Stores pipeline parameters so you can track exactly how a run was configured.
CONFIG_ROOT      = os.path.join(WORK_DIR, "config")
CONFIG_PROMPTS   = os.path.join(CONFIG_ROOT, "prompts")
CONFIG_THRESH    = os.path.join(CONFIG_ROOT, "thresholds")

# -----------------------------------------------------------------------------
# Experiments / cache
# -----------------------------------------------------------------------------
# EXPERIMENTS: Folder for storing isolated run outputs.
# MODEL_CACHE_DIR: Caches Hugging Face model files so they don't re-download.
EXPERIMENTS      = os.path.join(WORK_DIR, "experiments")
MODEL_CACHE_DIR  = os.path.join(WORK_DIR, "model_cache")

# -----------------------------------------------------------------------------
# Create directory tree
# -----------------------------------------------------------------------------
# Safely initializes all the defined folders above so later code doesn't crash.
for d in [
    WORK_DIR,
    STATE_DIR,
    DF_DIR,
    ART_DIR,
    BRONZE_ROOT,
    BRONZE_INPUT_ZIPS,
    BRONZE_IMAGES,
    XARM_EXTRACT_DIR,
    SILVER_ROOT,
    SILVER_TABLES,
    SILVER_CROSSARM_MASKS,
    SILVER_CROSSARM_CROPS,
    SILVER_PROMPT_RUNS,
    SILVER_DEBUG,
    GOLD_ROOT,
    GOLD_SUMMARIES,
    GOLD_VISUALS,
    GOLD_EXPORTS,
    CONFIG_ROOT,
    CONFIG_PROMPTS,
    CONFIG_THRESH,
    EXPERIMENTS,
    MODEL_CACHE_DIR,
]:
    os.makedirs(d, exist_ok=True)

# =============================================================================
# CONSTANTS
# =============================================================================

# -----------------------------------------------------------------------------
# Reproducibility constants
# -----------------------------------------------------------------------------
# Sets fixed random seeds. Helps produce consistent behavior if randomly
# sampling or augmenting later.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------------------------------------------------------------
# Project identity
# -----------------------------------------------------------------------------
# Labels the notebook and creates a unique timestamp for run bookkeeping.
PROJECT_NAME = "sam3_crossarm_detection"
RUN_STAMP = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M%S")

# -----------------------------------------------------------------------------
# Dataset / input hints
# -----------------------------------------------------------------------------
# LOCAL_ZIP_HINT: Reminder string of the local Windows path.
LOCAL_ZIP_HINT = r"C:\Users\shary\Documents\Defect_Project\Xarm.zip"

# ONE standard internal ZIP path only
# Ensures we are explicitly saving uploads to a clean, unified location.
PROJECT_ZIP_NAME = "uploaded_dataset.zip"
PROJECT_ZIP_PATH = os.path.join(BRONZE_INPUT_ZIPS, PROJECT_ZIP_NAME)

# Tells the notebook which files count as images when scanning extracted folders.
IMAGE_EXTS = (
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff",
    ".JPG", ".JPEG", ".PNG", ".BMP", ".WEBP", ".TIF", ".TIFF"
)

# -----------------------------------------------------------------------------
# SAM3 model and inference constants
# -----------------------------------------------------------------------------
# SAM3_MODEL_ID: Tells Hugging Face which model repo to load.
SAM3_MODEL_ID = "facebook/sam3"

# Used later to filter text-prompt results and turn soft masks binary.
TEXT_SCORE_THRESHOLD = 0.45
MASK_THRESHOLD = 0.50
MAX_IMAGES_FOR_DEBUG_GALLERY = 40

# Starter list. Later inference cells will loop through these to find the best result.
DEFAULT_CROSSARM_PROMPTS = [
    "utility pole crossarm",
    "electricity pole crossarm",
    "distribution pole crossarm",
]

# -----------------------------------------------------------------------------
# Persist starter prompt config
# -----------------------------------------------------------------------------
# Writes the starter prompt list to disk as a JSON file for safe-keeping outside RAM.
DEFAULT_PROMPTS_JSON = os.path.join(CONFIG_PROMPTS, "default_crossarm_prompts.json")
with open(DEFAULT_PROMPTS_JSON, "w", encoding="utf-8") as f:
    json.dump(DEFAULT_CROSSARM_PROMPTS, f, indent=2)

# -----------------------------------------------------------------------------
# Helpful runtime info
# -----------------------------------------------------------------------------
# Checks whether a GPU is available and chooses numeric precision (float16 is faster on GPU).
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# Prints out what was set up to verify the environment.
print("Project workspace ready.")
print(f"PROJECT_NAME          : {PROJECT_NAME}")
print(f"WORK_DIR              : {WORK_DIR}")
print(f"DEVICE                : {DEVICE}")
print(f"MODEL_DTYPE           : {MODEL_DTYPE}")
print(f"SAM3_MODEL_ID         : {SAM3_MODEL_ID}")
print(f"LOCAL_ZIP_HINT        : {LOCAL_ZIP_HINT}")
print(f"PROJECT_ZIP_PATH      : {PROJECT_ZIP_PATH}")
print(f"XARM_EXTRACT_DIR      : {XARM_EXTRACT_DIR}")
print(f"DEFAULT_PROMPTS_JSON  : {DEFAULT_PROMPTS_JSON}")
print("DEFAULT_CROSSARM_PROMPTS:")
for p in DEFAULT_CROSSARM_PROMPTS:
    print(f"  - {p}")



In [ ]:
# =============================================================================
# CELL 3 — CONTINUITY: SAVE / RESTORE
# =============================================================================
# EXPLANATION:
# Colab sessions can disconnect, restart, or clear in-memory variables.
# This cell gives the notebook a lightweight persistence mechanism so you do
# not lose important tables and configuration every time the runtime resets.
#
# WHAT THIS CELL DOES:
#   1) Defines which DataFrames are worth saving by default
#   2) Provides a save_state() function
#   3) Provides a restore_state() function
#   4) Saves both:
#        - configuration / paths / constants
#        - selected pandas DataFrames
#
# WHY THIS MATTERS:
# - Without this, a Colab disconnect means you may need to rerun earlier cells
#   just to rebuild paths, image manifests, or later SAM3 result tables.
# - This does NOT save the model in memory.
# - It saves enough notebook state so the pipeline is easier to resume.
#
# PIPELINE ALIGNMENT:
#   - Cell 2 defines workspace paths and constants
#   - Cell 2b builds images_df and stores uploaded ZIP metadata
#   - This cell allows those outputs to be saved and restored later

# =============================================================================
# DEFAULT DATAFRAME REGISTRY
# =============================================================================
# EXPLANATION:
# This is the default list of DataFrame variable names the notebook will try
# to save if you call save_state() without explicitly passing df_names.
#
# IMPORTANT:
# - These names are just defaults.
# - A DataFrame is only saved if a global variable with that name actually
#   exists AND is a pandas DataFrame.
# - Missing names do not crash the save process.
# =============================================================================
PIPELINE_DF_NAMES = [
    "images_df",             # one row per discovered image from Cell 2b
    "prompt_manifest_df",    # optional future table: planned image x prompt jobs
    "prompt_results_df",     # optional future table: raw SAM3 prompt results
    "crossarm_masks_df",     # optional future table: saved mask manifest
    "crossarm_crops_df",     # optional future table: saved crop manifest
    "run_summary_df",        # optional future table: per-run summary stats
    "eval_df",               # optional future table: evaluation / QA outputs
]

# =============================================================================
# SAVE STATE FUNCTION
# =============================================================================
def save_state(df_names=None, config_extra=None, nb_globals=None):
    """
    Save selected DataFrames and the current project configuration to disk.

    Args:
        df_names:
            Optional list of global DataFrame variable names to save.
            If None, the default PIPELINE_DF_NAMES list is used.

        config_extra:
            Optional dictionary of extra config values to merge into the
            saved JSON snapshot.

        nb_globals:
            Optional namespace dictionary.
            Defaults to the current notebook globals.

    Returns:
        str:
            Path to the saved config.json file.
    """

    # -------------------------------------------------------------------------
    # 1. Resolve the notebook namespace
    # -------------------------------------------------------------------------
    # Using an explicit namespace keeps the function flexible and makes the
    # restore/save logic easier to reason about.
    if nb_globals is None:
        nb_globals = globals()

    # -------------------------------------------------------------------------
    # 2. Resolve which DataFrame names should be considered for saving
    # -------------------------------------------------------------------------
    if df_names is None:
        df_names = PIPELINE_DF_NAMES

    # -------------------------------------------------------------------------
    # 3. Resolve required output directories safely
    # -------------------------------------------------------------------------
    # These must exist to save the config JSON and parquet DataFrames.
    state_dir = nb_globals.get("STATE_DIR")
    df_dir = nb_globals.get("DF_DIR")

    if not state_dir:
        raise NameError("STATE_DIR is not defined. Please run Cell 2 first.")

    if not df_dir:
        raise NameError("DF_DIR is not defined. Please run Cell 2 first.")

    os.makedirs(state_dir, exist_ok=True)
    os.makedirs(df_dir, exist_ok=True)

    # -------------------------------------------------------------------------
    # 4. Build the configuration snapshot dictionary
    # -------------------------------------------------------------------------
    # This captures the important notebook constants, paths, runtime details,
    # and pipeline settings so the environment can be reconstructed later.
    cfg = {
        # ---------------------------------------------------------------------
        # Timestamps
        # ---------------------------------------------------------------------
        "timestamp_utc": time.time(),
        "timestamp_utc_iso": pd.Timestamp.utcnow().isoformat(),

        # ---------------------------------------------------------------------
        # Project identity
        # ---------------------------------------------------------------------
        "PROJECT_NAME": nb_globals.get("PROJECT_NAME"),
        "RUN_STAMP": nb_globals.get("RUN_STAMP"),

        # ---------------------------------------------------------------------
        # Core workspace paths
        # ---------------------------------------------------------------------
        "WORK_DIR": nb_globals.get("WORK_DIR"),
        "STATE_DIR": nb_globals.get("STATE_DIR"),
        "DF_DIR": nb_globals.get("DF_DIR"),
        "ART_DIR": nb_globals.get("ART_DIR"),

        # ---------------------------------------------------------------------
        # Bronze layer paths
        # ---------------------------------------------------------------------
        "BRONZE_ROOT": nb_globals.get("BRONZE_ROOT"),
        "BRONZE_INPUT_ZIPS": nb_globals.get("BRONZE_INPUT_ZIPS"),
        "BRONZE_IMAGES": nb_globals.get("BRONZE_IMAGES"),
        "XARM_EXTRACT_DIR": nb_globals.get("XARM_EXTRACT_DIR"),

        # ---------------------------------------------------------------------
        # ZIP handling
        # ---------------------------------------------------------------------
        # Current pipeline uses ONE stable internal ZIP path only.
        # UPLOADED_ZIP_DISPLAY_NAME is just the user-facing filename from Colab.
        "LOCAL_ZIP_HINT": nb_globals.get("LOCAL_ZIP_HINT"),
        "PROJECT_ZIP_NAME": nb_globals.get("PROJECT_ZIP_NAME"),
        "PROJECT_ZIP_PATH": nb_globals.get("PROJECT_ZIP_PATH"),
        "UPLOADED_ZIP_DISPLAY_NAME": nb_globals.get("UPLOADED_ZIP_DISPLAY_NAME"),

        # ---------------------------------------------------------------------
        # Silver layer paths
        # ---------------------------------------------------------------------
        "SILVER_ROOT": nb_globals.get("SILVER_ROOT"),
        "SILVER_TABLES": nb_globals.get("SILVER_TABLES"),
        "SILVER_CROSSARM_MASKS": nb_globals.get("SILVER_CROSSARM_MASKS"),
        "SILVER_CROSSARM_CROPS": nb_globals.get("SILVER_CROSSARM_CROPS"),
        "SILVER_PROMPT_RUNS": nb_globals.get("SILVER_PROMPT_RUNS"),
        "SILVER_DEBUG": nb_globals.get("SILVER_DEBUG"),

        # ---------------------------------------------------------------------
        # Gold layer paths
        # ---------------------------------------------------------------------
        "GOLD_ROOT": nb_globals.get("GOLD_ROOT"),
        "GOLD_SUMMARIES": nb_globals.get("GOLD_SUMMARIES"),
        "GOLD_VISUALS": nb_globals.get("GOLD_VISUALS"),
        "GOLD_EXPORTS": nb_globals.get("GOLD_EXPORTS"),

        # ---------------------------------------------------------------------
        # Config / cache / experiments
        # ---------------------------------------------------------------------
        "CONFIG_ROOT": nb_globals.get("CONFIG_ROOT"),
        "CONFIG_PROMPTS": nb_globals.get("CONFIG_PROMPTS"),
        "CONFIG_THRESH": nb_globals.get("CONFIG_THRESH"),
        "EXPERIMENTS": nb_globals.get("EXPERIMENTS"),
        "MODEL_CACHE_DIR": nb_globals.get("MODEL_CACHE_DIR"),
        "DEFAULT_PROMPTS_JSON": nb_globals.get("DEFAULT_PROMPTS_JSON"),

        # ---------------------------------------------------------------------
        # Core constants
        # ---------------------------------------------------------------------
        # IMAGE_EXTS is intentionally cast to list so JSON can serialize it.
        "SEED": nb_globals.get("SEED"),
        "IMAGE_EXTS": list(nb_globals.get("IMAGE_EXTS", [])),

        # ---------------------------------------------------------------------
        # SAM3 config
        # ---------------------------------------------------------------------
        "SAM3_MODEL_ID": nb_globals.get("SAM3_MODEL_ID"),
        "TEXT_SCORE_THRESHOLD": nb_globals.get("TEXT_SCORE_THRESHOLD"),
        "MASK_THRESHOLD": nb_globals.get("MASK_THRESHOLD"),
        "MAX_IMAGES_FOR_DEBUG_GALLERY": nb_globals.get("MAX_IMAGES_FOR_DEBUG_GALLERY"),
        "DEFAULT_CROSSARM_PROMPTS": nb_globals.get("DEFAULT_CROSSARM_PROMPTS"),

        # ---------------------------------------------------------------------
        # Runtime info
        # ---------------------------------------------------------------------
        "DEVICE": nb_globals.get("DEVICE"),
        "MODEL_DTYPE": str(nb_globals.get("MODEL_DTYPE")),
    }

    # -------------------------------------------------------------------------
    # 5. Merge in any additional caller-supplied config
    # -------------------------------------------------------------------------
    if isinstance(config_extra, dict):
        cfg.update(config_extra)

    # -------------------------------------------------------------------------
    # 6. Save config snapshot to JSON
    # -------------------------------------------------------------------------
    cfg_path = os.path.join(state_dir, "config.json")
    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, default=str)

    # -------------------------------------------------------------------------
    # 7. Save selected DataFrames to parquet
    # -------------------------------------------------------------------------
    saved = []

    for name in df_names:
        obj = nb_globals.get(name)

        if isinstance(obj, pd.DataFrame):
            out_path = os.path.join(df_dir, f"{name}.parquet")
            obj.to_parquet(out_path, index=False)
            saved.append((name, len(obj)))

    # -------------------------------------------------------------------------
    # 8. Print save summary
    # -------------------------------------------------------------------------
    print(f"Saved config -> {cfg_path}")

    if saved:
        print("Saved DataFrames:")
        for name, n_rows in saved:
            print(f"  - {name}: {n_rows} rows")
    else:
        print("No DataFrames were found to save.")

    return cfg_path

# =============================================================================
# RESTORE STATE FUNCTION
# =============================================================================
def restore_state(nb_globals=None):
    """
    Restore saved config values and DataFrames back into notebook globals.

    Args:
        nb_globals:
            Optional namespace dictionary.
            Defaults to the current notebook globals.

    Returns:
        None
    """

    # -------------------------------------------------------------------------
    # 1. Resolve the notebook namespace
    # -------------------------------------------------------------------------
    if nb_globals is None:
        nb_globals = globals()

    # -------------------------------------------------------------------------
    # 2. Resolve required directories safely
    # -------------------------------------------------------------------------
    state_dir = nb_globals.get("STATE_DIR")
    df_dir = nb_globals.get("DF_DIR")

    if not state_dir:
        raise NameError(
            "STATE_DIR is not defined. Please run Cell 2 first before restore_state()."
        )

    if not df_dir:
        raise NameError(
            "DF_DIR is not defined. Please run Cell 2 first before restore_state()."
        )

    # -------------------------------------------------------------------------
    # 3. Locate the saved config snapshot
    # -------------------------------------------------------------------------
    cfg_path = os.path.join(state_dir, "config.json")

    # -------------------------------------------------------------------------
    # 4. Restore config values into notebook globals
    # -------------------------------------------------------------------------
    # IMPORTANT:
    # - this restores Python variables only
    # - it does NOT reload the SAM3 model into memory
    if os.path.exists(cfg_path):
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)

        for k, v in cfg.items():
            # IMAGE_EXTS was serialized as a list for JSON compatibility.
            # Convert it back to a tuple so str.endswith(IMAGE_EXTS) still works.
            if k == "IMAGE_EXTS" and isinstance(v, list):
                nb_globals[k] = tuple(v)
            else:
                nb_globals[k] = v

        print(f"Restored config keys: {sorted(cfg.keys())}")
    else:
        print("No config.json found.")

    # -------------------------------------------------------------------------
    # 5. Discover saved parquet DataFrames
    # -------------------------------------------------------------------------
    df_files = sorted(glob.glob(os.path.join(df_dir, "*.parquet")))

    if not df_files:
        print("No saved DataFrames found.")
        return

    # -------------------------------------------------------------------------
    # 6. Reload each parquet file into the notebook namespace
    # -------------------------------------------------------------------------
    for fp in df_files:
        name = os.path.splitext(os.path.basename(fp))[0]
        nb_globals[name] = pd.read_parquet(fp)
        print(f"  Restored {name}: {len(nb_globals[name])} rows")

# =============================================================================
# READY MESSAGE
# =============================================================================
print(f"State functions ready. Workspace: {globals().get('WORK_DIR')}")
print(f"Standard project ZIP path : {globals().get('PROJECT_ZIP_PATH')}")
print(f"Uploaded ZIP display name : {globals().get('UPLOADED_ZIP_DISPLAY_NAME')}")

In [ ]:
# =============================================================================
# CELL 4 — GLOBAL DIRECTORY CHECK
# =============================================================================
# EXPLANATION:
# This cell does NOT create or overwrite any paths.
# It only checks that the directory structure defined in Cell 2 still exists
# and that the notebook is using one consistent workspace layout.
#
# WHY THIS MATTERS:
# Later cells will try to save:
# - tables
# - masks
# - crops
# - prompt-run outputs
# - debug artifacts
#
# If any required folder is missing, later steps may fail in confusing ways.
# This cell gives an early sanity check before model loading or inference.
#
# PIPELINE ALIGNMENT:
# - Cell 2 defines the folder structure
# - Cell 2b uploads the ZIP and extracts images
# - Cell 3 provides save / restore
# - This cell verifies the filesystem side of that setup
#
# UPDATED FOR CURRENT ZIP FLOW:
# - uses ONE stable ZIP path: PROJECT_ZIP_PATH
# - shows uploaded filename via UPLOADED_ZIP_DISPLAY_NAME
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
# This keeps the cell more self-contained even if run after a partial restart.
import os
import pandas as pd

# -----------------------------------------------------------------------------
# 0. Safety check: confirm Cell 2 has already run
# -----------------------------------------------------------------------------
# We check the key variables needed to perform the directory validation.
required_globals = [
    "WORK_DIR",
    "STATE_DIR",
    "DF_DIR",
    "ART_DIR",
    "BRONZE_ROOT",
    "BRONZE_INPUT_ZIPS",
    "BRONZE_IMAGES",
    "XARM_EXTRACT_DIR",
    "SILVER_ROOT",
    "SILVER_TABLES",
    "SILVER_CROSSARM_MASKS",
    "SILVER_CROSSARM_CROPS",
    "SILVER_PROMPT_RUNS",
    "SILVER_DEBUG",
    "GOLD_ROOT",
    "GOLD_SUMMARIES",
    "GOLD_VISUALS",
    "GOLD_EXPORTS",
    "CONFIG_ROOT",
    "CONFIG_PROMPTS",
    "CONFIG_THRESH",
    "EXPERIMENTS",
    "MODEL_CACHE_DIR",
    "PROJECT_ZIP_PATH",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 4 cannot run because some required variables are missing.\n"
        "Please run Cell 2 first.\n"
        f"Missing globals: {missing_globals}"
    )

# -----------------------------------------------------------------------------
# 1. Build the list of directories that should already exist
# -----------------------------------------------------------------------------
# These directories are expected to exist because Cell 2 creates them.
required_dirs = [
    # -------------------------------------------------------------------------
    # Core workspace / state folders
    # -------------------------------------------------------------------------
    WORK_DIR,
    STATE_DIR,
    DF_DIR,
    ART_DIR,

    # -------------------------------------------------------------------------
    # Bronze folders
    # -------------------------------------------------------------------------
    BRONZE_ROOT,
    BRONZE_INPUT_ZIPS,
    BRONZE_IMAGES,
    XARM_EXTRACT_DIR,

    # -------------------------------------------------------------------------
    # Silver folders
    # -------------------------------------------------------------------------
    SILVER_ROOT,
    SILVER_TABLES,
    SILVER_CROSSARM_MASKS,
    SILVER_CROSSARM_CROPS,
    SILVER_PROMPT_RUNS,
    SILVER_DEBUG,

    # -------------------------------------------------------------------------
    # Gold folders
    # -------------------------------------------------------------------------
    GOLD_ROOT,
    GOLD_SUMMARIES,
    GOLD_VISUALS,
    GOLD_EXPORTS,

    # -------------------------------------------------------------------------
    # Config folders
    # -------------------------------------------------------------------------
    CONFIG_ROOT,
    CONFIG_PROMPTS,
    CONFIG_THRESH,

    # -------------------------------------------------------------------------
    # Experiment / cache folders
    # -------------------------------------------------------------------------
    EXPERIMENTS,
    MODEL_CACHE_DIR,
]

# -----------------------------------------------------------------------------
# 2. Detect missing directories
# -----------------------------------------------------------------------------
# If any path does not currently exist on disk, collect it here.
missing_dirs = [d for d in required_dirs if not os.path.isdir(d)]

# -----------------------------------------------------------------------------
# 3. Print validation result
# -----------------------------------------------------------------------------
if missing_dirs:
    print("WARNING: The following required directories are missing:")
    for d in missing_dirs:
        print(f"  - {d}")
else:
    print("All key directories exist and are consistent.")

# -----------------------------------------------------------------------------
# 4. ZIP file status check
# -----------------------------------------------------------------------------
# PROJECT_ZIP_PATH is the one stable internal ZIP location used by the current
# pipeline. UPLOADED_ZIP_DISPLAY_NAME is just the original filename selected
# during the Colab upload step.
print("\nZIP file summary:")

if os.path.isfile(PROJECT_ZIP_PATH):
    print(f"  PROJECT_ZIP_PATH         : {PROJECT_ZIP_PATH}")
else:
    print(f"  PROJECT_ZIP_PATH         : MISSING -> {PROJECT_ZIP_PATH}")

print(f"  UPLOADED_ZIP_DISPLAY_NAME: {globals().get('UPLOADED_ZIP_DISPLAY_NAME')}")

# -----------------------------------------------------------------------------
# 5. Extracted image folder status
# -----------------------------------------------------------------------------
# This helps confirm whether Cell 2b has likely been run.
if os.path.isdir(XARM_EXTRACT_DIR):
    extracted_items = sorted(os.listdir(XARM_EXTRACT_DIR))
    print(f"  XARM_EXTRACT_DIR exists  : Yes")
    print(f"  Extracted top-level items: {len(extracted_items)}")
else:
    print(f"  XARM_EXTRACT_DIR exists  : No")

# -----------------------------------------------------------------------------
# 6. images_df status check
# -----------------------------------------------------------------------------
# This helps confirm whether the image manifest table already exists in memory.
print("\nDataFrame summary:")

images_df_obj = globals().get("images_df")

if isinstance(images_df_obj, pd.DataFrame):
    print(f"  images_df exists         : Yes")
    print(f"  images_df rows           : {len(images_df_obj)}")
    print(f"  images_df columns        : {list(images_df_obj.columns)}")
else:
    print("  images_df exists         : No")

# -----------------------------------------------------------------------------
# 7. Print a readable summary of the active project structure
# -----------------------------------------------------------------------------
print("\nDirectory summary:")
print(f"  WORK_DIR                 : {WORK_DIR}")
print(f"  STATE_DIR                : {STATE_DIR}")
print(f"  DF_DIR                   : {DF_DIR}")
print(f"  ART_DIR                  : {ART_DIR}")

print(f"  BRONZE_ROOT              : {BRONZE_ROOT}")
print(f"  BRONZE_INPUT_ZIPS        : {BRONZE_INPUT_ZIPS}")
print(f"  BRONZE_IMAGES            : {BRONZE_IMAGES}")
print(f"  XARM_EXTRACT_DIR         : {XARM_EXTRACT_DIR}")

print(f"  SILVER_ROOT              : {SILVER_ROOT}")
print(f"  SILVER_TABLES            : {SILVER_TABLES}")
print(f"  SILVER_CROSSARM_MASKS    : {SILVER_CROSSARM_MASKS}")
print(f"  SILVER_CROSSARM_CROPS    : {SILVER_CROSSARM_CROPS}")
print(f"  SILVER_PROMPT_RUNS       : {SILVER_PROMPT_RUNS}")
print(f"  SILVER_DEBUG             : {SILVER_DEBUG}")

print(f"  GOLD_ROOT                : {GOLD_ROOT}")
print(f"  GOLD_SUMMARIES           : {GOLD_SUMMARIES}")
print(f"  GOLD_VISUALS             : {GOLD_VISUALS}")
print(f"  GOLD_EXPORTS             : {GOLD_EXPORTS}")

print(f"  CONFIG_ROOT              : {CONFIG_ROOT}")
print(f"  CONFIG_PROMPTS           : {CONFIG_PROMPTS}")
print(f"  CONFIG_THRESH            : {CONFIG_THRESH}")

print(f"  EXPERIMENTS              : {EXPERIMENTS}")
print(f"  MODEL_CACHE_DIR          : {MODEL_CACHE_DIR}")

In [ ]:
# =============================================================================
# CELL 5a — HF LOGIN & GATED ACCESS TEST
# =============================================================================
# EXPLANATION:
# This cell authenticates to Hugging Face and confirms that the current account
# can access the gated "facebook/sam3" repository.
#
# WHY THIS CELL EXISTS:
# - SAM3 access is gated, so package installation alone is not enough
# - we want to fail early if token access is missing
# - we do NOT load model weights here; this is auth/access validation only
#
# PIPELINE ALIGNMENT:
# - Cell 2 defines SAM3_MODEL_ID and MODEL_CACHE_DIR
# - Cell 3 defines save_state()
# - This cell verifies auth and access before Cell 5b loads the model
# =============================================================================

# -----------------------------------------------------------------------------
# Imports required for this cell
# -----------------------------------------------------------------------------
import os
from getpass import getpass
from huggingface_hub import login, HfApi, hf_hub_download

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
# These variables should already exist from Cell 2.
required_globals = [
    "SAM3_MODEL_ID",
    "MODEL_CACHE_DIR",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 5a cannot run because some required variables are missing.\n"
        "Please run Cell 2 first.\n"
        f"Missing globals: {missing_globals}"
    )

# -----------------------------------------------------------------------------
# 1. Try environment variables first
# -----------------------------------------------------------------------------
token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
token_source = "environment"

# -----------------------------------------------------------------------------
# 2. Try Colab secrets if not found
# -----------------------------------------------------------------------------
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        token_source = "colab_userdata"
    except Exception:
        token = None

# -----------------------------------------------------------------------------
# 3. Hidden manual prompt if still missing
# -----------------------------------------------------------------------------
if not token:
    token = getpass("Paste your HF token (hf_...): ").strip()
    token_source = "manual_getpass"

# -----------------------------------------------------------------------------
# 4. Basic token validation
# -----------------------------------------------------------------------------
if not token:
    raise ValueError("No Hugging Face token was provided.")

if not isinstance(token, str) or not token.startswith("hf_"):
    raise ValueError("Token does not look like a valid Hugging Face token.")

# -----------------------------------------------------------------------------
# 5. Store token for downstream cells
# -----------------------------------------------------------------------------
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token

# -----------------------------------------------------------------------------
# 6. Login
# -----------------------------------------------------------------------------
login(token=token, add_to_git_credential=False)

# -----------------------------------------------------------------------------
# 7. Verify identity
# -----------------------------------------------------------------------------
api = HfApi(token=token)
me = api.whoami()

HF_AUTHENTICATED_USER = me.get("name") or me.get("email") or str(me)
HF_TOKEN_SOURCE = token_source

print(f"Logged in as: {HF_AUTHENTICATED_USER}")
print(f"Token source: {HF_TOKEN_SOURCE}")

# -----------------------------------------------------------------------------
# 8. Gated repo access smoke test
# -----------------------------------------------------------------------------
# We download a small known file from the gated repo to confirm that:
# - the token is valid
# - the account has been approved for facebook/sam3
# - the local model cache path is working
try:
    SAM3_CONFIG_PATH = hf_hub_download(
        repo_id=SAM3_MODEL_ID,
        filename="config.json",
        token=token,
        cache_dir=MODEL_CACHE_DIR,
    )

    print("Gated file download OK.")
    print(f"SAM3 config accessible: {SAM3_CONFIG_PATH}")

except Exception as e:
    raise RuntimeError(
        f"Hugging Face login succeeded, but gated access to '{SAM3_MODEL_ID}' failed: {e}"
    )

# -----------------------------------------------------------------------------
# 9. Persist useful auth/access metadata
# -----------------------------------------------------------------------------
if "save_state" in globals():
    save_state(
        df_names=[],
        config_extra={
            "HF_AUTHENTICATED_USER": HF_AUTHENTICATED_USER,
            "HF_TOKEN_SOURCE": HF_TOKEN_SOURCE,
            "SAM3_CONFIG_PATH": SAM3_CONFIG_PATH,
        },
        nb_globals=globals(),
    )

In [ ]:
# =============================================================================
# CELL 5b — LOAD HF SAM3 MODEL + PROCESSOR
# =============================================================================
# EXPLANATION:
# Cell 5a already handled:
# - Hugging Face login
# - gated access validation
# - confirmation that the SAM3 repo can be reached
#
# This cell now loads the actual SAM3 model + processor into memory so later
# cells can run text-prompt-based crossarm masking on the uploaded images.
#
# OUTPUTS:
#   - DEVICE
#   - MODEL_DTYPE
#   - model
#   - processor
#   - SAM3_READY
#
# WHY THIS MATTERS:
# - Cell 5a proves access is valid
# - this cell does the heavier step of loading the model weights
# - later inference cells will use:
#     - model
#     - processor
#     - DEVICE
#
# DESIGN NOTES:
# - uses MODEL_CACHE_DIR from Cell 2
# - uses the HF token already stored by Cell 5a
# - safe to rerun because it clears old model objects first
# - saves a config snapshot using the updated Cell 3 save_state() signature
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
# os     : read token from environment
# gc     : free Python-side objects before reloading
# torch  : device + dtype + model placement
# transformers : load HF SAM3 classes
import os
import gc
import torch
import pandas as pd
from transformers import Sam3Model, Sam3Processor

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
# These variables should already exist if the earlier cells ran correctly.
required_globals = [
    "SAM3_MODEL_ID",
    "MODEL_CACHE_DIR",
    "save_state",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 5b cannot run because some required variables are missing.\n"
        "Please make sure Cells 2, 3, and 5a have already run.\n"
        f"Missing globals: {missing_globals}"
    )

# -----------------------------------------------------------------------------
# 1. Resolve HF token from environment
# -----------------------------------------------------------------------------
# Cell 5a stored the token in the process environment so downstream cells can
# reuse it without asking again.
token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")

if not token:
    raise ValueError(
        "HF token not found in environment.\n"
        "Please run Cell 5a first."
    )

# -----------------------------------------------------------------------------
# 2. Clean up any previously loaded SAM3 objects
# -----------------------------------------------------------------------------
# This makes the cell safe to rerun.
#
# WHY THIS MATTERS:
# - if you rerun the cell without cleanup, old objects may still occupy memory
# - clearing them first reduces the chance of GPU OOM issues
if "model" in globals():
    try:
        del model
    except Exception:
        pass

if "processor" in globals():
    try:
        del processor
    except Exception:
        pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# -----------------------------------------------------------------------------
# 3. Resolve runtime device and dtype
# -----------------------------------------------------------------------------
# DEVICE:
# - cuda if GPU is available
# - cpu otherwise
#
# MODEL_DTYPE:
# - float16 on GPU for lower memory usage / faster inference
# - float32 on CPU for safer compatibility
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print("Preparing SAM3 load...")
print(f"  SAM3_MODEL_ID   : {SAM3_MODEL_ID}")
print(f"  MODEL_CACHE_DIR : {MODEL_CACHE_DIR}")
print(f"  DEVICE          : {DEVICE}")
print(f"  MODEL_DTYPE     : {MODEL_DTYPE}")

# -----------------------------------------------------------------------------
# 4. Load processor
# -----------------------------------------------------------------------------
# The processor handles text/image preparation and later post-processing.
print("\nLoading SAM3 processor...")

processor = Sam3Processor.from_pretrained(
    SAM3_MODEL_ID,
    token=token,
    cache_dir=MODEL_CACHE_DIR,
)

print("Processor loaded.")

# -----------------------------------------------------------------------------
# 5. Load model weights
# -----------------------------------------------------------------------------
# low_cpu_mem_usage=True helps reduce RAM pressure during loading.
#
# After loading:
# - move model to DEVICE
# - switch to eval() because this is inference, not training
print("Loading SAM3 model weights...")

model = Sam3Model.from_pretrained(
    SAM3_MODEL_ID,
    token=token,
    cache_dir=MODEL_CACHE_DIR,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
)

model = model.to(DEVICE)
model.eval()

# -----------------------------------------------------------------------------
# 6. Runtime / hardware summary
# -----------------------------------------------------------------------------
SAM3_READY = True

print("\nSAM3 load complete.")

if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    alloc_gb = torch.cuda.memory_allocated() / 1e9
    reserved_gb = torch.cuda.memory_reserved() / 1e9

    print("\nGPU summary:")
    print(f"  GPU name        : {gpu_name}")
    print(f"  Total VRAM      : {vram_gb:.2f} GB")
    print(f"  Allocated VRAM  : {alloc_gb:.2f} GB")
    print(f"  Reserved VRAM   : {reserved_gb:.2f} GB")
else:
    print("\nRunning on CPU.")

print("\nSAM3 objects ready:")
print(f"  DEVICE          : {DEVICE}")
print(f"  MODEL_DTYPE     : {MODEL_DTYPE}")
print(f"  SAM3_READY      : {SAM3_READY}")

# -----------------------------------------------------------------------------
# 7. Persist runtime snapshot
# -----------------------------------------------------------------------------
# Save the key runtime values so a later restore has the updated state.
#
# NOTE:
# - this does NOT save model weights to your notebook state
# - it only saves metadata / config
save_state(
    df_names=["images_df"] if isinstance(globals().get("images_df"), pd.DataFrame) else [],
    config_extra={
        "DEVICE": DEVICE,
        "MODEL_DTYPE": str(MODEL_DTYPE),
        "SAM3_DEVICE": DEVICE,
        "SAM3_DTYPE": str(MODEL_DTYPE),
        "SAM3_READY": SAM3_READY,
        "SAM3_MODEL_ID": SAM3_MODEL_ID,
        "MODEL_CACHE_DIR": MODEL_CACHE_DIR,
    },
    nb_globals=globals(),
)

print("\nCell 5b finished successfully.")

In [ ]:
# =============================================================================
# CELL 6a — PREPARE IMAGE DATAFRAME FOR DEBUG / INFERENCE
# =============================================================================
# EXPLANATION:
# This cell does NOT run SAM3 yet.
#
# It only prepares the uploaded image manifest for later processing.
#
# PURPOSE:
# - start from images_df created in Cell 2b
# - validate that the required columns exist
# - create a clean working copy
# - ensure useful helper columns are present
# - sort images into a stable processing order
# - optionally select a small subset for debug work
#
# WHY THIS MATTERS:
# Later cells should not work directly on the raw images_df if we want:
# - safer experimentation
# - easier debugging
# - reproducible image order
# - smaller debug subsets when needed
#
# OUTPUTS:
# - run_images_df   : full cleaned working image table
# - debug_images_df : optional small subset for prompt/debug testing
# =============================================================================
# 0. Safety checks
# -----------------------------------------------------------------------------
# images_df should already exist from Cell 2b.
if "images_df" not in globals():
    raise NameError(
        "images_df not found.\n"
        "Please run Cell 2b first."
    )

if not isinstance(images_df, pd.DataFrame):
    raise TypeError(
        "images_df exists but is not a pandas DataFrame."
    )

# -----------------------------------------------------------------------------
# 1. Check that the minimum required column exists
# -----------------------------------------------------------------------------
# For this pipeline, image_path is the essential column because later cells
# use it to load the actual images from disk.
required_cols = ["image_path"]
missing_cols = [c for c in required_cols if c not in images_df.columns]

if missing_cols:
    raise ValueError(
        f"images_df is missing required columns: {missing_cols}"
    )

# -----------------------------------------------------------------------------
# 2. Make a working copy
# -----------------------------------------------------------------------------
# We do NOT want to mutate the raw manifest from Cell 2b directly.
# So we create a separate copy for downstream debug / inference work.
run_images_df = images_df.copy()

# -----------------------------------------------------------------------------
# 3. Ensure helper columns exist
# -----------------------------------------------------------------------------
# file_name:
# - just the filename portion of the image path
#
# stem:
# - filename without extension
#
# ext:
# - file extension such as .jpg or .png
if "file_name" not in run_images_df.columns:
    run_images_df["file_name"] = run_images_df["image_path"].map(os.path.basename)

if "stem" not in run_images_df.columns:
    run_images_df["stem"] = run_images_df["file_name"].map(
        lambda x: os.path.splitext(x)[0] if isinstance(x, str) else None
    )

if "ext" not in run_images_df.columns:
    run_images_df["ext"] = run_images_df["file_name"].map(
        lambda x: os.path.splitext(x)[1] if isinstance(x, str) else None
    )

# -----------------------------------------------------------------------------
# 4. Sort into a stable processing order
# -----------------------------------------------------------------------------
# This helps make runs easier to debug and compare across reruns.
sort_cols = [c for c in ["file_name", "image_path"] if c in run_images_df.columns]
run_images_df = run_images_df.sort_values(sort_cols).reset_index(drop=True)

# -----------------------------------------------------------------------------
# 5. Create a stable unique image_id if one does not already exist
# -----------------------------------------------------------------------------
# IMPORTANT:
# Using only "stem" may not be unique if different folders contain files with
# the same filename. So we append the row index to make the ID unique.
if "image_id" not in run_images_df.columns:
    run_images_df["image_id"] = (
        run_images_df["stem"].fillna("image").astype(str)
        + "_"
        + run_images_df.index.astype(str)
    )

# -----------------------------------------------------------------------------
# 6. Optional: build a smaller debug subset
# -----------------------------------------------------------------------------
# Use the global debug-gallery limit if available; otherwise fall back to 6.
NUM_DEBUG_IMAGES = int(globals().get("MAX_IMAGES_FOR_DEBUG_GALLERY", 6))

debug_images_df = run_images_df.head(NUM_DEBUG_IMAGES).copy()

# -----------------------------------------------------------------------------
# 7. Print summary
# -----------------------------------------------------------------------------
print("Image manifest preparation complete.\n")

print("Full working DataFrame:")
print(f"  run_images_df rows   : {len(run_images_df)}")
print(f"  run_images_df cols   : {list(run_images_df.columns)}")

print("\nDebug subset:")
print(f"  debug_images_df rows : {len(debug_images_df)}")

# -----------------------------------------------------------------------------
# 8. Preview tables
# -----------------------------------------------------------------------------
print("\nrun_images_df preview:")
try:
    display(run_images_df.head(10))
except NameError:
    print(run_images_df.head(10))

print("\ndebug_images_df preview:")
try:
    display(debug_images_df)
except NameError:
    print(debug_images_df)

In [ ]:
# =============================================================================
# CELL 6b — LOAD AND INSPECT ONE DEBUG IMAGE
# =============================================================================
# EXPLANATION:
# This cell does NOT run SAM3 yet.
#
# It performs a single-image sanity check using the debug image subset created
# in Cell 6a.
#
# PURPOSE:
# - confirm that debug_images_df exists and is non-empty
# - confirm that the selected image file actually exists on disk
# - load exactly one image using PIL
# - print key metadata for inspection
# - display the image inline
# - confirm that the image is in RGB mode (or convert it if not)
#
# WHY THIS MATTERS:
# Before running any model inference, we want to verify that:
# - the image paths from Cell 2b / 6a are valid
# - the files were extracted correctly
# - the image can be opened without error
# - the mode is compatible with SAM3
#
# IMPORTANT:
# - this is observation only
# - no resizing
# - no SAM3 inference
# - no loop over all images
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
# debug_images_df should already exist from Cell 6a.
if "debug_images_df" not in globals():
    raise NameError(
        "debug_images_df not found.\n"
        "Please run Cell 6a first."
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError(
        "debug_images_df exists but is not a pandas DataFrame."
    )

if debug_images_df.empty:
    raise ValueError(
        "debug_images_df exists but is empty.\n"
        "Please check Cell 6a."
    )

# -----------------------------------------------------------------------------
# 1. Choose exactly one debug row
# -----------------------------------------------------------------------------
# Keep this as a single-image check only.
DEBUG_ROW_INDEX = 0

if DEBUG_ROW_INDEX >= len(debug_images_df):
    raise IndexError(
        f"DEBUG_ROW_INDEX={DEBUG_ROW_INDEX} is out of range for "
        f"debug_images_df with {len(debug_images_df)} rows."
    )

row = debug_images_df.iloc[DEBUG_ROW_INDEX]

# -----------------------------------------------------------------------------
# 2. Extract metadata from the selected row
# -----------------------------------------------------------------------------
# Use safe fallbacks in case helper fields are missing or NaN.
image_path = row["image_path"]

image_id = row.get("image_id", None)
if pd.isna(image_id):
    image_id = None

file_name = row.get("file_name", None)
if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
    file_name = os.path.basename(image_path)

# -----------------------------------------------------------------------------
# 3. Confirm the file exists before opening it
# -----------------------------------------------------------------------------
if not isinstance(image_path, str) or len(image_path.strip()) == 0:
    raise ValueError(
        f"Selected image_path is invalid: {image_path}"
    )

if not os.path.exists(image_path):
    raise FileNotFoundError(
        "The selected image file does not exist on disk.\n"
        f"image_path: {image_path}\n\n"
        "This likely means Cell 2b extraction did not complete correctly "
        "or the extracted files were later removed."
    )

# -----------------------------------------------------------------------------
# 4. Load the image using PIL
# -----------------------------------------------------------------------------
# We inspect the original mode first, then convert to RGB if needed.
#
# IMPORTANT:
# We force the copied image to fully load inside the `with` block so it remains
# safe to use after the file handle is closed.
with Image.open(image_path) as img:
    original_mode = img.mode
    width, height = img.size

    # -------------------------------------------------------------------------
    # 5. Convert to RGB if needed
    # -------------------------------------------------------------------------
    if original_mode != "RGB":
        print(
            f"WARNING: Image mode is '{original_mode}', not 'RGB'. "
            "Converting to RGB for downstream compatibility."
        )
        img_rgb = img.convert("RGB")
    else:
        img_rgb = img.copy()

    # Force full decode into memory before leaving the with-block
    img_rgb.load()

# -----------------------------------------------------------------------------
# 6. Print metadata summary
# -----------------------------------------------------------------------------
print("Single-image sanity check:\n")
print(f"  DEBUG_ROW_INDEX : {DEBUG_ROW_INDEX}")
print(f"  image_id        : {image_id}")
print(f"  file_name       : {file_name}")
print(f"  image_path      : {image_path}")
print(f"  width           : {width}")
print(f"  height          : {height}")
print(f"  original_mode   : {original_mode}")
print(f"  final_mode      : {img_rgb.mode}")

# -----------------------------------------------------------------------------
# 7. Display the image inline
# -----------------------------------------------------------------------------
plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.title(
    f"{file_name}\n"
    f"image_id={image_id} | size={width}x{height} | mode={img_rgb.mode}",
    fontsize=11
)
plt.axis("off")
plt.show()
plt.close()

# -----------------------------------------------------------------------------
# 8. Final confirmation
# -----------------------------------------------------------------------------
print("Cell 6b completed successfully.")
print("The selected image loaded correctly and is ready for the next step.")

# -----------------------------------------------------------------------------
# 9. Cleanup
# -----------------------------------------------------------------------------
# Keep this cell purely diagnostic.
del img_rgb

In [ ]:
# =============================================================================
# CELL 7a — DETECT ONE POLE ON ONE DEBUG IMAGE
# =============================================================================
# EXPLANATION:
# This cell runs SAM3 on ONE debug image using pole-focused prompts.
#
# PURPOSE:
# - detect the utility / electricity pole first
# - inspect all pole candidate boxes
# - choose one best pole box for downstream ROI cropping
# - store the selected pole result for Cell 7b
#
# IMPORTANT:
# - this is a ONE-IMAGE prototype first
# - it does NOT crop yet
# - it does NOT run crossarm detection inside the crop yet
# - it stores:
#       POLE_CANDIDATES_DF
#       POLE_DETECTION_RESULT
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
required_globals = [
    "debug_images_df",
    "model",
    "processor",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 7a requires objects from earlier cells.\n"
        "Please run Cells 5b and 6a/6b first.\n"
        f"Missing globals: {missing_globals}"
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError("debug_images_df exists but is not a pandas DataFrame.")

if debug_images_df.empty:
    raise ValueError("debug_images_df is empty.")

# -----------------------------------------------------------------------------
# 1. Config
# -----------------------------------------------------------------------------
DEBUG_ROW_INDEX = 3

POLE_PROMPTS = [
    "utility pole",
    "electricity pole",
    "power pole",
    "wooden utility pole",
]

TEXT_THRESHOLD = float(globals().get("TEXT_SCORE_THRESHOLD", 0.30))
MASK_THRESHOLD = float(globals().get("MASK_THRESHOLD", 0.50))
RUN_DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------------------------------------------------
# 2. Small local helpers
# -----------------------------------------------------------------------------
def _to_numpy_safe(x):
    """
    Convert tensors / lists / arrays to a numpy array on CPU.

    Args:
        x: torch tensor, numpy array, list, tuple, or None

    Returns:
        np.ndarray or None
    """
    if x is None:
        return None

    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()

    if isinstance(x, (list, tuple)):
        return np.asarray(x)

    return np.asarray(x)


def _infer_num_detections(raw_boxes, raw_scores, raw_masks):
    """
    Infer how many detections exist from boxes/scores/masks.

    Args:
        raw_boxes: raw boxes object from SAM3 post-processing
        raw_scores: raw scores object from SAM3 post-processing
        raw_masks: raw masks object from SAM3 post-processing

    Returns:
        int: inferred number of detections
    """
    boxes_arr = _to_numpy_safe(raw_boxes)
    scores_arr = _to_numpy_safe(raw_scores)

    if boxes_arr is not None:
        if boxes_arr.ndim == 2 and boxes_arr.shape[1] == 4:
            return int(boxes_arr.shape[0])
        if boxes_arr.ndim == 1 and boxes_arr.size == 4:
            return 1

    if scores_arr is not None:
        scores_arr = scores_arr.reshape(-1)
        if scores_arr.size > 0:
            return int(scores_arr.size)

    if raw_masks is not None:
        if isinstance(raw_masks, list):
            return len(raw_masks)

        masks_arr = _to_numpy_safe(raw_masks)
        if masks_arr is not None and masks_arr.ndim == 3:
            return int(masks_arr.shape[0])

    return 0


def _normalize_boxes_local(raw_boxes, num_detections):
    """
    Convert raw boxes to a stable numpy array of shape (N, 4).

    Args:
        raw_boxes: raw boxes object
        num_detections: expected number of detections

    Returns:
        np.ndarray: shape (N, 4)
    """
    if num_detections <= 0:
        return np.zeros((0, 4), dtype=np.float32)

    if raw_boxes is None:
        return np.zeros((num_detections, 4), dtype=np.float32)

    arr = _to_numpy_safe(raw_boxes).astype(np.float32)

    if arr.ndim == 1 and arr.size == 4:
        arr = arr.reshape(1, 4)

    if arr.ndim != 2 or arr.shape[1] != 4:
        return np.zeros((num_detections, 4), dtype=np.float32)

    if arr.shape[0] < num_detections:
        pad = np.zeros((num_detections - arr.shape[0], 4), dtype=np.float32)
        arr = np.vstack([arr, pad])

    return arr[:num_detections]


def _normalize_scores_local(raw_scores, num_detections):
    """
    Convert raw scores to a stable numpy array of shape (N,).

    Args:
        raw_scores: raw scores object
        num_detections: expected number of detections

    Returns:
        np.ndarray: shape (N,)
    """
    if num_detections <= 0:
        return np.zeros((0,), dtype=np.float32)

    if raw_scores is None:
        return np.zeros((num_detections,), dtype=np.float32)

    arr = _to_numpy_safe(raw_scores).astype(np.float32).reshape(-1)

    if arr.size < num_detections:
        pad = np.zeros((num_detections - arr.size,), dtype=np.float32)
        arr = np.concatenate([arr, pad])

    return arr[:num_detections]
# -----------------------------------------------------------------------------
# 3. Choose one debug image
# -----------------------------------------------------------------------------
if DEBUG_ROW_INDEX >= len(debug_images_df):
    raise IndexError(
        f"DEBUG_ROW_INDEX={DEBUG_ROW_INDEX} is out of range for "
        f"debug_images_df with {len(debug_images_df)} rows."
    )

row = debug_images_df.iloc[DEBUG_ROW_INDEX]

image_path = row["image_path"]

image_id = row.get("image_id", None)
if pd.isna(image_id):
    image_id = None

file_name = row.get("file_name", None)
if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
    file_name = os.path.basename(image_path)

if not isinstance(image_path, str) or len(image_path.strip()) == 0:
    raise ValueError(f"Invalid image_path: {image_path}")

if not os.path.exists(image_path):
    raise FileNotFoundError(f"Image file not found: {image_path}")

# -----------------------------------------------------------------------------
# 4. Load image
# -----------------------------------------------------------------------------
with Image.open(image_path) as img:
    original_mode = img.mode
    if original_mode != "RGB":
        image = img.convert("RGB")
    else:
        image = img.copy()
    image.load()

image_w, image_h = image.size
image_cx = image_w / 2.0
image_cy = image_h / 2.0
image_diag = math.sqrt(image_w**2 + image_h**2)

print("Pole-detection prototype:\n")
print(f"  DEBUG_ROW_INDEX : {DEBUG_ROW_INDEX}")
print(f"  image_id        : {image_id}")
print(f"  file_name       : {file_name}")
print(f"  image_path      : {image_path}")
print(f"  image_size      : {image_w} x {image_h}")
print(f"  RUN_DEVICE      : {RUN_DEVICE}")
print(f"  TEXT_THRESHOLD  : {TEXT_THRESHOLD}")
print(f"  MASK_THRESHOLD  : {MASK_THRESHOLD}")
print("  POLE_PROMPTS    :")
for p in POLE_PROMPTS:
    print(f"    - {p}")

# -----------------------------------------------------------------------------
# 5. Run all pole prompts and collect candidates
# -----------------------------------------------------------------------------
candidate_rows = []

for prompt_text in POLE_PROMPTS:
    inputs = processor(
        images=image,
        text=prompt_text,
        return_tensors="pt",
    ).to(RUN_DEVICE)

    target_sizes = inputs["original_sizes"].detach().cpu().tolist()

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=TEXT_THRESHOLD,
        mask_threshold=MASK_THRESHOLD,
        target_sizes=target_sizes,
    )

    result = results[0] if isinstance(results, list) and len(results) > 0 else {}

    raw_boxes = result.get("boxes", None)
    raw_scores = result.get("scores", None)
    raw_masks = result.get("masks", None)

    num_detections = _infer_num_detections(raw_boxes, raw_scores, raw_masks)
    boxes = _normalize_boxes_local(raw_boxes, num_detections)
    scores = _normalize_scores_local(raw_scores, num_detections)

    for det_idx in range(num_detections):
        x1, y1, x2, y2 = [float(v) for v in boxes[det_idx]]

        bbox_w = max(0.0, x2 - x1)
        bbox_h = max(0.0, y2 - y1)
        cx = x1 + bbox_w / 2.0
        cy = y1 + bbox_h / 2.0
        area = bbox_w * bbox_h
        vertical_ratio = bbox_h / max(bbox_w, 1e-6)
        center_dist = math.sqrt((cx - image_cx) ** 2 + (cy - image_cy) ** 2)
        center_dist_norm = center_dist / max(image_diag, 1e-6)
        is_vertical = bool(bbox_h > bbox_w)

        candidate_rows.append({
            "prompt_text": prompt_text,
            "det_idx": int(det_idx),
            "score": float(scores[det_idx]),
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "bbox_w": bbox_w,
            "bbox_h": bbox_h,
            "cx": cx,
            "cy": cy,
            "area": area,
            "vertical_ratio": vertical_ratio,
            "center_dist_norm": center_dist_norm,
            "is_vertical": is_vertical,
        })

POLE_CANDIDATES_DF = pd.DataFrame(candidate_rows)

if POLE_CANDIDATES_DF.empty:
    raise ValueError(
        "No pole candidates were returned by SAM3 for the current debug image.\n"
        "Try adding more pole prompts or lowering the text threshold slightly."
    )

# -----------------------------------------------------------------------------
# 6. Rank candidates and choose one best pole box
# -----------------------------------------------------------------------------
# EXPLANATION:
# We gently prefer:
# - vertical boxes
# - higher score
# - taller boxes
# - boxes closer to the image centre
#
# This is still a first prototype, not the final production rule.
POLE_CANDIDATES_DF = POLE_CANDIDATES_DF.sort_values(
    by=["is_vertical", "score", "bbox_h", "center_dist_norm"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

best_row = POLE_CANDIDATES_DF.iloc[0]

POLE_DETECTION_RESULT = {
    "debug_row_index": int(DEBUG_ROW_INDEX),
    "image_id": image_id,
    "file_name": file_name,
    "image_path": image_path,
    "image_w": int(image_w),
    "image_h": int(image_h),
    "prompt_text": str(best_row["prompt_text"]),
    "det_idx": int(best_row["det_idx"]),
    "score": float(best_row["score"]),
    "x1": float(best_row["x1"]),
    "y1": float(best_row["y1"]),
    "x2": float(best_row["x2"]),
    "y2": float(best_row["y2"]),
    "bbox_w": float(best_row["bbox_w"]),
    "bbox_h": float(best_row["bbox_h"]),
    "cx": float(best_row["cx"]),
    "cy": float(best_row["cy"]),
    "vertical_ratio": float(best_row["vertical_ratio"]),
    "center_dist_norm": float(best_row["center_dist_norm"]),
    "is_vertical": bool(best_row["is_vertical"]),
}

print("\nSelected pole detection:")
for k, v in POLE_DETECTION_RESULT.items():
    if k == "image_path":
        print(f"  {k:<18}: {v}")
    else:
        print(f"  {k:<18}: {v}")

# -----------------------------------------------------------------------------
# 7. Visualize all candidates + selected pole box
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(image)

# Show all candidates first
for _, cand in POLE_CANDIDATES_DF.iterrows():
    x1 = float(cand["x1"])
    y1 = float(cand["y1"])
    x2 = float(cand["x2"])
    y2 = float(cand["y2"])

    rect = patches.Rectangle(
        (x1, y1),
        max(0.0, x2 - x1),
        max(0.0, y2 - y1),
        linewidth=1.4,
        edgecolor="cyan",
        linestyle="--",
        facecolor="none",
        alpha=0.55,
    )
    ax.add_patch(rect)

# Show selected box on top
x1 = POLE_DETECTION_RESULT["x1"]
y1 = POLE_DETECTION_RESULT["y1"]
x2 = POLE_DETECTION_RESULT["x2"]
y2 = POLE_DETECTION_RESULT["y2"]

selected_rect = patches.Rectangle(
    (x1, y1),
    max(0.0, x2 - x1),
    max(0.0, y2 - y1),
    linewidth=2.8,
    edgecolor="red",
    facecolor="none",
)
ax.add_patch(selected_rect)

ax.text(
    x1,
    max(5, y1 - 6),
    f"POLE | {POLE_DETECTION_RESULT['prompt_text']} | score={POLE_DETECTION_RESULT['score']:.3f}",
    color="white",
    fontsize=9,
    bbox=dict(facecolor="red", alpha=0.95, pad=1.5, edgecolor="none"),
)

ax.set_title(
    f"Cell 7a — Pole Detection\n{file_name}",
    fontsize=12
)
ax.axis("off")
plt.show()
plt.close()

# -----------------------------------------------------------------------------
# 8. Candidate table
# -----------------------------------------------------------------------------
print("\nPole candidates table:")
try:
    display(
        POLE_CANDIDATES_DF[
            [
                "prompt_text",
                "det_idx",
                "score",
                "bbox_w",
                "bbox_h",
                "vertical_ratio",
                "center_dist_norm",
                "is_vertical",
                "x1",
                "y1",
                "x2",
                "y2",
            ]
        ]
    )
except NameError:
    print(
        POLE_CANDIDATES_DF[
            [
                "prompt_text",
                "det_idx",
                "score",
                "bbox_w",
                "bbox_h",
                "vertical_ratio",
                "center_dist_norm",
                "is_vertical",
                "x1",
                "y1",
                "x2",
                "y2",
            ]
        ]
    )

print("\nCell 7a completed successfully.")
print("Saved globals:")
print("  - POLE_CANDIDATES_DF")
print("  - POLE_DETECTION_RESULT")

In [ ]:
# =============================================================================
# CELL 7a — BATCH POLE DETECTION ON ALL DEBUG IMAGES
#            + POWER POLE ONLY
#            + EXPANDED POLE ROI BOX
# =============================================================================
# EXPLANATION:
# This cell runs SAM3 pole detection on ALL images in debug_images_df using
# only the prompt:
#     "power pole"
#
# PURPOSE:
# - test whether "power pole" is the best single prompt for pole localization
# - keep the raw SAM3 pole bbox
# - build a larger expanded ROI box around that detected pole
# - visualize both boxes across multiple images
#
# VISUAL STYLE:
# - raw detected pole bbox      = red solid box
# - expanded pole ROI bbox      = cyan dashed box
#
# IMPORTANT:
# - this cell does NOT crop yet
# - this cell does NOT rerun crossarm detection yet
# - this is a batch QA / debugging cell
#
# OUTPUTS:
# - pole_batch_results_df : one row per debug image
# - pole_success_df       : successful detections only
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
required_globals = [
    "debug_images_df",
    "model",
    "processor",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 7a requires objects from earlier cells.\n"
        "Please run Cells 5b and 6a/6b first.\n"
        f"Missing globals: {missing_globals}"
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError("debug_images_df exists but is not a pandas DataFrame.")

if debug_images_df.empty:
    raise ValueError("debug_images_df is empty.")

# -----------------------------------------------------------------------------
# 1. Config
# -----------------------------------------------------------------------------
POWER_POLE_PROMPT = "power pole"

TEXT_THRESHOLD = float(globals().get("TEXT_SCORE_THRESHOLD", 0.30))
MASK_THRESHOLD = float(globals().get("MASK_THRESHOLD", 0.50))
RUN_DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------------------------------------------------
# 1b. Expanded ROI box settings
# -----------------------------------------------------------------------------
# EXPLANATION:
# The raw pole box is usually too tight for downstream crossarm work.
# So we build a larger ROI box around the detected pole center.
#
# Strategy:
# - use POLE HEIGHT as the main size reference
# - make the ROI wider than the pole so crossarms can fit
# - make the ROI taller than the raw pole box
#
# These are first-pass tuning values.
EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT = 0.95
MIN_EXPANDED_BOX_WIDTH = 800

TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT = 0.20
BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT = 0.05

MIN_TOP_EXTRA_PIXELS = 120
MIN_BOTTOM_EXTRA_PIXELS = 10
# -----------------------------------------------------------------------------
# 2. Grid display settings
# -----------------------------------------------------------------------------
N_COLS = 3
PANEL_W = 4.8
PANEL_H = 3.8
GRID_DPI = 120

# -----------------------------------------------------------------------------
# 3. Local helpers
# -----------------------------------------------------------------------------
def _to_numpy_safe(x):
    """
    Convert tensors / lists / arrays to a numpy array on CPU.

    Args:
        x:
            torch tensor, numpy array, list, tuple, or None

    Returns:
        np.ndarray or None
    """
    if x is None:
        return None

    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()

    return np.asarray(x)


def _infer_num_detections(raw_boxes, raw_scores):
    """
    Infer number of detections from boxes or scores.

    Args:
        raw_boxes:
            raw boxes object from SAM3 post-processing
        raw_scores:
            raw scores object from SAM3 post-processing

    Returns:
        int
    """
    boxes_arr = _to_numpy_safe(raw_boxes)
    scores_arr = _to_numpy_safe(raw_scores)

    if boxes_arr is not None:
        if boxes_arr.ndim == 2 and boxes_arr.shape[1] == 4:
            return int(boxes_arr.shape[0])
        if boxes_arr.ndim == 1 and boxes_arr.size == 4:
            return 1

    if scores_arr is not None:
        scores_arr = scores_arr.reshape(-1)
        if scores_arr.size > 0:
            return int(scores_arr.size)

    return 0


def _normalize_boxes_local(raw_boxes, num_detections):
    """
    Convert raw boxes to numpy shape (N, 4).

    Args:
        raw_boxes:
            raw boxes object
        num_detections:
            expected number of detections

    Returns:
        np.ndarray
    """
    if num_detections <= 0:
        return np.zeros((0, 4), dtype=np.float32)

    if raw_boxes is None:
        return np.zeros((num_detections, 4), dtype=np.float32)

    arr = _to_numpy_safe(raw_boxes).astype(np.float32)

    if arr.ndim == 1 and arr.size == 4:
        arr = arr.reshape(1, 4)

    if arr.ndim != 2 or arr.shape[1] != 4:
        return np.zeros((num_detections, 4), dtype=np.float32)

    if arr.shape[0] < num_detections:
        pad = np.zeros((num_detections - arr.shape[0], 4), dtype=np.float32)
        arr = np.vstack([arr, pad])

    return arr[:num_detections]


def _normalize_scores_local(raw_scores, num_detections):
    """
    Convert raw scores to numpy shape (N,).

    Args:
        raw_scores:
            raw scores object
        num_detections:
            expected number of detections

    Returns:
        np.ndarray
    """
    if num_detections <= 0:
        return np.zeros((0,), dtype=np.float32)

    if raw_scores is None:
        return np.zeros((num_detections,), dtype=np.float32)

    arr = _to_numpy_safe(raw_scores).astype(np.float32).reshape(-1)

    if arr.size < num_detections:
        pad = np.zeros((num_detections - arr.size,), dtype=np.float32)
        arr = np.concatenate([arr, pad])

    return arr[:num_detections]


def _build_expanded_box_from_pole(
    pole_x1,
    pole_y1,
    pole_x2,
    pole_y2,
    image_w,
    image_h,
    width_factor_from_pole_height,
    min_box_width,
    top_extra_factor_from_pole_height,
    bottom_extra_factor_from_pole_height,
    min_top_extra_pixels,
    min_bottom_extra_pixels,
):
    """
    Build a larger ROI box around the detected pole, with separate control
    for how much the box extends ABOVE and BELOW the pole.

    Args:
        pole_x1, pole_y1, pole_x2, pole_y2:
            raw pole bbox coords
        image_w, image_h:
            full image size
        width_factor_from_pole_height:
            ROI width multiplier using pole height
        min_box_width:
            lower bound for ROI width
        top_extra_factor_from_pole_height:
            extra height above pole top
        bottom_extra_factor_from_pole_height:
            extra height below pole bottom
        min_top_extra_pixels, min_bottom_extra_pixels:
            lower bounds for top/bottom extra height

    Returns:
        dict with expanded box geometry
    """
    pole_w = max(0.0, pole_x2 - pole_x1)
    pole_h = max(0.0, pole_y2 - pole_y1)
    pole_cx = pole_x1 + pole_w / 2.0

    expanded_w = max(
        float(min_box_width),
        float(round(pole_h * width_factor_from_pole_height)),
    )

    top_extra = max(
        float(min_top_extra_pixels),
        float(round(pole_h * top_extra_factor_from_pole_height)),
    )

    bottom_extra = max(
        float(min_bottom_extra_pixels),
        float(round(pole_h * bottom_extra_factor_from_pole_height)),
    )

    half_w = expanded_w / 2.0

    ex1 = pole_cx - half_w
    ex2 = pole_cx + half_w

    # IMPORTANT:
    # top and bottom are now controlled separately
    ey1 = pole_y1 - top_extra
    ey2 = pole_y2 + bottom_extra

    # Clamp to image bounds
    if ex1 < 0:
        ex2 -= ex1
        ex1 = 0.0
    if ex2 > image_w:
        shift = ex2 - image_w
        ex1 -= shift
        ex2 = float(image_w)

    if ey1 < 0:
        ey1 = 0.0
    if ey2 > image_h:
        ey2 = float(image_h)

    ex1 = max(0.0, ex1)
    ex2 = min(float(image_w), ex2)
    ey1 = max(0.0, ey1)
    ey2 = min(float(image_h), ey2)

    final_w = max(0.0, ex2 - ex1)
    final_h = max(0.0, ey2 - ey1)

    return {
        "expanded_x1": float(ex1),
        "expanded_y1": float(ey1),
        "expanded_x2": float(ex2),
        "expanded_y2": float(ey2),
        "expanded_w": float(final_w),
        "expanded_h": float(final_h),
        "pole_cx": float(pole_cx),
    }

# -----------------------------------------------------------------------------
# 4. Choose images and create grid
# -----------------------------------------------------------------------------
grid_df = debug_images_df.copy().reset_index(drop=True)

n_images = len(grid_df)
n_rows = math.ceil(n_images / N_COLS)

print("Batch pole-detection run over debug_images_df:\n")
print(f"  n_images                               : {n_images}")
print(f"  prompt                                 : {POWER_POLE_PROMPT}")
print(f"  TEXT_THRESHOLD                         : {TEXT_THRESHOLD}")
print(f"  MASK_THRESHOLD                         : {MASK_THRESHOLD}")
print(f"  RUN_DEVICE                             : {RUN_DEVICE}")
print(f"  expanded width factor from pole height : {EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT}")
print(f"  expanded height factor from pole height: {EXPANDED_BOX_HEIGHT_FACTOR_FROM_POLE_HEIGHT}")
print(f"  MIN_EXPANDED_BOX_WIDTH                 : {MIN_EXPANDED_BOX_WIDTH}")
print(f"  MIN_EXPANDED_BOX_HEIGHT                : {MIN_EXPANDED_BOX_HEIGHT}")
print(f"  grid_cols                              : {N_COLS}")
print(f"  grid_rows                              : {n_rows}")
print(f"  panel_size                             : {PANEL_W} x {PANEL_H}")

fig, axes = plt.subplots(
    n_rows,
    N_COLS,
    figsize=(N_COLS * PANEL_W, n_rows * PANEL_H),
    dpi=GRID_DPI,
    constrained_layout=True,
)

axes = np.array(axes).reshape(-1)

# -----------------------------------------------------------------------------
# 5. Loop through all debug images
# -----------------------------------------------------------------------------
batch_rows = []

for plot_idx, (_, row) in enumerate(grid_df.iterrows()):
    ax = axes[plot_idx]

    image_path = row["image_path"]

    image_id = row.get("image_id", None)
    if pd.isna(image_id):
        image_id = None

    file_name = row.get("file_name", None)
    if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
        file_name = os.path.basename(image_path)

    try:
        # ---------------------------------------------------------------------
        # 5a. Validate and load image
        # ---------------------------------------------------------------------
        if not isinstance(image_path, str) or len(image_path.strip()) == 0:
            raise ValueError(f"Invalid image_path: {image_path}")

        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image file not found: {image_path}")

        with Image.open(image_path) as img:
            original_mode = img.mode
            if original_mode != "RGB":
                image = img.convert("RGB")
            else:
                image = img.copy()
            image.load()

        image_w, image_h = image.size
        image_cx = image_w / 2.0
        image_cy = image_h / 2.0
        image_diag = math.sqrt(image_w**2 + image_h**2)

        # ---------------------------------------------------------------------
        # 5b. Run SAM3 on one pole prompt only
        # ---------------------------------------------------------------------
        inputs = processor(
            images=image,
            text=POWER_POLE_PROMPT,
            return_tensors="pt",
        ).to(RUN_DEVICE)

        target_sizes = inputs["original_sizes"].detach().cpu().tolist()

        with torch.no_grad():
            outputs = model(**inputs)

        results = processor.post_process_instance_segmentation(
            outputs,
            threshold=TEXT_THRESHOLD,
            mask_threshold=MASK_THRESHOLD,
            target_sizes=target_sizes,
        )

        result = results[0] if isinstance(results, list) and len(results) > 0 else {}

        raw_boxes = result.get("boxes", None)
        raw_scores = result.get("scores", None)

        num_detections = _infer_num_detections(raw_boxes, raw_scores)
        boxes = _normalize_boxes_local(raw_boxes, num_detections)
        scores = _normalize_scores_local(raw_scores, num_detections)

        if num_detections == 0:
            raise ValueError("No pole detections returned.")

        # ---------------------------------------------------------------------
        # 5c. Build candidate table and choose best pole
        # ---------------------------------------------------------------------
        candidate_rows = []

        for det_idx in range(num_detections):
            x1, y1, x2, y2 = [float(v) for v in boxes[det_idx]]

            bbox_w = max(0.0, x2 - x1)
            bbox_h = max(0.0, y2 - y1)
            cx = x1 + bbox_w / 2.0
            cy = y1 + bbox_h / 2.0
            area = bbox_w * bbox_h
            vertical_ratio = bbox_h / max(bbox_w, 1e-6)
            center_dist = math.sqrt((cx - image_cx) ** 2 + (cy - image_cy) ** 2)
            center_dist_norm = center_dist / max(image_diag, 1e-6)
            is_vertical = bool(bbox_h > bbox_w)

            candidate_rows.append({
                "det_idx": int(det_idx),
                "score": float(scores[det_idx]),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "bbox_w": bbox_w,
                "bbox_h": bbox_h,
                "cx": cx,
                "cy": cy,
                "area": area,
                "vertical_ratio": vertical_ratio,
                "center_dist_norm": center_dist_norm,
                "is_vertical": is_vertical,
            })

        candidates_df = pd.DataFrame(candidate_rows)

        candidates_df = candidates_df.sort_values(
            by=["is_vertical", "score", "bbox_h", "center_dist_norm"],
            ascending=[False, False, False, True],
        ).reset_index(drop=True)

        best_row = candidates_df.iloc[0]

        pole_x1 = float(best_row["x1"])
        pole_y1 = float(best_row["y1"])
        pole_x2 = float(best_row["x2"])
        pole_y2 = float(best_row["y2"])
        pole_w = float(best_row["bbox_w"])
        pole_h = float(best_row["bbox_h"])
        pole_score = float(best_row["score"])
        pole_vertical_ratio = float(best_row["vertical_ratio"])

        expanded_box = _build_expanded_box_from_pole(
            pole_x1=pole_x1,
            pole_y1=pole_y1,
            pole_x2=pole_x2,
            pole_y2=pole_y2,
            image_w=image_w,
            image_h=image_h,
            width_factor_from_pole_height=EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT,
            min_box_width=MIN_EXPANDED_BOX_WIDTH,
            top_extra_factor_from_pole_height=TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT,
            bottom_extra_factor_from_pole_height=BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT,
            min_top_extra_pixels=MIN_TOP_EXTRA_PIXELS,
            min_bottom_extra_pixels=MIN_BOTTOM_EXTRA_PIXELS,
        )

        # ---------------------------------------------------------------------
        # 5d. Draw image + raw pole box + expanded ROI box
        # ---------------------------------------------------------------------
        ax.imshow(image)

        raw_rect = patches.Rectangle(
            (pole_x1, pole_y1),
            pole_w,
            pole_h,
            linewidth=2.4,
            edgecolor="red",
            facecolor="none",
        )
        ax.add_patch(raw_rect)

        expanded_rect = patches.Rectangle(
            (expanded_box["expanded_x1"], expanded_box["expanded_y1"]),
            expanded_box["expanded_w"],
            expanded_box["expanded_h"],
            linewidth=2.0,
            edgecolor="cyan",
            linestyle="--",
            facecolor="none",
            alpha=0.95,
        )
        ax.add_patch(expanded_rect)

        ax.text(
            pole_x1,
            max(5, pole_y1 - 6),
            f"POLE | {POWER_POLE_PROMPT} | score={pole_score:.3f}",
            color="white",
            fontsize=8,
            bbox=dict(facecolor="red", alpha=0.95, pad=1.3, edgecolor="none"),
        )

        ax.set_title(
            f"{file_name}\nDetections={num_detections}",
            fontsize=9,
            pad=8,
        )
        ax.axis("off")

        # ---------------------------------------------------------------------
        # 5e. Save summary row
        # ---------------------------------------------------------------------
        batch_rows.append({
            "debug_row_index": int(plot_idx),
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": original_mode,
            "image_w": int(image_w),
            "image_h": int(image_h),
            "prompt_text": POWER_POLE_PROMPT,
            "num_detections": int(num_detections),
            "selected_det_idx": int(best_row["det_idx"]),
            "selected_score": pole_score,
            "selected_is_vertical": bool(best_row["is_vertical"]),
            "selected_vertical_ratio": pole_vertical_ratio,
            "selected_center_dist_norm": float(best_row["center_dist_norm"]),
            "pole_x1": pole_x1,
            "pole_y1": pole_y1,
            "pole_x2": pole_x2,
            "pole_y2": pole_y2,
            "pole_w": pole_w,
            "pole_h": pole_h,
            "expanded_x1": expanded_box["expanded_x1"],
            "expanded_y1": expanded_box["expanded_y1"],
            "expanded_x2": expanded_box["expanded_x2"],
            "expanded_y2": expanded_box["expanded_y2"],
            "expanded_w": expanded_box["expanded_w"],
            "expanded_h": expanded_box["expanded_h"],
            "pole_cx": expanded_box["pole_cx"],
            "pole_cy": expanded_box["pole_cy"],
            "run_status": "ok",
            "error_message": "",
        })

        del image

    except Exception as e:
        ax.set_facecolor("white")
        ax.text(
            0.5,
            0.5,
            f"FAILED\n{file_name}\n\n{type(e).__name__}\n{str(e)}",
            ha="center",
            va="center",
            fontsize=9,
            wrap=True,
            transform=ax.transAxes,
        )
        ax.set_title(f"{file_name}\nFAILED", fontsize=9, pad=8)
        ax.axis("off")

        batch_rows.append({
            "debug_row_index": int(plot_idx),
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": None,
            "image_w": np.nan,
            "image_h": np.nan,
            "prompt_text": POWER_POLE_PROMPT,
            "num_detections": np.nan,
            "selected_det_idx": np.nan,
            "selected_score": np.nan,
            "selected_is_vertical": np.nan,
            "selected_vertical_ratio": np.nan,
            "selected_center_dist_norm": np.nan,
            "pole_x1": np.nan,
            "pole_y1": np.nan,
            "pole_x2": np.nan,
            "pole_y2": np.nan,
            "pole_w": np.nan,
            "pole_h": np.nan,
            "expanded_x1": np.nan,
            "expanded_y1": np.nan,
            "expanded_x2": np.nan,
            "expanded_y2": np.nan,
            "expanded_w": np.nan,
            "expanded_h": np.nan,
            "pole_cx": np.nan,
            "pole_cy": np.nan,
            "run_status": "failed",
            "error_message": str(e),
        })

# -----------------------------------------------------------------------------
# 6. Hide unused axes
# -----------------------------------------------------------------------------
for ax in axes[n_images:]:
    ax.axis("off")

# -----------------------------------------------------------------------------
# 7. Global title
# -----------------------------------------------------------------------------
fig.suptitle(
    f"Cell 7a — Batch Pole Detection | Prompt: {POWER_POLE_PROMPT}",
    fontsize=14,
)

plt.show()
plt.close()

# -----------------------------------------------------------------------------
# 8. Build output tables
# -----------------------------------------------------------------------------
pole_batch_results_df = pd.DataFrame(batch_rows)

pole_success_df = pole_batch_results_df[
    pole_batch_results_df["run_status"] == "ok"
].copy().reset_index(drop=True)

print("\nPole batch summary table:")
try:
    display(pole_batch_results_df)
except NameError:
    print(pole_batch_results_df)

if len(pole_success_df) > 0:
    print("\nSuccessful pole detections only:")
    cols_to_show = [
        "file_name",
        "num_detections",
        "selected_score",
        "pole_w",
        "pole_h",
        "selected_vertical_ratio",
        "expanded_w",
        "expanded_h",
    ]
    try:
        display(pole_success_df[cols_to_show])
    except NameError:
        print(pole_success_df[cols_to_show])

print("\nCell 7a completed successfully.")
print("Saved globals:")
print("  - pole_batch_results_df")
print("  - pole_success_df")

In [ ]:
# =============================================================================
# CELL 7a — BATCH POLE DETECTION ON ALL DEBUG IMAGES
#            + POWER POLE ONLY
#            + EXPANDED POLE ROI BOX
# =============================================================================
# EXPLANATION:
# This cell runs SAM3 pole detection on ALL images in debug_images_df using
# only the prompt:
#     "power pole"
#
# PURPOSE:
# - test whether "power pole" is the best single prompt for pole localization
# - keep the raw SAM3 pole bbox
# - build a larger expanded ROI box around that detected pole
# - visualize both boxes across multiple images
#
# VISUAL STYLE:
# - raw detected pole bbox      = red solid box
# - expanded pole ROI bbox      = cyan dashed box
#
# IMPORTANT:
# - this cell does NOT crop yet
# - this cell does NOT rerun crossarm detection yet
# - this is a batch QA / debugging cell
#
# OUTPUTS:
# - pole_batch_results_df : one row per debug image
# - pole_success_df       : successful detections only
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
required_globals = [
    "debug_images_df",
    "model",
    "processor",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 7a requires objects from earlier cells.\n"
        "Please run Cells 5b and 6a/6b first.\n"
        f"Missing globals: {missing_globals}"
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError("debug_images_df exists but is not a pandas DataFrame.")

if debug_images_df.empty:
    raise ValueError("debug_images_df is empty.")

# -----------------------------------------------------------------------------
# 1. Config
# -----------------------------------------------------------------------------
POWER_POLE_PROMPT = "power pole"

TEXT_THRESHOLD = float(globals().get("TEXT_SCORE_THRESHOLD", 0.30))
MASK_THRESHOLD = float(globals().get("MASK_THRESHOLD", 0.50))
RUN_DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------------------------------------------------
# 1b. Expanded ROI box settings
# -----------------------------------------------------------------------------
# EXPLANATION:
# The raw pole box is usually too tight for downstream crossarm work.
# So we build a larger ROI box around the detected pole.
#
# Strategy:
# - width is controlled from pole height
# - top and bottom expansion are controlled separately
EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT = 0.95
MIN_EXPANDED_BOX_WIDTH = 800

TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT = 0.20
BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT = 0.45

MIN_TOP_EXTRA_PIXELS = 120
MIN_BOTTOM_EXTRA_PIXELS = 160

# -----------------------------------------------------------------------------
# 2. Grid display settings
# -----------------------------------------------------------------------------
N_COLS = 3
PANEL_W = 4.8
PANEL_H = 3.8
GRID_DPI = 120

# -----------------------------------------------------------------------------
# 3. Local helpers
# -----------------------------------------------------------------------------
def _to_numpy_safe(x):
    """
    Convert tensors / lists / arrays to a numpy array on CPU.

    Args:
        x:
            torch tensor, numpy array, list, tuple, or None

    Returns:
        np.ndarray or None
    """
    if x is None:
        return None

    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()

    return np.asarray(x)


def _infer_num_detections(raw_boxes, raw_scores):
    """
    Infer number of detections from boxes or scores.

    Args:
        raw_boxes:
            raw boxes object from SAM3 post-processing
        raw_scores:
            raw scores object from SAM3 post-processing

    Returns:
        int
    """
    boxes_arr = _to_numpy_safe(raw_boxes)
    scores_arr = _to_numpy_safe(raw_scores)

    if boxes_arr is not None:
        if boxes_arr.ndim == 2 and boxes_arr.shape[1] == 4:
            return int(boxes_arr.shape[0])
        if boxes_arr.ndim == 1 and boxes_arr.size == 4:
            return 1

    if scores_arr is not None:
        scores_arr = scores_arr.reshape(-1)
        if scores_arr.size > 0:
            return int(scores_arr.size)

    return 0


def _normalize_boxes_local(raw_boxes, num_detections):
    """
    Convert raw boxes to numpy shape (N, 4).

    Args:
        raw_boxes:
            raw boxes object
        num_detections:
            expected number of detections

    Returns:
        np.ndarray
    """
    if num_detections <= 0:
        return np.zeros((0, 4), dtype=np.float32)

    if raw_boxes is None:
        return np.zeros((num_detections, 4), dtype=np.float32)

    arr = _to_numpy_safe(raw_boxes).astype(np.float32)

    if arr.ndim == 1 and arr.size == 4:
        arr = arr.reshape(1, 4)

    if arr.ndim != 2 or arr.shape[1] != 4:
        return np.zeros((num_detections, 4), dtype=np.float32)

    if arr.shape[0] < num_detections:
        pad = np.zeros((num_detections - arr.shape[0], 4), dtype=np.float32)
        arr = np.vstack([arr, pad])

    return arr[:num_detections]


def _normalize_scores_local(raw_scores, num_detections):
    """
    Convert raw scores to numpy shape (N,).

    Args:
        raw_scores:
            raw scores object
        num_detections:
            expected number of detections

    Returns:
        np.ndarray
    """
    if num_detections <= 0:
        return np.zeros((0,), dtype=np.float32)

    if raw_scores is None:
        return np.zeros((num_detections,), dtype=np.float32)

    arr = _to_numpy_safe(raw_scores).astype(np.float32).reshape(-1)

    if arr.size < num_detections:
        pad = np.zeros((num_detections - arr.size,), dtype=np.float32)
        arr = np.concatenate([arr, pad])

    return arr[:num_detections]


def _build_expanded_box_from_pole(
    pole_x1,
    pole_y1,
    pole_x2,
    pole_y2,
    image_w,
    image_h,
    width_factor_from_pole_height,
    min_box_width,
    top_extra_factor_from_pole_height,
    bottom_extra_factor_from_pole_height,
    min_top_extra_pixels,
    min_bottom_extra_pixels,
):
    """
    Build a larger ROI box around the detected pole, with separate control
    for how much the box extends ABOVE and BELOW the pole.

    Args:
        pole_x1, pole_y1, pole_x2, pole_y2:
            raw pole bbox coords
        image_w, image_h:
            full image size
        width_factor_from_pole_height:
            ROI width multiplier using pole height
        min_box_width:
            lower bound for ROI width
        top_extra_factor_from_pole_height:
            extra height above pole top
        bottom_extra_factor_from_pole_height:
            extra height below pole bottom
        min_top_extra_pixels, min_bottom_extra_pixels:
            lower bounds for top/bottom extra height

    Returns:
        dict with expanded box geometry
    """
    pole_w = max(0.0, pole_x2 - pole_x1)
    pole_h = max(0.0, pole_y2 - pole_y1)
    pole_cx = pole_x1 + pole_w / 2.0
    pole_cy = pole_y1 + pole_h / 2.0

    expanded_w = max(
        float(min_box_width),
        float(round(pole_h * width_factor_from_pole_height)),
    )

    top_extra = max(
        float(min_top_extra_pixels),
        float(round(pole_h * top_extra_factor_from_pole_height)),
    )

    bottom_extra = max(
        float(min_bottom_extra_pixels),
        float(round(pole_h * bottom_extra_factor_from_pole_height)),
    )

    half_w = expanded_w / 2.0

    ex1 = pole_cx - half_w
    ex2 = pole_cx + half_w

    # IMPORTANT:
    # top and bottom are controlled separately
    ey1 = pole_y1 - top_extra
    ey2 = pole_y2 + bottom_extra

    # Clamp to image bounds
    if ex1 < 0:
        ex2 -= ex1
        ex1 = 0.0
    if ex2 > image_w:
        shift = ex2 - image_w
        ex1 -= shift
        ex2 = float(image_w)

    if ey1 < 0:
        ey1 = 0.0
    if ey2 > image_h:
        ey2 = float(image_h)

    ex1 = max(0.0, ex1)
    ex2 = min(float(image_w), ex2)
    ey1 = max(0.0, ey1)
    ey2 = min(float(image_h), ey2)

    final_w = max(0.0, ex2 - ex1)
    final_h = max(0.0, ey2 - ey1)

    return {
        "expanded_x1": float(ex1),
        "expanded_y1": float(ey1),
        "expanded_x2": float(ex2),
        "expanded_y2": float(ey2),
        "expanded_w": float(final_w),
        "expanded_h": float(final_h),
        "pole_cx": float(pole_cx),
        "pole_cy": float(pole_cy),
    }

# -----------------------------------------------------------------------------
# 4. Choose images and create grid
# -----------------------------------------------------------------------------
grid_df = debug_images_df.copy().reset_index(drop=True)

n_images = len(grid_df)
n_rows = math.ceil(n_images / N_COLS)

print("Batch pole-detection run over debug_images_df:\n")
print(f"  n_images                               : {n_images}")
print(f"  prompt                                 : {POWER_POLE_PROMPT}")
print(f"  TEXT_THRESHOLD                         : {TEXT_THRESHOLD}")
print(f"  MASK_THRESHOLD                         : {MASK_THRESHOLD}")
print(f"  RUN_DEVICE                             : {RUN_DEVICE}")
print(f"  expanded width factor from pole height : {EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT}")
print(f"  MIN_EXPANDED_BOX_WIDTH                 : {MIN_EXPANDED_BOX_WIDTH}")
print(f"  TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT      : {TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT}")
print(f"  BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT   : {BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT}")
print(f"  MIN_TOP_EXTRA_PIXELS                   : {MIN_TOP_EXTRA_PIXELS}")
print(f"  MIN_BOTTOM_EXTRA_PIXELS                : {MIN_BOTTOM_EXTRA_PIXELS}")
print(f"  grid_cols                              : {N_COLS}")
print(f"  grid_rows                              : {n_rows}")
print(f"  panel_size                             : {PANEL_W} x {PANEL_H}")

fig, axes = plt.subplots(
    n_rows,
    N_COLS,
    figsize=(N_COLS * PANEL_W, n_rows * PANEL_H),
    dpi=GRID_DPI,
    constrained_layout=True,
)

axes = np.array(axes).reshape(-1)

# -----------------------------------------------------------------------------
# 5. Loop through all debug images
# -----------------------------------------------------------------------------
batch_rows = []

for plot_idx, (_, row) in enumerate(grid_df.iterrows()):
    ax = axes[plot_idx]

    image_path = row["image_path"]

    image_id = row.get("image_id", None)
    if pd.isna(image_id):
        image_id = None

    file_name = row.get("file_name", None)
    if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
        file_name = os.path.basename(image_path)

    try:
        # ---------------------------------------------------------------------
        # 5a. Validate and load image
        # ---------------------------------------------------------------------
        if not isinstance(image_path, str) or len(image_path.strip()) == 0:
            raise ValueError(f"Invalid image_path: {image_path}")

        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image file not found: {image_path}")

        with Image.open(image_path) as img:
            original_mode = img.mode
            if original_mode != "RGB":
                image = img.convert("RGB")
            else:
                image = img.copy()
            image.load()

        image_w, image_h = image.size
        image_cx = image_w / 2.0
        image_cy = image_h / 2.0
        image_diag = math.sqrt(image_w**2 + image_h**2)

        # ---------------------------------------------------------------------
        # 5b. Run SAM3 on one pole prompt only
        # ---------------------------------------------------------------------
        inputs = processor(
            images=image,
            text=POWER_POLE_PROMPT,
            return_tensors="pt",
        ).to(RUN_DEVICE)

        target_sizes = inputs["original_sizes"].detach().cpu().tolist()

        with torch.no_grad():
            outputs = model(**inputs)

        results = processor.post_process_instance_segmentation(
            outputs,
            threshold=TEXT_THRESHOLD,
            mask_threshold=MASK_THRESHOLD,
            target_sizes=target_sizes,
        )

        result = results[0] if isinstance(results, list) and len(results) > 0 else {}

        raw_boxes = result.get("boxes", None)
        raw_scores = result.get("scores", None)

        num_detections = _infer_num_detections(raw_boxes, raw_scores)
        boxes = _normalize_boxes_local(raw_boxes, num_detections)
        scores = _normalize_scores_local(raw_scores, num_detections)

        if num_detections == 0:
            raise ValueError("No pole detections returned.")

        # ---------------------------------------------------------------------
        # 5c. Build candidate table and choose best pole
        # ---------------------------------------------------------------------
        candidate_rows = []

        for det_idx in range(num_detections):
            x1, y1, x2, y2 = [float(v) for v in boxes[det_idx]]

            bbox_w = max(0.0, x2 - x1)
            bbox_h = max(0.0, y2 - y1)
            cx = x1 + bbox_w / 2.0
            cy = y1 + bbox_h / 2.0
            area = bbox_w * bbox_h
            vertical_ratio = bbox_h / max(bbox_w, 1e-6)
            center_dist = math.sqrt((cx - image_cx) ** 2 + (cy - image_cy) ** 2)
            center_dist_norm = center_dist / max(image_diag, 1e-6)
            is_vertical = bool(bbox_h > bbox_w)

            candidate_rows.append({
                "det_idx": int(det_idx),
                "score": float(scores[det_idx]),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "bbox_w": bbox_w,
                "bbox_h": bbox_h,
                "cx": cx,
                "cy": cy,
                "area": area,
                "vertical_ratio": vertical_ratio,
                "center_dist_norm": center_dist_norm,
                "is_vertical": is_vertical,
            })

        candidates_df = pd.DataFrame(candidate_rows)

        candidates_df = candidates_df.sort_values(
            by=["is_vertical", "score", "bbox_h", "center_dist_norm"],
            ascending=[False, False, False, True],
        ).reset_index(drop=True)

        best_row = candidates_df.iloc[0]

        pole_x1 = float(best_row["x1"])
        pole_y1 = float(best_row["y1"])
        pole_x2 = float(best_row["x2"])
        pole_y2 = float(best_row["y2"])
        pole_w = float(best_row["bbox_w"])
        pole_h = float(best_row["bbox_h"])
        pole_score = float(best_row["score"])
        pole_vertical_ratio = float(best_row["vertical_ratio"])

        expanded_box = _build_expanded_box_from_pole(
            pole_x1=pole_x1,
            pole_y1=pole_y1,
            pole_x2=pole_x2,
            pole_y2=pole_y2,
            image_w=image_w,
            image_h=image_h,
            width_factor_from_pole_height=EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT,
            min_box_width=MIN_EXPANDED_BOX_WIDTH,
            top_extra_factor_from_pole_height=TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT,
            bottom_extra_factor_from_pole_height=BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT,
            min_top_extra_pixels=MIN_TOP_EXTRA_PIXELS,
            min_bottom_extra_pixels=MIN_BOTTOM_EXTRA_PIXELS,
        )

        # ---------------------------------------------------------------------
        # 5d. Draw image + raw pole box + expanded ROI box
        # ---------------------------------------------------------------------
        ax.imshow(image)

        raw_rect = patches.Rectangle(
            (pole_x1, pole_y1),
            pole_w,
            pole_h,
            linewidth=2.4,
            edgecolor="red",
            facecolor="none",
        )
        ax.add_patch(raw_rect)

        expanded_rect = patches.Rectangle(
            (expanded_box["expanded_x1"], expanded_box["expanded_y1"]),
            expanded_box["expanded_w"],
            expanded_box["expanded_h"],
            linewidth=2.0,
            edgecolor="cyan",
            linestyle="--",
            facecolor="none",
            alpha=0.95,
        )
        ax.add_patch(expanded_rect)

        ax.text(
            pole_x1,
            max(5, pole_y1 - 6),
            f"POLE | {POWER_POLE_PROMPT} | score={pole_score:.3f}",
            color="white",
            fontsize=8,
            bbox=dict(facecolor="red", alpha=0.95, pad=1.3, edgecolor="none"),
        )

        ax.set_title(
            f"{file_name}\nDetections={num_detections}",
            fontsize=9,
            pad=8,
        )
        ax.axis("off")

        # ---------------------------------------------------------------------
        # 5e. Save summary row
        # ---------------------------------------------------------------------
        batch_rows.append({
            "debug_row_index": int(plot_idx),
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": original_mode,
            "image_w": int(image_w),
            "image_h": int(image_h),
            "prompt_text": POWER_POLE_PROMPT,
            "num_detections": int(num_detections),
            "selected_det_idx": int(best_row["det_idx"]),
            "selected_score": pole_score,
            "selected_is_vertical": bool(best_row["is_vertical"]),
            "selected_vertical_ratio": pole_vertical_ratio,
            "selected_center_dist_norm": float(best_row["center_dist_norm"]),
            "pole_x1": pole_x1,
            "pole_y1": pole_y1,
            "pole_x2": pole_x2,
            "pole_y2": pole_y2,
            "pole_w": pole_w,
            "pole_h": pole_h,
            "expanded_x1": expanded_box["expanded_x1"],
            "expanded_y1": expanded_box["expanded_y1"],
            "expanded_x2": expanded_box["expanded_x2"],
            "expanded_y2": expanded_box["expanded_y2"],
            "expanded_w": expanded_box["expanded_w"],
            "expanded_h": expanded_box["expanded_h"],
            "pole_cx": expanded_box["pole_cx"],
            "pole_cy": expanded_box["pole_cy"],
            "run_status": "ok",
            "error_message": "",
        })

        del image

    except Exception as e:
        ax.set_facecolor("white")
        ax.text(
            0.5,
            0.5,
            f"FAILED\n{file_name}\n\n{type(e).__name__}\n{str(e)}",
            ha="center",
            va="center",
            fontsize=9,
            wrap=True,
            transform=ax.transAxes,
        )
        ax.set_title(f"{file_name}\nFAILED", fontsize=9, pad=8)
        ax.axis("off")

        batch_rows.append({
            "debug_row_index": int(plot_idx),
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": None,
            "image_w": np.nan,
            "image_h": np.nan,
            "prompt_text": POWER_POLE_PROMPT,
            "num_detections": np.nan,
            "selected_det_idx": np.nan,
            "selected_score": np.nan,
            "selected_is_vertical": np.nan,
            "selected_vertical_ratio": np.nan,
            "selected_center_dist_norm": np.nan,
            "pole_x1": np.nan,
            "pole_y1": np.nan,
            "pole_x2": np.nan,
            "pole_y2": np.nan,
            "pole_w": np.nan,
            "pole_h": np.nan,
            "expanded_x1": np.nan,
            "expanded_y1": np.nan,
            "expanded_x2": np.nan,
            "expanded_y2": np.nan,
            "expanded_w": np.nan,
            "expanded_h": np.nan,
            "pole_cx": np.nan,
            "pole_cy": np.nan,
            "run_status": "failed",
            "error_message": str(e),
        })

# -----------------------------------------------------------------------------
# 6. Hide unused axes
# -----------------------------------------------------------------------------
for ax in axes[n_images:]:
    ax.axis("off")

# -----------------------------------------------------------------------------
# 7. Global title
# -----------------------------------------------------------------------------
fig.suptitle(
    f"Cell 7a — Batch Pole Detection | Prompt: {POWER_POLE_PROMPT}",
    fontsize=14,
)

plt.show()
plt.close()

# -----------------------------------------------------------------------------
# 8. Build output tables
# -----------------------------------------------------------------------------
pole_batch_results_df = pd.DataFrame(batch_rows)

pole_success_df = pole_batch_results_df[
    pole_batch_results_df["run_status"] == "ok"
].copy().reset_index(drop=True)

print("\nPole batch summary table:")
try:
    display(pole_batch_results_df)
except NameError:
    print(pole_batch_results_df)

if len(pole_success_df) > 0:
    print("\nSuccessful pole detections only:")
    cols_to_show = [
        "file_name",
        "num_detections",
        "selected_score",
        "pole_w",
        "pole_h",
        "selected_vertical_ratio",
        "expanded_w",
        "expanded_h",
    ]
    try:
        display(pole_success_df[cols_to_show])
    except NameError:
        print(pole_success_df[cols_to_show])

print("\nCell 7a completed successfully.")
print("Saved globals:")
print("  - pole_batch_results_df")
print("  - pole_success_df")

In [ ]:
# =============================================================================
# CELL 7a — BATCH POLE DETECTION ON ALL DEBUG IMAGES
#            + POWER POLE ONLY
#            + EXPANDED POLE ROI BOX
# =============================================================================
# EXPLANATION:
# This cell runs SAM3 pole detection on ALL images in debug_images_df using
# only the prompt:
#     "power pole"
#
# PURPOSE:
# - test whether "power pole" is the best single prompt for pole localization
# - keep the raw SAM3 pole bbox
# - build a larger expanded ROI box around that detected pole
# - visualize both boxes across multiple images
#
# VISUAL STYLE:
# - raw detected pole bbox      = red solid box
# - expanded pole ROI bbox      = cyan dashed box
#
# IMPORTANT:
# - this cell does NOT crop yet
# - this cell does NOT rerun crossarm detection yet
# - this is a batch QA / debugging cell
#
# OUTPUTS:
# - pole_batch_results_df : one row per debug image
# - pole_success_df       : successful detections only
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
required_globals = [
    "debug_images_df",
    "model",
    "processor",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 7a requires objects from earlier cells.\n"
        "Please run Cells 5b and 6a/6b first.\n"
        f"Missing globals: {missing_globals}"
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError("debug_images_df exists but is not a pandas DataFrame.")

if debug_images_df.empty:
    raise ValueError("debug_images_df is empty.")

# -----------------------------------------------------------------------------
# 1. Config
# -----------------------------------------------------------------------------
POWER_POLE_PROMPT = "power pole"

TEXT_THRESHOLD = float(globals().get("TEXT_SCORE_THRESHOLD", 0.30))
MASK_THRESHOLD = float(globals().get("MASK_THRESHOLD", 0.50))
RUN_DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------------------------------------------------
# 1b. Expanded ROI box settings
# -----------------------------------------------------------------------------
# EXPLANATION:
# The raw pole box is usually too tight for downstream crossarm work.
# So we build a larger ROI box around the detected pole.
#
# Strategy:
# - width is controlled from pole height
# - top and bottom expansion are controlled separately
EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT = 0.95
MIN_EXPANDED_BOX_WIDTH = 800

TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT = 0.20
BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT = 0.05

MIN_TOP_EXTRA_PIXELS = 120
MIN_BOTTOM_EXTRA_PIXELS = 10

# -----------------------------------------------------------------------------
# 2. Grid display settings
# -----------------------------------------------------------------------------
N_COLS = 3
PANEL_W = 4.8
PANEL_H = 3.8
GRID_DPI = 120

# -----------------------------------------------------------------------------
# 3. Local helpers
# -----------------------------------------------------------------------------
def _to_numpy_safe(x):
    """
    Convert tensors / lists / arrays to a numpy array on CPU.

    Args:
        x:
            torch tensor, numpy array, list, tuple, or None

    Returns:
        np.ndarray or None
    """
    if x is None:
        return None

    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()

    return np.asarray(x)


def _infer_num_detections(raw_boxes, raw_scores):
    """
    Infer number of detections from boxes or scores.

    Args:
        raw_boxes:
            raw boxes object from SAM3 post-processing
        raw_scores:
            raw scores object from SAM3 post-processing

    Returns:
        int
    """
    boxes_arr = _to_numpy_safe(raw_boxes)
    scores_arr = _to_numpy_safe(raw_scores)

    if boxes_arr is not None:
        if boxes_arr.ndim == 2 and boxes_arr.shape[1] == 4:
            return int(boxes_arr.shape[0])
        if boxes_arr.ndim == 1 and boxes_arr.size == 4:
            return 1

    if scores_arr is not None:
        scores_arr = scores_arr.reshape(-1)
        if scores_arr.size > 0:
            return int(scores_arr.size)

    return 0


def _normalize_boxes_local(raw_boxes, num_detections):
    """
    Convert raw boxes to numpy shape (N, 4).

    Args:
        raw_boxes:
            raw boxes object
        num_detections:
            expected number of detections

    Returns:
        np.ndarray
    """
    if num_detections <= 0:
        return np.zeros((0, 4), dtype=np.float32)

    if raw_boxes is None:
        return np.zeros((num_detections, 4), dtype=np.float32)

    arr = _to_numpy_safe(raw_boxes).astype(np.float32)

    if arr.ndim == 1 and arr.size == 4:
        arr = arr.reshape(1, 4)

    if arr.ndim != 2 or arr.shape[1] != 4:
        return np.zeros((num_detections, 4), dtype=np.float32)

    if arr.shape[0] < num_detections:
        pad = np.zeros((num_detections - arr.shape[0], 4), dtype=np.float32)
        arr = np.vstack([arr, pad])

    return arr[:num_detections]


def _normalize_scores_local(raw_scores, num_detections):
    """
    Convert raw scores to numpy shape (N,).

    Args:
        raw_scores:
            raw scores object
        num_detections:
            expected number of detections

    Returns:
        np.ndarray
    """
    if num_detections <= 0:
        return np.zeros((0,), dtype=np.float32)

    if raw_scores is None:
        return np.zeros((num_detections,), dtype=np.float32)

    arr = _to_numpy_safe(raw_scores).astype(np.float32).reshape(-1)

    if arr.size < num_detections:
        pad = np.zeros((num_detections - arr.size,), dtype=np.float32)
        arr = np.concatenate([arr, pad])

    return arr[:num_detections]


def _build_expanded_box_from_pole(
    pole_x1,
    pole_y1,
    pole_x2,
    pole_y2,
    image_w,
    image_h,
    width_factor_from_pole_height,
    min_box_width,
    top_extra_factor_from_pole_height,
    bottom_extra_factor_from_pole_height,
    min_top_extra_pixels,
    min_bottom_extra_pixels,
):
    """
    Build a larger ROI box around the detected pole, with separate control
    for how much the box extends ABOVE and BELOW the pole.

    Args:
        pole_x1, pole_y1, pole_x2, pole_y2:
            raw pole bbox coords
        image_w, image_h:
            full image size
        width_factor_from_pole_height:
            ROI width multiplier using pole height
        min_box_width:
            lower bound for ROI width
        top_extra_factor_from_pole_height:
            extra height above pole top
        bottom_extra_factor_from_pole_height:
            extra height below pole bottom
        min_top_extra_pixels, min_bottom_extra_pixels:
            lower bounds for top/bottom extra height

    Returns:
        dict with expanded box geometry
    """
    pole_w = max(0.0, pole_x2 - pole_x1)
    pole_h = max(0.0, pole_y2 - pole_y1)
    pole_cx = pole_x1 + pole_w / 2.0
    pole_cy = pole_y1 + pole_h / 2.0

    expanded_w = max(
        float(min_box_width),
        float(round(pole_h * width_factor_from_pole_height)),
    )

    top_extra = max(
        float(min_top_extra_pixels),
        float(round(pole_h * top_extra_factor_from_pole_height)),
    )

    bottom_extra = max(
        float(min_bottom_extra_pixels),
        float(round(pole_h * bottom_extra_factor_from_pole_height)),
    )

    half_w = expanded_w / 2.0

    ex1 = pole_cx - half_w
    ex2 = pole_cx + half_w

    # IMPORTANT:
    # top and bottom are controlled separately
    ey1 = pole_y1 - top_extra
    ey2 = pole_y2 + bottom_extra

    # Clamp to image bounds
    if ex1 < 0:
        ex2 -= ex1
        ex1 = 0.0
    if ex2 > image_w:
        shift = ex2 - image_w
        ex1 -= shift
        ex2 = float(image_w)

    if ey1 < 0:
        ey1 = 0.0
    if ey2 > image_h:
        ey2 = float(image_h)

    ex1 = max(0.0, ex1)
    ex2 = min(float(image_w), ex2)
    ey1 = max(0.0, ey1)
    ey2 = min(float(image_h), ey2)

    final_w = max(0.0, ex2 - ex1)
    final_h = max(0.0, ey2 - ey1)

    return {
        "expanded_x1": float(ex1),
        "expanded_y1": float(ey1),
        "expanded_x2": float(ex2),
        "expanded_y2": float(ey2),
        "expanded_w": float(final_w),
        "expanded_h": float(final_h),
        "pole_cx": float(pole_cx),
        "pole_cy": float(pole_cy),
    }

# -----------------------------------------------------------------------------
# 4. Choose images and create grid
# -----------------------------------------------------------------------------
grid_df = debug_images_df.copy().reset_index(drop=True)

n_images = len(grid_df)
n_rows = math.ceil(n_images / N_COLS)

print("Batch pole-detection run over debug_images_df:\n")
print(f"  n_images                               : {n_images}")
print(f"  prompt                                 : {POWER_POLE_PROMPT}")
print(f"  TEXT_THRESHOLD                         : {TEXT_THRESHOLD}")
print(f"  MASK_THRESHOLD                         : {MASK_THRESHOLD}")
print(f"  RUN_DEVICE                             : {RUN_DEVICE}")
print(f"  expanded width factor from pole height : {EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT}")
print(f"  MIN_EXPANDED_BOX_WIDTH                 : {MIN_EXPANDED_BOX_WIDTH}")
print(f"  TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT      : {TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT}")
print(f"  BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT   : {BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT}")
print(f"  MIN_TOP_EXTRA_PIXELS                   : {MIN_TOP_EXTRA_PIXELS}")
print(f"  MIN_BOTTOM_EXTRA_PIXELS                : {MIN_BOTTOM_EXTRA_PIXELS}")
print(f"  grid_cols                              : {N_COLS}")
print(f"  grid_rows                              : {n_rows}")
print(f"  panel_size                             : {PANEL_W} x {PANEL_H}")

fig, axes = plt.subplots(
    n_rows,
    N_COLS,
    figsize=(N_COLS * PANEL_W, n_rows * PANEL_H),
    dpi=GRID_DPI,
    constrained_layout=True,
)

axes = np.array(axes).reshape(-1)

# -----------------------------------------------------------------------------
# 5. Loop through all debug images
# -----------------------------------------------------------------------------
batch_rows = []

for plot_idx, (_, row) in enumerate(grid_df.iterrows()):
    ax = axes[plot_idx]

    image_path = row["image_path"]

    image_id = row.get("image_id", None)
    if pd.isna(image_id):
        image_id = None

    file_name = row.get("file_name", None)
    if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
        file_name = os.path.basename(image_path)

    try:
        # ---------------------------------------------------------------------
        # 5a. Validate and load image
        # ---------------------------------------------------------------------
        if not isinstance(image_path, str) or len(image_path.strip()) == 0:
            raise ValueError(f"Invalid image_path: {image_path}")

        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image file not found: {image_path}")

        with Image.open(image_path) as img:
            original_mode = img.mode
            if original_mode != "RGB":
                image = img.convert("RGB")
            else:
                image = img.copy()
            image.load()

        image_w, image_h = image.size
        image_cx = image_w / 2.0
        image_cy = image_h / 2.0
        image_diag = math.sqrt(image_w**2 + image_h**2)

        # ---------------------------------------------------------------------
        # 5b. Run SAM3 on one pole prompt only
        # ---------------------------------------------------------------------
        inputs = processor(
            images=image,
            text=POWER_POLE_PROMPT,
            return_tensors="pt",
        ).to(RUN_DEVICE)

        target_sizes = inputs["original_sizes"].detach().cpu().tolist()

        with torch.no_grad():
            outputs = model(**inputs)

        results = processor.post_process_instance_segmentation(
            outputs,
            threshold=TEXT_THRESHOLD,
            mask_threshold=MASK_THRESHOLD,
            target_sizes=target_sizes,
        )

        result = results[0] if isinstance(results, list) and len(results) > 0 else {}

        raw_boxes = result.get("boxes", None)
        raw_scores = result.get("scores", None)

        num_detections = _infer_num_detections(raw_boxes, raw_scores)
        boxes = _normalize_boxes_local(raw_boxes, num_detections)
        scores = _normalize_scores_local(raw_scores, num_detections)

        if num_detections == 0:
            raise ValueError("No pole detections returned.")

        # ---------------------------------------------------------------------
        # 5c. Build candidate table and choose best pole
        # ---------------------------------------------------------------------
        candidate_rows = []

        for det_idx in range(num_detections):
            x1, y1, x2, y2 = [float(v) for v in boxes[det_idx]]

            bbox_w = max(0.0, x2 - x1)
            bbox_h = max(0.0, y2 - y1)
            cx = x1 + bbox_w / 2.0
            cy = y1 + bbox_h / 2.0
            area = bbox_w * bbox_h
            vertical_ratio = bbox_h / max(bbox_w, 1e-6)
            center_dist = math.sqrt((cx - image_cx) ** 2 + (cy - image_cy) ** 2)
            center_dist_norm = center_dist / max(image_diag, 1e-6)
            is_vertical = bool(bbox_h > bbox_w)

            candidate_rows.append({
                "det_idx": int(det_idx),
                "score": float(scores[det_idx]),
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "bbox_w": bbox_w,
                "bbox_h": bbox_h,
                "cx": cx,
                "cy": cy,
                "area": area,
                "vertical_ratio": vertical_ratio,
                "center_dist_norm": center_dist_norm,
                "is_vertical": is_vertical,
            })

        candidates_df = pd.DataFrame(candidate_rows)

        candidates_df = candidates_df.sort_values(
            by=["is_vertical", "score", "bbox_h", "center_dist_norm"],
            ascending=[False, False, False, True],
        ).reset_index(drop=True)

        best_row = candidates_df.iloc[0]

        pole_x1 = float(best_row["x1"])
        pole_y1 = float(best_row["y1"])
        pole_x2 = float(best_row["x2"])
        pole_y2 = float(best_row["y2"])
        pole_w = float(best_row["bbox_w"])
        pole_h = float(best_row["bbox_h"])
        pole_score = float(best_row["score"])
        pole_vertical_ratio = float(best_row["vertical_ratio"])

        expanded_box = _build_expanded_box_from_pole(
            pole_x1=pole_x1,
            pole_y1=pole_y1,
            pole_x2=pole_x2,
            pole_y2=pole_y2,
            image_w=image_w,
            image_h=image_h,
            width_factor_from_pole_height=EXPANDED_BOX_WIDTH_FACTOR_FROM_POLE_HEIGHT,
            min_box_width=MIN_EXPANDED_BOX_WIDTH,
            top_extra_factor_from_pole_height=TOP_EXTRA_FACTOR_FROM_POLE_HEIGHT,
            bottom_extra_factor_from_pole_height=BOTTOM_EXTRA_FACTOR_FROM_POLE_HEIGHT,
            min_top_extra_pixels=MIN_TOP_EXTRA_PIXELS,
            min_bottom_extra_pixels=MIN_BOTTOM_EXTRA_PIXELS,
        )

        # ---------------------------------------------------------------------
        # 5d. Draw image + raw pole box + expanded ROI box
        # ---------------------------------------------------------------------
        ax.imshow(image)

        raw_rect = patches.Rectangle(
            (pole_x1, pole_y1),
            pole_w,
            pole_h,
            linewidth=2.4,
            edgecolor="red",
            facecolor="none",
        )
        ax.add_patch(raw_rect)

        expanded_rect = patches.Rectangle(
            (expanded_box["expanded_x1"], expanded_box["expanded_y1"]),
            expanded_box["expanded_w"],
            expanded_box["expanded_h"],
            linewidth=2.0,
            edgecolor="cyan",
            linestyle="--",
            facecolor="none",
            alpha=0.95,
        )
        ax.add_patch(expanded_rect)

        ax.text(
            pole_x1,
            max(5, pole_y1 - 6),
            f"POLE | {POWER_POLE_PROMPT} | score={pole_score:.3f}",
            color="white",
            fontsize=8,
            bbox=dict(facecolor="red", alpha=0.95, pad=1.3, edgecolor="none"),
        )

        ax.set_title(
            f"{file_name}\nDetections={num_detections}",
            fontsize=9,
            pad=8,
        )
        ax.axis("off")

        # ---------------------------------------------------------------------
        # 5e. Save summary row
        # ---------------------------------------------------------------------
        batch_rows.append({
            "debug_row_index": int(plot_idx),
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": original_mode,
            "image_w": int(image_w),
            "image_h": int(image_h),
            "prompt_text": POWER_POLE_PROMPT,
            "num_detections": int(num_detections),
            "selected_det_idx": int(best_row["det_idx"]),
            "selected_score": pole_score,
            "selected_is_vertical": bool(best_row["is_vertical"]),
            "selected_vertical_ratio": pole_vertical_ratio,
            "selected_center_dist_norm": float(best_row["center_dist_norm"]),
            "pole_x1": pole_x1,
            "pole_y1": pole_y1,
            "pole_x2": pole_x2,
            "pole_y2": pole_y2,
            "pole_w": pole_w,
            "pole_h": pole_h,
            "expanded_x1": expanded_box["expanded_x1"],
            "expanded_y1": expanded_box["expanded_y1"],
            "expanded_x2": expanded_box["expanded_x2"],
            "expanded_y2": expanded_box["expanded_y2"],
            "expanded_w": expanded_box["expanded_w"],
            "expanded_h": expanded_box["expanded_h"],
            "pole_cx": expanded_box["pole_cx"],
            "pole_cy": expanded_box["pole_cy"],
            "run_status": "ok",
            "error_message": "",
        })

        del image

    except Exception as e:
        ax.set_facecolor("white")
        ax.text(
            0.5,
            0.5,
            f"FAILED\n{file_name}\n\n{type(e).__name__}\n{str(e)}",
            ha="center",
            va="center",
            fontsize=9,
            wrap=True,
            transform=ax.transAxes,
        )
        ax.set_title(f"{file_name}\nFAILED", fontsize=9, pad=8)
        ax.axis("off")

        batch_rows.append({
            "debug_row_index": int(plot_idx),
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": None,
            "image_w": np.nan,
            "image_h": np.nan,
            "prompt_text": POWER_POLE_PROMPT,
            "num_detections": np.nan,
            "selected_det_idx": np.nan,
            "selected_score": np.nan,
            "selected_is_vertical": np.nan,
            "selected_vertical_ratio": np.nan,
            "selected_center_dist_norm": np.nan,
            "pole_x1": np.nan,
            "pole_y1": np.nan,
            "pole_x2": np.nan,
            "pole_y2": np.nan,
            "pole_w": np.nan,
            "pole_h": np.nan,
            "expanded_x1": np.nan,
            "expanded_y1": np.nan,
            "expanded_x2": np.nan,
            "expanded_y2": np.nan,
            "expanded_w": np.nan,
            "expanded_h": np.nan,
            "pole_cx": np.nan,
            "pole_cy": np.nan,
            "run_status": "failed",
            "error_message": str(e),
        })

# -----------------------------------------------------------------------------
# 6. Hide unused axes
# -----------------------------------------------------------------------------
for ax in axes[n_images:]:
    ax.axis("off")

# -----------------------------------------------------------------------------
# 7. Global title
# -----------------------------------------------------------------------------
fig.suptitle(
    f"Cell 7a — Batch Pole Detection | Prompt: {POWER_POLE_PROMPT}",
    fontsize=14,
)

plt.show()
plt.close()

# -----------------------------------------------------------------------------
# 8. Build output tables
# -----------------------------------------------------------------------------
pole_batch_results_df = pd.DataFrame(batch_rows)

pole_success_df = pole_batch_results_df[
    pole_batch_results_df["run_status"] == "ok"
].copy().reset_index(drop=True)

print("\nPole batch summary table:")
try:
    display(pole_batch_results_df)
except NameError:
    print(pole_batch_results_df)

if len(pole_success_df) > 0:
    print("\nSuccessful pole detections only:")
    cols_to_show = [
        "file_name",
        "num_detections",
        "selected_score",
        "pole_w",
        "pole_h",
        "selected_vertical_ratio",
        "expanded_w",
        "expanded_h",
    ]
    try:
        display(pole_success_df[cols_to_show])
    except NameError:
        print(pole_success_df[cols_to_show])

print("\nCell 7a completed successfully.")
print("Saved globals:")
print("  - pole_batch_results_df")
print("  - pole_success_df")

In [ ]:
# =============================================================================
# CELL 6c — ONE PROMPT ON ONE DEBUG IMAGE
#            + CONTAINMENT SUPPRESSION
#            + MAIN-CLUSTER FALSE-POSITIVE FILTER
#            + MULTI-COLOUR MASK DISPLAY
#            + Xarm_1 / Xarm_2 / ... LABELS
# =============================================================================
# EXPLANATION:
# This cell runs SAM3 on exactly ONE debug image using exactly ONE prompt:
#
#   "utility pole crossarm"
#
# It then applies two post-processing steps:
#
#   1) containment suppression
#      - if a shorter / smaller detection is mostly inside a larger crossarm
#        detection, remove the smaller one
#
#   2) main-cluster filtering
#      - if a detection is far away from the main crossarm group / cluster,
#        remove it as an isolated false positive
#
# Finally, it visualizes ONLY the final kept detections using:
# - different colours per mask
# - yellow SAM3 boxes
# - Xarm_1 / Xarm_2 / ... labels
# - a fixed blue label background
#
# IMPORTANT:
# - the SAM3 mask itself is NOT changed
# - this cell does NOT merge separated fragments across visible gaps
# - this is still a debugging / inspection cell
#
# PIPELINE POSITION:
# - Cell 6a prepared run_images_df and debug_images_df
# - Cell 6b confirmed one debug image loads correctly
# - This cell is the first full single-prompt, single-image SAM3 inference check
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
# os        : file existence checks / filename handling
# numpy     : array math and mask/box handling
# pandas    : tables for detections and filtering summaries
# matplotlib: visualization of image, masks, boxes, and labels
# torch     : model inference
# PIL       : image loading and RGB conversion
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image

# =============================================================================
# 0. SAFETY CHECKS
# =============================================================================
# EXPLANATION:
# This cell assumes:
# - debug_images_df already exists from Cell 6a
# - model and processor already exist from Cell 5b
# We fail early here if any of these are missing.
# ============================================================================
# Confirms the required objects from earlier cells already exist.
if "debug_images_df" not in globals():
    raise NameError(
        "debug_images_df not found.\n"
        "Please run Cell 6a first."
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError("debug_images_df exists but is not a pandas DataFrame.")

if debug_images_df.empty:
    raise ValueError("debug_images_df is empty.")

if "model" not in globals():
    raise NameError("model not found. Please run Cell 5b first.")

if "processor" not in globals():
    raise NameError("processor not found. Please run Cell 5b first.")

# =============================================================================
# 1. CHOOSE EXACTLY ONE DEBUG IMAGE
# =============================================================================
# EXPLANATION:
# We keep this cell intentionally simple:
# - one image
# - one prompt
# - inspect everything carefully
#
# Change DEBUG_ROW_INDEX when you want to inspect a different image from the
# debug subset prepared in Cell 6a.
# =============================================================================
# This cell is intentionally restricted to one image only for troubleshooting.
DEBUG_ROW_INDEX = 0

# Fails early if the selected row index is out of range.
if DEBUG_ROW_INDEX >= len(debug_images_df):
    raise IndexError(
        f"DEBUG_ROW_INDEX={DEBUG_ROW_INDEX} is out of range for "
        f"debug_images_df with {len(debug_images_df)} rows."
    )

# Pull the selected row from the debug subset.
row = debug_images_df.iloc[DEBUG_ROW_INDEX]

# =============================================================================
# 2. EXTRACT METADATA FROM THE SELECTED ROW
# =============================================================================
# EXPLANATION:
# We pull the essential identifiers from the selected debug row.
# - image_path is required
# - image_id and file_name are helpful for display and debugging
# - safe fallbacks are used if helper columns are missing or NaN
# =============================================================================
# Reads the image path plus helper identifiers used later in prints/plots.
image_path = row["image_path"]

# Safely fetch image_id if available.
image_id = row.get("image_id", None)
if pd.isna(image_id):
    image_id = None

# Safely fetch file_name if available; otherwise derive it from the path.
file_name = row.get("file_name", None)
if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
    file_name = os.path.basename(image_path)

# =============================================================================
# 3. CONFIRM THE IMAGE FILE EXISTS
# =============================================================================
# EXPLANATION:
# Before trying to open the file, confirm:
# - the path is a non-empty string
# - the file actually exists on disk
#
# This catches extraction/path problems early and clearly.
# =============================================================================
# Validates that the selected path is a usable non-empty string.
if not isinstance(image_path, str) or len(image_path.strip()) == 0:
    raise ValueError(f"Invalid image_path: {image_path}")

# Validates that the image really exists on disk.
if not os.path.exists(image_path):
    raise FileNotFoundError(
        "Selected image file does not exist on disk.\n"
        f"image_path: {image_path}"
    )

# =============================================================================
# 4. LOAD THE IMAGE AS RGB
# =============================================================================
# EXPLANATION:
# SAM3 expects a normal RGB-style image input.
#
# This block:
# - opens the image with PIL
# - checks its original mode
# - converts to RGB if needed
# - forces a full image load before leaving the with-block
#
# WHY FORCE image.load()?
# - so the image data is safely in memory after the file handle closes
# =============================================================================
# Opens the image and converts to RGB if required.
# image.load() forces the pixel data into memory before the file handle closes.
with Image.open(image_path) as img:
    original_mode = img.mode

    # Converts non-RGB images so later SAM3 input is predictable.
    if original_mode != "RGB":
        print(
            f"WARNING: Image mode is '{original_mode}', not 'RGB'. "
            "Converting to RGB."
        )
        image = img.convert("RGB")
    else:
        image = img.copy()

    # Fully decodes image pixels before exiting the with-block.
    image.load()

# Stores image dimensions for debug prints and later reasoning if needed.
image_w, image_h = image.size

# =============================================================================
# 5. PROMPT, THRESHOLDS, AND POST-PROCESSING SETTINGS
# =============================================================================
# EXPLANATION:
# This cell uses exactly one prompt for debugging:
#   "utility pole crossarm"
#
# Thresholds are pulled from notebook globals if they already exist from Cell 2.
# A CPU/GPU device is also selected from the notebook globals if available.
# =============================================================================
# Uses exactly one prompt for this diagnostic cell.
PROMPT_TEXT = "utility pole crossarm"

# Pulls thresholds from notebook globals if they were defined earlier.
TEXT_THRESHOLD = float(globals().get("TEXT_SCORE_THRESHOLD", 0.30))
MASK_THRESHOLD = float(globals().get("MASK_THRESHOLD", 0.50))

# Uses the existing notebook device if present; otherwise chooses a sensible fallback.
RUN_DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------------------------------------------------
# Containment suppression settings
# -----------------------------------------------------------------------------
# A smaller detection will be removed if:
# - at least CONTAINMENT_THRESHOLD fraction of its box is covered by a larger box
# - the larger box is at least MIN_AREA_RATIO times bigger
# - the larger box has at least the same (or better) score
CONTAINMENT_THRESHOLD = 0.80
MIN_AREA_RATIO = 1.20
MIN_SCORE_ADVANTAGE = 0.0

# -----------------------------------------------------------------------------
# Main-cluster filtering settings
# -----------------------------------------------------------------------------
# Detections that are too far from the main crossarm group are removed.
#
# CENTER_DIST_FACTOR:
# - pairwise center distance threshold = CENTER_DIST_FACTOR * median box diagonal
# - detections connected by this rule belong to the same cluster
# - we keep the strongest / largest connected component
CENTER_DIST_FACTOR = 2.75

print("Single-prompt SAM3 debug run:\n")
print(f"  DEBUG_ROW_INDEX       : {DEBUG_ROW_INDEX}")
print(f"  image_id              : {image_id}")
print(f"  file_name             : {file_name}")
print(f"  image_path            : {image_path}")
print(f"  width                 : {image_w}")
print(f"  height                : {image_h}")
print(f"  original_mode         : {original_mode}")
print(f"  final_mode            : {image.mode}")
print(f"  prompt                : {PROMPT_TEXT}")
print(f"  TEXT_THRESHOLD        : {TEXT_THRESHOLD}")
print(f"  MASK_THRESHOLD        : {MASK_THRESHOLD}")
print(f"  RUN_DEVICE            : {RUN_DEVICE}")
print(f"  CONTAINMENT_THRESHOLD : {CONTAINMENT_THRESHOLD}")
print(f"  MIN_AREA_RATIO        : {MIN_AREA_RATIO}")
print(f"  MIN_SCORE_ADVANTAGE   : {MIN_SCORE_ADVANTAGE}")
print(f"  CENTER_DIST_FACTOR    : {CENTER_DIST_FACTOR}")

# =============================================================================
# 6. HELPER FUNCTIONS — NORMALIZE SAM3 OUTPUTS
# =============================================================================
# EXPLANATION:
# SAM3 post-processing may return tensors / arrays / different shapes depending
# on the result. These helpers normalize masks, boxes, and scores into a stable
# NumPy format that the rest of the cell can use safely.
# =============================================================================
def to_numpy(x):
    """
    Convert tensor-like objects to NumPy arrays.

    Args:
        x:
            Tensor, list, scalar, or array-like value.

    Returns:
        np.ndarray
    """
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def normalize_masks(masks):
    """
    Normalize masks to shape (N, H, W), bool.

    Args:
        masks:
            Raw masks returned by SAM3 post-processing.

    Returns:
        np.ndarray
    """
    # If SAM3 returned nothing, create an empty standard shape.
    if masks is None:
        return np.zeros((0, 0, 0), dtype=bool)

    arr = to_numpy(masks)

    # Handles an empty result safely.
    if arr.size == 0:
        return np.zeros((0, 0, 0), dtype=bool)

    # If a single mask came back as H x W, wrap it to 1 x H x W.
    if arr.ndim == 2:
        arr = arr[None, ...]

    return arr.astype(bool)


def normalize_boxes(boxes, num_masks):
    """
    Normalize boxes to shape (N, 4).

    Args:
        boxes:
            Raw boxes returned by SAM3 post-processing.
        num_masks:
            Number of masks.

    Returns:
        np.ndarray
    """
    # If boxes are missing, return a placeholder array aligned to mask count.
    if boxes is None:
        return np.zeros((num_masks, 4), dtype=np.float32)

    arr = to_numpy(boxes).astype(np.float32)

    # If one box came back as a flat vector, reshape to 1 x 4.
    if arr.ndim == 1 and arr.shape[0] == 4:
        arr = arr.reshape(1, 4)

    # If the shape is not usable, return a zero placeholder.
    if arr.ndim != 2 or arr.shape[1] != 4:
        return np.zeros((num_masks, 4), dtype=np.float32)

    # Creates a fixed-size output aligned to the mask count.
    out = np.zeros((num_masks, 4), dtype=np.float32)
    n = min(num_masks, len(arr))
    out[:n] = arr[:n]
    return out


def normalize_scores(scores, num_masks):
    """
    Normalize scores to shape (N,).

    Args:
        scores:
            Raw scores returned by SAM3 post-processing.
        num_masks:
            Number of masks.

    Returns:
        np.ndarray
    """
    # If scores are missing, return zero scores.
    if scores is None:
        return np.zeros((num_masks,), dtype=np.float32)

    arr = to_numpy(scores)

    # Handles the scalar-score case.
    if arr.ndim == 0:
        if num_masks == 1:
            return np.array([float(arr)], dtype=np.float32)
        return np.zeros((num_masks,), dtype=np.float32)

    # Flattens to a simple vector.
    arr = arr.astype(np.float32).reshape(-1)

    # Creates a fixed-size output aligned to the mask count.
    out = np.zeros((num_masks,), dtype=np.float32)
    n = min(num_masks, len(arr))
    out[:n] = arr[:n]
    return out

# =============================================================================
# 6b. HELPER FUNCTIONS — CONTAINMENT-BASED SUPPRESSION
# =============================================================================
# EXPLANATION:
# These helpers support the first cleanup stage:
#
# "If a shorter/smaller detection is mostly inside a bigger detection, and the
# bigger one is at least as good, suppress the smaller one."
#
# This is useful when SAM3 returns a short fragment that is really just part
# of a longer crossarm already detected nearby.
# =============================================================================
# Removes a shorter detection if it is mostly inside a larger one.
def box_area_xyxy(box_xyxy):
    """
    Compute area of an xyxy box.

    Args:
        box_xyxy:
            [x1, y1, x2, y2]

    Returns:
        float
    """
    x1, y1, x2, y2 = [float(v) for v in box_xyxy]
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def intersection_area_xyxy(box_a, box_b):
    """
    Compute intersection area between two xyxy boxes.

    Args:
        box_a:
            [x1, y1, x2, y2]
        box_b:
            [x1, y1, x2, y2]

    Returns:
        float
    """
    ax1, ay1, ax2, ay2 = [float(v) for v in box_a]
    bx1, by1, bx2, by2 = [float(v) for v in box_b]

    # Computes the overlap rectangle.
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    return max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)


def suppress_contained_shorter_detections(
    detections_df,
    containment_threshold=0.80,
    min_area_ratio=1.20,
    min_score_advantage=0.0,
):
    """
    Suppress smaller detections that are mostly contained inside a larger one.

    LOGIC:
    Detection j is suppressed if there exists detection i such that:
    - j is mostly inside i
    - i has at least the same (or better) score
    - i is meaningfully larger than j

    Args:
        detections_df:
            DataFrame with columns:
            - orig_det_idx
            - score
            - x1, y1, x2, y2

        containment_threshold:
            Fraction of the smaller box that must be covered by the larger box.

        min_area_ratio:
            Larger box area must be at least this ratio times the smaller box.

        min_score_advantage:
            Larger box score must exceed smaller box by at least this amount.

    Returns:
        tuple:
            kept_df, removed_df
    """
    if detections_df.empty:
        return detections_df.copy(), detections_df.iloc[0:0].copy()

    # Work on a clean copy so the original detection table stays untouched.
    df = detections_df.copy().reset_index(drop=True)

    # -------------------------------------------------------------------------
    # Precompute box areas once
    # -------------------------------------------------------------------------
    df["box_area"] = df.apply(
        lambda r: box_area_xyxy([r["x1"], r["y1"], r["x2"], r["y2"]]),
        axis=1,
    )

    # Starts with everything kept.
    keep_mask = np.ones(len(df), dtype=bool)
    removal_reason = [None] * len(df)

    # -------------------------------------------------------------------------
    # Compare each detection j against every other detection i
    # -------------------------------------------------------------------------
    for j in range(len(df)):
        if not keep_mask[j]:
            continue

        area_j = float(df.loc[j, "box_area"])
        score_j = float(df.loc[j, "score"])
        box_j = [df.loc[j, "x1"], df.loc[j, "y1"], df.loc[j, "x2"], df.loc[j, "y2"]]

        # Invalid-area detections are removed immediately.
        if area_j <= 0:
            keep_mask[j] = False
            removal_reason[j] = "invalid_box_area"
            continue

        for i in range(len(df)):
            if i == j:
                continue

            area_i = float(df.loc[i, "box_area"])
            score_i = float(df.loc[i, "score"])
            box_i = [df.loc[i, "x1"], df.loc[i, "y1"], df.loc[i, "x2"], df.loc[i, "y2"]]

            # Candidate i must be meaningfully larger than j.
            if area_i < area_j * min_area_ratio:
                continue

            # Fraction of the smaller box covered by the larger one.
            inter_ij = intersection_area_xyxy(box_i, box_j)
            frac_small_inside_big = inter_ij / area_j if area_j > 0 else 0.0

            # Larger box must have at least equal-or-better score.
            score_ok = (score_i >= score_j + min_score_advantage)

            # If both conditions pass, j is considered redundant and removed.
            if frac_small_inside_big >= containment_threshold and score_ok:
                keep_mask[j] = False
                removal_reason[j] = f"contained_in_det_{int(df.loc[i, 'orig_det_idx'])}"
                break

    # Build kept and removed outputs.
    kept_df = df[keep_mask].copy().reset_index(drop=True)
    removed_df = df[~keep_mask].copy().reset_index(drop=True)

    if len(removed_df) > 0:
        removed_df["removal_reason"] = [
            removal_reason[idx] for idx in np.where(~keep_mask)[0]
        ]

    return kept_df, removed_df

# =============================================================================
# 6c. HELPER FUNCTIONS — MAIN-CLUSTER FILTERING
# =============================================================================
# EXPLANATION:
# These helpers support the second cleanup stage:
#
# "If a detection is far from the main crossarm cluster, remove it as an
# isolated false positive."
#
# HOW IT WORKS:
# - compute box centers
# - connect detections whose centers are close enough
# - find connected components
# - keep the strongest component
#
# STRONGEST COMPONENT RULE:
# - prefer larger component size
# - if tied, prefer higher total score
# =============================================================================
# Keeps only detections near the main crossarm group.
def compute_centers_and_scale(df):
    """
    Compute box centers and a characteristic size scale.

    Args:
        df:
            DataFrame with x1, y1, x2, y2.

    Returns:
        tuple:
            df_with_centers, median_diag
    """
    out = df.copy()

    # Computes center, width, height, and diagonal length per detection.
    out["cx"] = (out["x1"] + out["x2"]) / 2.0
    out["cy"] = (out["y1"] + out["y2"]) / 2.0
    out["w"] = (out["x2"] - out["x1"]).clip(lower=0.0)
    out["h"] = (out["y2"] - out["y1"]).clip(lower=0.0)
    out["diag"] = np.sqrt(out["w"] ** 2 + out["h"] ** 2)

    # Uses the median diagonal as a scale estimate.
    median_diag = float(np.median(out["diag"])) if len(out) > 0 else 0.0
    median_diag = max(median_diag, 1.0)

    return out, median_diag


def connected_components_from_center_distance(df, center_dist_factor=2.75):
    """
    Build connected components based on box-center distance.

    Two detections are connected if their center distance is less than:
        center_dist_factor * median_box_diagonal

    Args:
        df:
            DataFrame that already contains cx, cy, diag.
        center_dist_factor:
            Distance multiplier.

    Returns:
        tuple:
            components, center_dist_threshold
    """
    if len(df) == 0:
        return [], 0.0

    # Builds a distance threshold tied to the typical box size.
    median_diag = float(np.median(df["diag"])) if len(df) > 0 else 1.0
    median_diag = max(median_diag, 1.0)
    center_dist_threshold = center_dist_factor * median_diag

    n = len(df)
    visited = np.zeros(n, dtype=bool)
    components = []

    # Extracts center coordinates for distance calculations.
    centers = df[["cx", "cy"]].to_numpy(dtype=float)

    # adjacency[i] will store which other detections are "close" to i.
    adjacency = [[] for _ in range(n)]

    # -------------------------------------------------------------------------
    # Build pairwise adjacency using center distance
    # -------------------------------------------------------------------------
    for i in range(n):
        for j in range(i + 1, n):
            dist_ij = np.linalg.norm(centers[i] - centers[j])

            if dist_ij <= center_dist_threshold:
                adjacency[i].append(j)
                adjacency[j].append(i)

    # -------------------------------------------------------------------------
    # DFS connected components -Finds connected components using a DFS-style
    # traversal.
    # -------------------------------------------------------------------------
    for start in range(n):
        if visited[start]:
            continue

        stack = [start]
        comp = []
        visited[start] = True

        while stack:
            node = stack.pop()
            comp.append(node)

            for nbr in adjacency[node]:
                if not visited[nbr]:
                    visited[nbr] = True
                    stack.append(nbr)

        components.append(sorted(comp))

    return components, center_dist_threshold


def keep_main_detection_cluster(detections_df, center_dist_factor=2.75):
    """
    Keep only detections belonging to the strongest connected cluster.

    CLUSTER SELECTION RULE:
    - prefer larger component size
    - break ties using higher total score

    Args:
        detections_df:
            DataFrame with orig_det_idx, score, x1, y1, x2, y2
        center_dist_factor:
            Center-distance multiplier.

    Returns:
        tuple:
            kept_df, removed_df, cluster_threshold
    """
    if detections_df.empty:
        return detections_df.copy(), detections_df.iloc[0:0].copy(), 0.0

    # With only one detection, nothing needs clustering.
    if len(detections_df) == 1:
        df1, median_diag = compute_centers_and_scale(detections_df)
        return df1.reset_index(drop=True), df1.iloc[0:0].copy(), center_dist_factor * median_diag

    # Computes center geometry and connected components.
    df, median_diag = compute_centers_and_scale(detections_df)
    components, cluster_threshold = connected_components_from_center_distance(
        df,
        center_dist_factor=center_dist_factor,
    )

    best_component = None
    best_key = None

    # Pick the strongest component:
    # - larger component size first
    # - higher total score second
    for comp in components:
        comp_df = df.iloc[comp]
        comp_size = len(comp)
        comp_score_sum = float(comp_df["score"].sum())

        key = (comp_size, comp_score_sum)

        if best_key is None or key > best_key:
            best_key = key
            best_component = comp

    # Keep only the selected strongest component.
    keep_idx = set(best_component)
    kept_df = df.iloc[sorted(keep_idx)].copy().reset_index(drop=True)
    removed_df = df.drop(index=sorted(keep_idx)).copy().reset_index(drop=True)

    if len(removed_df) > 0:
        removed_df["removal_reason"] = "outside_main_cluster"

    return kept_df, removed_df, cluster_threshold

# =============================================================================
# 7. RUN SAM3 FOR THIS ONE IMAGE + ONE PROMPT
# =============================================================================
# EXPLANATION:
# The processor converts the image + prompt into model inputs.
# Then the SAM3 model is run in no-grad inference mode.
# =============================================================================
# Builds model inputs from the image + text prompt.
inputs = processor(
    images=image,
    text=PROMPT_TEXT,
    return_tensors="pt",
).to(RUN_DEVICE)

# -----------------------------------------------------------------------------
# IMPORTANT NOTE ABOUT TARGET SIZES
# -----------------------------------------------------------------------------
# Uses processor-reported original sizes so post-processing maps back to image space.
target_sizes = inputs["original_sizes"].detach().cpu().tolist()

# Runs the model without gradient tracking since this is inference only.
with torch.no_grad():
    outputs = model(**inputs)

# =============================================================================
# 8. POST-PROCESS THE RAW SAM3 OUTPUTS INTO MASKS / BOXES / SCORES
# =============================================================================
# EXPLANATION:
# The processor converts raw model output into usable instance segmentation
# results for the original image coordinate space.
# =============================================================================
# Converts raw model outputs into instance segmentation results.
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=TEXT_THRESHOLD,
    mask_threshold=MASK_THRESHOLD,
    target_sizes=target_sizes,
)

# Gets the first image result because this cell only runs one image.
result = results[0] if isinstance(results, list) and len(results) > 0 else {}

# Normalizes masks / boxes / scores into stable NumPy arrays.
masks = normalize_masks(result.get("masks", None))
num_masks = int(masks.shape[0]) if masks.ndim == 3 else 0

boxes = normalize_boxes(result.get("boxes", None), num_masks)
scores = normalize_scores(result.get("scores", None), num_masks)

# =============================================================================
# 9. BUILD RAW DETECTIONS TABLE + APPLY CONTAINMENT SUPPRESSION
# =============================================================================
print("\nDetection summary:")
print(f"  raw_num_masks returned : {num_masks}")

if num_masks == 0:
    print("  No detections were returned for this prompt.")

    raw_detections_df = pd.DataFrame(
        columns=["orig_det_idx", "score", "x1", "y1", "x2", "y2"]
    )
    kept_after_containment_df = raw_detections_df.copy()
    removed_by_containment_df = raw_detections_df.copy()

# Constructs the raw per-detection table directly from SAM3 outputs.
else:
    raw_detections_df = pd.DataFrame({
        "orig_det_idx": np.arange(num_masks, dtype=int),
        "score": scores.astype(float),
        "x1": boxes[:, 0].astype(float),
        "y1": boxes[:, 1].astype(float),
        "x2": boxes[:, 2].astype(float),
        "y2": boxes[:, 3].astype(float),
    })

    # -------------------------------------------------------------------------
    # Sort raw detections by score descending for easier reading
    # -------------------------------------------------------------------------
    raw_detections_df = raw_detections_df.sort_values(
        "score",
        ascending=False
    ).reset_index(drop=True)

    print("\nRaw detections table:")
    try:
        display(raw_detections_df)
    except NameError:
        print(raw_detections_df)

    # -------------------------------------------------------------------------
    # First pass: suppress shorter detections contained in bigger ones
    # -------------------------------------------------------------------------
    kept_after_containment_df, removed_by_containment_df = suppress_contained_shorter_detections(
        detections_df=raw_detections_df,
        containment_threshold=CONTAINMENT_THRESHOLD,
        min_area_ratio=MIN_AREA_RATIO,
        min_score_advantage=MIN_SCORE_ADVANTAGE,
    )

    print(f"\nKept after containment suppression   : {len(kept_after_containment_df)}")
    print(f"Removed by containment suppression   : {len(removed_by_containment_df)}")

    print("\nKept after containment suppression:")
    try:
        display(kept_after_containment_df)
    except NameError:
        print(kept_after_containment_df)

    if len(removed_by_containment_df) > 0:
        print("\nRemoved by containment suppression:")
        try:
            display(removed_by_containment_df)
        except NameError:
            print(removed_by_containment_df)

# =============================================================================
# 9b. SECOND PASS — REMOVE DETECTIONS FAR FROM THE MAIN CLUSTER
# =============================================================================
# If nothing survived containment suppression, keep the outputs empty.
if len(kept_after_containment_df) == 0:
    final_kept_detections_df = kept_after_containment_df.copy()
    removed_by_cluster_df = kept_after_containment_df.copy()
    cluster_threshold_used = 0.0

# Otherwise, keep only the strongest spatial cluster.
else:
    final_kept_detections_df, removed_by_cluster_df, cluster_threshold_used = keep_main_detection_cluster(
        detections_df=kept_after_containment_df,
        center_dist_factor=CENTER_DIST_FACTOR,
    )

print(f"\nCluster distance threshold used : {cluster_threshold_used:.2f}")
print(f"Final kept detections           : {len(final_kept_detections_df)}")
print(f"Removed as isolated false-pos   : {len(removed_by_cluster_df)}")

print("\nFinal kept detections table:")
try:
    display(final_kept_detections_df)
except NameError:
    print(final_kept_detections_df)

if len(removed_by_cluster_df) > 0:
    print("\nRemoved by main-cluster filter:")
    try:
        display(removed_by_cluster_df)
    except NameError:
        print(removed_by_cluster_df)

# =============================================================================
# 9c. ASSIGN ORDERED CROSSARM LABELS TO FINAL KEPT DETECTIONS
# =============================================================================
# EXPLANATION:
# We label crossarms from top to bottom using the vertical center of the final
# kept predicted box.
#
# RESULT:
# - topmost crossarm  -> Xarm_1
# - next lower one    -> Xarm_2
# - next lower one    -> Xarm_3
# =============================================================================
# Computes center coordinates and labels detections from top to bottom.
if len(final_kept_detections_df) > 0:
    final_kept_detections_df = final_kept_detections_df.copy()

    # -------------------------------------------------------------------------
    # If cx/cy were not already present, create them
    # -------------------------------------------------------------------------
    # Creates center coordinates if they do not already exist.
    if "cx" not in final_kept_detections_df.columns:
        final_kept_detections_df["cx"] = (
            final_kept_detections_df["x1"] + final_kept_detections_df["x2"]
        ) / 2.0

    if "cy" not in final_kept_detections_df.columns:
        final_kept_detections_df["cy"] = (
            final_kept_detections_df["y1"] + final_kept_detections_df["y2"]
        ) / 2.0

    # -------------------------------------------------------------------------
    # Sort top-to-bottom, then left-to-right as tie-breaker
    # -------------------------------------------------------------------------
    final_kept_detections_df = final_kept_detections_df.sort_values(
        ["cy", "cx"],
        ascending=[True, True]
    ).reset_index(drop=True)

    final_kept_detections_df["xarm_label"] = [
        f"Xarm_{i+1}" for i in range(len(final_kept_detections_df))
    ]

    print("\nFinal kept detections with Xarm labels:")
    try:
        display(
            final_kept_detections_df[
                ["xarm_label", "orig_det_idx", "score", "x1", "y1", "x2", "y2", "cx", "cy"]
            ]
        )
    except NameError:
        print(
            final_kept_detections_df[
                ["xarm_label", "orig_det_idx", "score", "x1", "y1", "x2", "y2", "cx", "cy"]
            ]
        )

# =============================================================================
# 10. VISUALIZE ONLY THE FINAL KEPT DETECTIONS ON THE IMAGE
# =============================================================================
# EXPLANATION:
# This final visualization step shows:
# - the base image
# - each final kept mask in a different colour
# - the SAM3 predicted box in yellow
# - the crossarm label in a fixed blue label box
#
# IMPORTANT:
# - the mask itself is unchanged
# - this is just a visualization overlay
# =============================================================================
plt.figure(figsize=(12, 9))
ax = plt.gca()

# Show the base image first
ax.imshow(image)

num_final_kept = len(final_kept_detections_df)

if num_final_kept > 0:
    # -------------------------------------------------------------------------
    # Use a distinct colour for each kept detection
    # -------------------------------------------------------------------------
    try:
        cmap = plt.colormaps.get_cmap("tab10").resampled(max(num_final_kept, 1))
    except Exception:
        cmap = plt.cm.get_cmap("tab10", max(num_final_kept, 1))

    # -------------------------------------------------------------------------
    # Plot final kept detections in the SAME order as final_kept_detections_df
    # -------------------------------------------------------------------------
    for plot_idx, det_row in final_kept_detections_df.iterrows():
        orig_idx = int(det_row["orig_det_idx"])

        mask_i = masks[orig_idx]
        box_i = boxes[orig_idx]
        score_i = float(det_row["score"])
        xarm_label = det_row["xarm_label"]

        # -------------------------------------------------------------
        # Multi-colour mask fill
        # -------------------------------------------------------------
        color_rgba = cmap(plot_idx)

        overlay = np.zeros((mask_i.shape[0], mask_i.shape[1], 4), dtype=np.float32)
        overlay[..., 0] = color_rgba[0]
        overlay[..., 1] = color_rgba[1]
        overlay[..., 2] = color_rgba[2]
        overlay[..., 3] = mask_i.astype(np.float32) * 0.45
        ax.imshow(overlay)

        # -------------------------------------------------------------
        # Predicted box (yellow)
        # -------------------------------------------------------------
        x1, y1, x2, y2 = [float(v) for v in box_i]

        rect = patches.Rectangle(
            (x1, y1),
            max(0.0, x2 - x1),
            max(0.0, y2 - y1),
            linewidth=2.0,
            edgecolor="yellow",
            facecolor="none",
        )
        ax.add_patch(rect)

        # -------------------------------------------------------------
        # Detection label (fixed blue)
        # -------------------------------------------------------------
        label_bg = "#1E90FF"

        ax.text(
            x1,
            max(5, y1 - 5),
            f"{xarm_label}: {score_i:.3f}",
            color="white",
            fontsize=9,
            bbox=dict(facecolor=label_bg, alpha=0.95, pad=1.5, edgecolor="none"),
        )

ax.set_title(
    f"{file_name}\n"
    f"Prompt: {PROMPT_TEXT} | Raw detections: {num_masks} | Final kept: {num_final_kept}",
    fontsize=12,
    pad=12,
)
ax.axis("off")
plt.show()
plt.close()

# =============================================================================
# 11. FINAL NOTE
# =============================================================================
print("\nCell 6c completed successfully.")
print("This shows ONE prompt on ONE image with:")
print("- containment-based suppression")
print("- main-cluster false-positive filtering")
print("- different colours per final kept mask")
print("- yellow SAM3 boxes")
print("- Xarm_1 / Xarm_2 / ... labels")

# =============================================================================
# 12. OPTIONAL CLEANUP
# =============================================================================
del image

In [ ]:
# =============================================================================
# CELL 6d — RUN ONE PROMPT ON ALL DEBUG IMAGES
#            + CONTAINMENT SUPPRESSION
#            + MAIN-CLUSTER FALSE-POSITIVE FILTER
#            + MULTI-COLOUR MASK DISPLAY
#            + Xarm_1 / Xarm_2 / ... LABELS
#            + 3-COLUMN PRESENTATION GRID
#            + CROSSARM BBOX DIMENSION DIAGNOSTICS
# =============================================================================
# EXPLANATION:
# This cell applies the SAME single-prompt crossarm workflow from Cell 6c to
# ALL images in debug_images_df.
#
# WHAT IT DOES:
# - loops through every image in debug_images_df
# - runs SAM3 with the single prompt:
#       "utility pole crossarm"
# - applies the SAME two post-processing stages from Cell 6c:
#       1) containment suppression
#       2) main-cluster false-positive filtering
# - assigns Xarm_1 / Xarm_2 / ... labels per image
# - displays all debug images in one grid:
#       - 3 images per row
#       - fixed panel size
#       - presentation-friendly overall size
# - builds a NEW bbox diagnostics table for FINAL kept crossarms so we can inspect:
#       - bbox width / height
#       - long side / short side
#       - long/short aspect ratio
#       - whether the box is provisionally "square-ish"
#
# VISUAL STYLE:
# - final kept masks = different colours per crossarm
# - light transparency
# - no border / no outline
# - SAM3 predicted boxes = yellow
# - Xarm labels = fixed blue label box with white text
#
# IMPORTANT:
# - this cell does NOT save masks/crops
# - this cell is still a debug / QA visualization step
# - this cell REUSES helper functions already defined in Cell 6c
# - the square-ish flag is ONLY a diagnostic at this stage, not a final rule
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
# os        : file/path checks
# math      : grid row calculation
# numpy     : numeric helpers
# pandas    : summary tables
# matplotlib: grid plotting
# torch     : device fallback
# PIL       : image loading / RGB conversion
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
# Confirms the required objects from earlier cells already exist.
required_globals = [
    "debug_images_df",
    "model",
    "processor",
    "normalize_masks",
    "normalize_boxes",
    "normalize_scores",
    "suppress_contained_shorter_detections",
    "keep_main_detection_cluster",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 6d requires objects/functions from earlier cells.\n"
        "Please run Cell 5b and the current Cell 6c first.\n"
        f"Missing globals: {missing_globals}"
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError("debug_images_df exists but is not a pandas DataFrame.")

if debug_images_df.empty:
    raise ValueError("debug_images_df is empty.")

# -----------------------------------------------------------------------------
# 1. Prompt, thresholds, and post-processing settings
# -----------------------------------------------------------------------------
# Uses the SAME single prompt and cleanup settings as Cell 6c.
PROMPT_TEXT = "utility pole crossarm"

TEXT_THRESHOLD = float(globals().get("TEXT_SCORE_THRESHOLD", 0.30))
MASK_THRESHOLD = float(globals().get("MASK_THRESHOLD", 0.50))

RUN_DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# Containment suppression settings
CONTAINMENT_THRESHOLD = 0.80
MIN_AREA_RATIO = 1.20
MIN_SCORE_ADVANTAGE = 0.0

# Main-cluster filtering settings
CENTER_DIST_FACTOR = 2.75

# -----------------------------------------------------------------------------
# 2. Grid display settings
# -----------------------------------------------------------------------------
# Uses a fixed, presentation-friendly panel size and a 3-column layout.
N_COLS = 3
PANEL_W = 4.8
PANEL_H = 3.6
GRID_DPI = 120

# Multi-colour mask display settings
# Each kept mask gets a different colour from the colormap.
MASK_ALPHA = 0.45

# Fixed blue label style to match your preferred 6c view
LABEL_BG = "#1E90FF"

# -----------------------------------------------------------------------------
# 2b. BBox diagnostic settings
# -----------------------------------------------------------------------------
# EXPLANATION:
# We are NOT making a final decision rule yet.
# This is only to inspect whether "square-ish" boxes are a useful first-stage
# diagnostic for suspicious crossarm detections.
#
# aspect_ratio_long_over_short:
# - 1.0  -> perfect square
# - high -> elongated rectangle
#
# This threshold is only for QA inspection and can be changed later.
SQUAREISH_ASPECT_THRESHOLD = 1.10

# -----------------------------------------------------------------------------
# 3. Choose which images to show
# -----------------------------------------------------------------------------
# This cell uses the current debug subset directly.
grid_df = debug_images_df.copy().reset_index(drop=True)

n_images = len(grid_df)
n_rows = math.ceil(n_images / N_COLS)

print("Batch debug run over debug_images_df:\n")
print(f"  n_images                       : {n_images}")
print(f"  prompt                         : {PROMPT_TEXT}")
print(f"  TEXT_THRESHOLD                 : {TEXT_THRESHOLD}")
print(f"  MASK_THRESHOLD                 : {MASK_THRESHOLD}")
print(f"  RUN_DEVICE                     : {RUN_DEVICE}")
print(f"  CONTAINMENT_THRESHOLD          : {CONTAINMENT_THRESHOLD}")
print(f"  MIN_AREA_RATIO                 : {MIN_AREA_RATIO}")
print(f"  MIN_SCORE_ADVANTAGE            : {MIN_SCORE_ADVANTAGE}")
print(f"  CENTER_DIST_FACTOR             : {CENTER_DIST_FACTOR}")
print(f"  SQUAREISH_ASPECT_THRESHOLD     : {SQUAREISH_ASPECT_THRESHOLD}")
print(f"  grid_cols                      : {N_COLS}")
print(f"  grid_rows                      : {n_rows}")
print(f"  panel_size                     : {PANEL_W} x {PANEL_H}")

# -----------------------------------------------------------------------------
# 4. Create the figure grid
# -----------------------------------------------------------------------------
# Creates one subplot per image and later turns off any unused panels.
fig, axes = plt.subplots(
    n_rows,
    N_COLS,
    figsize=(N_COLS * PANEL_W, n_rows * PANEL_H),
    dpi=GRID_DPI,
    constrained_layout=True,
)

# Normalizes axes into a flat 1D array so iteration is simple.
axes = np.array(axes).reshape(-1)

# -----------------------------------------------------------------------------
# 5. Loop over every debug image
# -----------------------------------------------------------------------------
# Each image is processed independently so one failure does not stop the rest.
batch_rows = []
bbox_rows = []

for plot_idx, (_, row) in enumerate(grid_df.iterrows()):
    ax = axes[plot_idx]

    # -------------------------------------------------------------------------
    # 5a. Extract metadata for the current image
    # -------------------------------------------------------------------------
    image_path = row["image_path"]

    image_id = row.get("image_id", None)
    if pd.isna(image_id):
        image_id = None

    file_name = row.get("file_name", None)
    if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
        file_name = os.path.basename(image_path)

    try:
        # ---------------------------------------------------------------------
        # 5b. Validate image path
        # ---------------------------------------------------------------------
        # Fails cleanly if the path is missing or invalid.
        if not isinstance(image_path, str) or len(image_path.strip()) == 0:
            raise ValueError(f"Invalid image_path: {image_path}")

        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image file not found: {image_path}")

        # ---------------------------------------------------------------------
        # 5c. Load the image as RGB
        # ---------------------------------------------------------------------
        # Converts to RGB if required and fully loads pixels into memory.
        with Image.open(image_path) as img:
            original_mode = img.mode

            if original_mode != "RGB":
                image = img.convert("RGB")
            else:
                image = img.copy()

            image.load()

        image_w, image_h = image.size

        # ---------------------------------------------------------------------
        # 5d. Build SAM3 inputs and run inference
        # ---------------------------------------------------------------------
        # Runs the single crossarm prompt on the current image.
        inputs = processor(
            images=image,
            text=PROMPT_TEXT,
            return_tensors="pt",
        ).to(RUN_DEVICE)

        # Uses processor-reported original sizes for coordinate-space mapping.
        target_sizes = inputs["original_sizes"].detach().cpu().tolist()

        with torch.no_grad():
            outputs = model(**inputs)

        # ---------------------------------------------------------------------
        # 5e. Post-process raw SAM3 outputs
        # ---------------------------------------------------------------------
        # Converts raw model outputs into masks / boxes / scores.
        results = processor.post_process_instance_segmentation(
            outputs,
            threshold=TEXT_THRESHOLD,
            mask_threshold=MASK_THRESHOLD,
            target_sizes=target_sizes,
        )

        result = results[0] if isinstance(results, list) and len(results) > 0 else {}

        masks = normalize_masks(result.get("masks", None))
        num_masks = int(masks.shape[0]) if masks.ndim == 3 else 0

        boxes = normalize_boxes(result.get("boxes", None), num_masks)
        scores = normalize_scores(result.get("scores", None), num_masks)

        # ---------------------------------------------------------------------
        # 5f. Build raw detections table
        # ---------------------------------------------------------------------
        # Creates a per-detection table for this image before filtering.
        if num_masks == 0:
            raw_detections_df = pd.DataFrame(
                columns=["orig_det_idx", "score", "x1", "y1", "x2", "y2"]
            )
            kept_after_containment_df = raw_detections_df.copy()
            removed_by_containment_df = raw_detections_df.copy()
        else:
            raw_detections_df = pd.DataFrame({
                "orig_det_idx": np.arange(num_masks, dtype=int),
                "score": scores.astype(float),
                "x1": boxes[:, 0].astype(float),
                "y1": boxes[:, 1].astype(float),
                "x2": boxes[:, 2].astype(float),
                "y2": boxes[:, 3].astype(float),
            })

            # Sorts raw detections by score descending for stable reading.
            raw_detections_df = raw_detections_df.sort_values(
                "score",
                ascending=False
            ).reset_index(drop=True)

            # -------------------------------------------------------------
            # First pass: containment suppression
            # -------------------------------------------------------------
            # Removes smaller detections mostly inside bigger ones.
            kept_after_containment_df, removed_by_containment_df = suppress_contained_shorter_detections(
                detections_df=raw_detections_df,
                containment_threshold=CONTAINMENT_THRESHOLD,
                min_area_ratio=MIN_AREA_RATIO,
                min_score_advantage=MIN_SCORE_ADVANTAGE,
            )

        # ---------------------------------------------------------------------
        # 5g. Second pass: keep only the main cluster
        # ---------------------------------------------------------------------
        # Removes detections too far from the main crossarm group.
        if len(kept_after_containment_df) == 0:
            final_kept_detections_df = kept_after_containment_df.copy()
            removed_by_cluster_df = kept_after_containment_df.copy()
            cluster_threshold_used = 0.0
        else:
            final_kept_detections_df, removed_by_cluster_df, cluster_threshold_used = keep_main_detection_cluster(
                detections_df=kept_after_containment_df,
                center_dist_factor=CENTER_DIST_FACTOR,
            )

        # ---------------------------------------------------------------------
        # 5h. Assign Xarm_1 / Xarm_2 / ... labels
        # ---------------------------------------------------------------------
        # Orders final detections top-to-bottom using box center coordinates.
        if len(final_kept_detections_df) > 0:
            final_kept_detections_df = final_kept_detections_df.copy()

            if "cx" not in final_kept_detections_df.columns:
                final_kept_detections_df["cx"] = (
                    final_kept_detections_df["x1"] + final_kept_detections_df["x2"]
                ) / 2.0

            if "cy" not in final_kept_detections_df.columns:
                final_kept_detections_df["cy"] = (
                    final_kept_detections_df["y1"] + final_kept_detections_df["y2"]
                ) / 2.0

            final_kept_detections_df = final_kept_detections_df.sort_values(
                ["cy", "cx"],
                ascending=[True, True]
            ).reset_index(drop=True)

            final_kept_detections_df["xarm_label"] = [
                f"Xarm_{i+1}" for i in range(len(final_kept_detections_df))
            ]

        # ---------------------------------------------------------------------
        # 5h.1 Collect final-kept bbox diagnostics
        # ---------------------------------------------------------------------
        # EXPLANATION:
        # This stores one row per FINAL kept crossarm detection so we can inspect:
        # - bbox width / height
        # - long vs short side
        # - aspect ratio (long / short)
        # - a provisional square-ish flag
        #
        # We sort/display this later to see whether square-ish boxes are a good
        # first-stage diagnostic for suspicious crossarm merges.
        if len(final_kept_detections_df) > 0:
            for _, det_row in final_kept_detections_df.iterrows():
                x1 = float(det_row["x1"])
                y1 = float(det_row["y1"])
                x2 = float(det_row["x2"])
                y2 = float(det_row["y2"])

                bbox_w = max(0.0, x2 - x1)
                bbox_h = max(0.0, y2 - y1)

                long_side = max(bbox_w, bbox_h)
                short_side = min(bbox_w, bbox_h)

                aspect_ratio_long_over_short = (
                    long_side / short_side if short_side > 0 else np.inf
                )

                aspect_ratio_short_over_long = (
                    short_side / long_side if long_side > 0 else 0.0
                )

                bbox_rows.append({
                    "debug_row_index": plot_idx,
                    "image_id": image_id,
                    "file_name": file_name,
                    "image_path": image_path,
                    "orig_det_idx": int(det_row["orig_det_idx"]),
                    "xarm_label": det_row["xarm_label"],
                    "score": float(det_row["score"]),
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "bbox_w": bbox_w,
                    "bbox_h": bbox_h,
                    "long_side": long_side,
                    "short_side": short_side,
                    "aspect_ratio_long_over_short": aspect_ratio_long_over_short,
                    "aspect_ratio_short_over_long": aspect_ratio_short_over_long,
                    "is_squareish_by_threshold": bool(
                        aspect_ratio_long_over_short <= SQUAREISH_ASPECT_THRESHOLD
                    ),
                })

        # ---------------------------------------------------------------------
        # 5i. Plot the current image on its subplot
        # ---------------------------------------------------------------------
        # Draws base image first, then coloured masks, yellow boxes, and blue labels.
        ax.imshow(image)

        num_final_kept = len(final_kept_detections_df)

        if num_final_kept > 0:
            # -------------------------------------------------------------
            # Build a colormap for this image's kept detections
            # -------------------------------------------------------------
            # Each kept crossarm gets a different colour, matching the 6c style.
            try:
                cmap = plt.colormaps.get_cmap("tab10").resampled(max(num_final_kept, 1))
            except Exception:
                cmap = plt.cm.get_cmap("tab10", max(num_final_kept, 1))

            for det_plot_idx, det_row in final_kept_detections_df.iterrows():
                orig_idx = int(det_row["orig_det_idx"])

                mask_i = masks[orig_idx]
                box_i = boxes[orig_idx]
                score_i = float(det_row["score"])
                xarm_label = det_row["xarm_label"]

                # ---------------------------------------------------------
                # Multi-colour mask fill
                # ---------------------------------------------------------
                # Uses a different RGB colour for each kept crossarm.
                color_rgba = cmap(det_plot_idx)

                overlay = np.zeros((mask_i.shape[0], mask_i.shape[1], 4), dtype=np.float32)
                overlay[..., 0] = color_rgba[0]
                overlay[..., 1] = color_rgba[1]
                overlay[..., 2] = color_rgba[2]

                # Only masked pixels receive alpha; background stays transparent.
                overlay[..., 3] = mask_i.astype(np.float32) * MASK_ALPHA
                ax.imshow(overlay)

                # ---------------------------------------------------------
                # SAM3 predicted box (yellow)
                # ---------------------------------------------------------
                # Keeps the original SAM3 box styling for easy comparison.
                x1, y1, x2, y2 = [float(v) for v in box_i]

                rect = patches.Rectangle(
                    (x1, y1),
                    max(0.0, x2 - x1),
                    max(0.0, y2 - y1),
                    linewidth=1.8,
                    edgecolor="yellow",
                    facecolor="none",
                )
                ax.add_patch(rect)

                # ---------------------------------------------------------
                # Xarm label (fixed blue label box)
                # ---------------------------------------------------------
                # Keeps all labels the same blue, as per your preferred style.
                ax.text(
                    x1,
                    max(5, y1 - 5),
                    f"{xarm_label}: {score_i:.3f}",
                    color="white",
                    fontsize=8,
                    bbox=dict(facecolor=LABEL_BG, alpha=0.95, pad=1.2, edgecolor="none"),
                )

        # ---------------------------------------------------------------------
        # 5j. Set subplot title
        # ---------------------------------------------------------------------
        # Keeps titles short enough for a multi-image grid.
        ax.set_title(
            f"{file_name}\nRaw={num_masks} | Kept={num_final_kept}",
            fontsize=9,
            pad=8,
        )
        ax.axis("off")

        # ---------------------------------------------------------------------
        # 5k. Save summary row for this image
        # ---------------------------------------------------------------------
        # Stores a compact result summary for the batch table shown later.
        batch_rows.append({
            "debug_row_index": plot_idx,
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": original_mode,
            "width": image_w,
            "height": image_h,
            "raw_num_masks": int(num_masks),
            "kept_after_containment": int(len(kept_after_containment_df)),
            "removed_by_containment": int(len(removed_by_containment_df)),
            "final_kept": int(len(final_kept_detections_df)),
            "removed_by_cluster": int(len(removed_by_cluster_df)),
            "cluster_threshold_used": float(cluster_threshold_used),
            "xarm_labels": ", ".join(final_kept_detections_df["xarm_label"].tolist()) if len(final_kept_detections_df) > 0 else "",
            "run_status": "ok",
            "error_message": "",
        })

        # Frees the per-image object before moving to the next image.
        del image

    except Exception as e:
        # ---------------------------------------------------------------------
        # 5l. Graceful error display for the current image
        # ---------------------------------------------------------------------
        # Shows an error message inside the subplot instead of breaking the grid.
        ax.set_facecolor("white")
        ax.text(
            0.5, 0.5,
            f"FAILED\n{file_name}\n\n{type(e).__name__}\n{str(e)}",
            ha="center",
            va="center",
            fontsize=9,
            wrap=True,
            transform=ax.transAxes,
        )
        ax.set_title(f"{file_name}\nFAILED", fontsize=9, pad=8)
        ax.axis("off")

        batch_rows.append({
            "debug_row_index": plot_idx,
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": None,
            "width": np.nan,
            "height": np.nan,
            "raw_num_masks": np.nan,
            "kept_after_containment": np.nan,
            "removed_by_containment": np.nan,
            "final_kept": np.nan,
            "removed_by_cluster": np.nan,
            "cluster_threshold_used": np.nan,
            "xarm_labels": "",
            "run_status": "failed",
            "error_message": str(e),
        })

# -----------------------------------------------------------------------------
# 6. Turn off any unused axes
# -----------------------------------------------------------------------------
# Hides extra empty subplot panels when image count is not a multiple of 3.
for ax in axes[n_images:]:
    ax.axis("off")

# -----------------------------------------------------------------------------
# 7. Global figure title
# -----------------------------------------------------------------------------
# Adds one overall title for the whole presentation grid.
fig.suptitle(
    f"Debug Grid — Prompt: {PROMPT_TEXT}",
    fontsize=14,
)

plt.show()
plt.close()

# -----------------------------------------------------------------------------
# 8. Build and display batch summary table
# -----------------------------------------------------------------------------
# Creates a compact per-image summary table for the whole debug batch.
debug_batch_results_df = pd.DataFrame(batch_rows)

print("\nBatch summary table:")
try:
    display(debug_batch_results_df)
except NameError:
    print(debug_batch_results_df)

# -----------------------------------------------------------------------------
# 8b. Build and display crossarm bbox diagnostics table
# -----------------------------------------------------------------------------
# EXPLANATION:
# This table is the key QA output for deciding whether bbox shape is a useful
# first-stage diagnostic.
#
# IMPORTANT:
# - this is built from FINAL kept detections only
# - aspect_ratio_long_over_short near 1.0 means more square-like
# - larger values mean more elongated
crossarm_bbox_dims_df = pd.DataFrame(bbox_rows)

if crossarm_bbox_dims_df.empty:
    print("\nNo final kept crossarm detections were available for bbox diagnostics.")
else:
    crossarm_bbox_dims_df = crossarm_bbox_dims_df.sort_values(
        ["aspect_ratio_long_over_short", "file_name", "xarm_label"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    squareish_bbox_dims_df = crossarm_bbox_dims_df[
        crossarm_bbox_dims_df["aspect_ratio_long_over_short"] <= SQUAREISH_ASPECT_THRESHOLD
    ].reset_index(drop=True)

    print("\nCrossarm bbox diagnostics table (FINAL kept detections):")
    print(f"Square-ish threshold (long/short <=): {SQUAREISH_ASPECT_THRESHOLD:.2f}")
    print(f"Total final kept crossarms           : {len(crossarm_bbox_dims_df)}")
    print(f"Square-ish flagged crossarms         : {len(squareish_bbox_dims_df)}")

    cols_to_show = [
        "file_name",
        "xarm_label",
        "score",
        "bbox_w",
        "bbox_h",
        "long_side",
        "short_side",
        "aspect_ratio_long_over_short",
        "aspect_ratio_short_over_long",
        "is_squareish_by_threshold",
    ]

    print("\nAll final kept crossarm bbox dimensions:")
    try:
        display(crossarm_bbox_dims_df[cols_to_show])
    except NameError:
        print(crossarm_bbox_dims_df[cols_to_show])

    print("\nSquare-ish subset only:")
    try:
        display(squareish_bbox_dims_df[cols_to_show])
    except NameError:
        print(squareish_bbox_dims_df[cols_to_show])

    print("\nAspect-ratio summary (long/short):")
    print(crossarm_bbox_dims_df["aspect_ratio_long_over_short"].describe())

# -----------------------------------------------------------------------------
# 9. Final note
# -----------------------------------------------------------------------------
print("\nCell 6d completed successfully.")
print("This shows ALL images in debug_images_df using:")
print("- one prompt only")
print("- containment-based suppression")
print("- main-cluster false-positive filtering")
print("- different colours per final kept mask")
print("- yellow SAM3 boxes")
print("- blue Xarm_1 / Xarm_2 / ... labels")
print("- 3-column presentation grid")
print("- bbox dimension diagnostics for final kept crossarms")

In [ ]:
# =============================================================================
# CELL 6d — RUN ONE PROMPT ON ALL DEBUG IMAGES
#            + CONTAINMENT SUPPRESSION
#            + MAIN-CLUSTER FALSE-POSITIVE FILTER
#            + MULTI-COLOUR MASK DISPLAY
#            + Xarm_1 / Xarm_2 / ... LABELS
#            + 3-COLUMN PRESENTATION GRID
# =============================================================================
# EXPLANATION:
# This cell applies the SAME single-prompt crossarm workflow from Cell 6c to
# ALL images in debug_images_df.
#
# WHAT IT DOES:
# - loops through every image in debug_images_df
# - runs SAM3 with the single prompt:
#       "utility pole crossarm"
# - applies the SAME two post-processing stages from Cell 6c:
#       1) containment suppression
#       2) main-cluster false-positive filtering
# - assigns Xarm_1 / Xarm_2 / ... labels per image
# - displays all debug images in one grid:
#       - 3 images per row
#       - fixed panel size
#       - presentation-friendly overall size
#
# VISUAL STYLE:
# - final kept masks = different colours per crossarm
# - light transparency
# - no border / no outline
# - SAM3 predicted boxes = yellow
# - Xarm labels = fixed blue label box with white text
#
# IMPORTANT:
# - this cell does NOT save masks/crops
# - this cell is still a debug / QA visualization step
# - this cell REUSES helper functions already defined in Cell 6c
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
# os        : file/path checks
# math      : grid row calculation
# numpy     : numeric helpers
# pandas    : summary tables
# matplotlib: grid plotting
# torch     : device fallback
# PIL       : image loading / RGB conversion
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
# Confirms the required objects from earlier cells already exist.
required_globals = [
    "debug_images_df",
    "model",
    "processor",
    "normalize_masks",
    "normalize_boxes",
    "normalize_scores",
    "suppress_contained_shorter_detections",
    "keep_main_detection_cluster",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 6d requires objects/functions from earlier cells.\n"
        "Please run Cell 5b and the current Cell 6c first.\n"
        f"Missing globals: {missing_globals}"
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError("debug_images_df exists but is not a pandas DataFrame.")

if debug_images_df.empty:
    raise ValueError("debug_images_df is empty.")

# -----------------------------------------------------------------------------
# 1. Prompt, thresholds, and post-processing settings
# -----------------------------------------------------------------------------
# Uses the SAME single prompt and cleanup settings as Cell 6c.
PROMPT_TEXT = "utility pole crossarm"

TEXT_THRESHOLD = float(globals().get("TEXT_SCORE_THRESHOLD", 0.30))
MASK_THRESHOLD = float(globals().get("MASK_THRESHOLD", 0.50))

RUN_DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# Containment suppression settings
CONTAINMENT_THRESHOLD = 0.80
MIN_AREA_RATIO = 1.20
MIN_SCORE_ADVANTAGE = 0.0

# Main-cluster filtering settings
CENTER_DIST_FACTOR = 2.75

# -----------------------------------------------------------------------------
# 2. Grid display settings
# -----------------------------------------------------------------------------
# Uses a fixed, presentation-friendly panel size and a 3-column layout.
N_COLS = 3
PANEL_W = 4.8
PANEL_H = 3.6
GRID_DPI = 120

# Multi-colour mask display settings
# Each kept mask gets a different colour from the colormap.
MASK_ALPHA = 0.45

# Fixed blue label style to match your preferred 6c view
LABEL_BG = "#1E90FF"

# -----------------------------------------------------------------------------
# 3. Choose which images to show
# -----------------------------------------------------------------------------
# This cell uses the current debug subset directly.
grid_df = debug_images_df.copy().reset_index(drop=True)

n_images = len(grid_df)
n_rows = math.ceil(n_images / N_COLS)

print("Batch debug run over debug_images_df:\n")
print(f"  n_images              : {n_images}")
print(f"  prompt                : {PROMPT_TEXT}")
print(f"  TEXT_THRESHOLD        : {TEXT_THRESHOLD}")
print(f"  MASK_THRESHOLD        : {MASK_THRESHOLD}")
print(f"  RUN_DEVICE            : {RUN_DEVICE}")
print(f"  CONTAINMENT_THRESHOLD : {CONTAINMENT_THRESHOLD}")
print(f"  MIN_AREA_RATIO        : {MIN_AREA_RATIO}")
print(f"  MIN_SCORE_ADVANTAGE   : {MIN_SCORE_ADVANTAGE}")
print(f"  CENTER_DIST_FACTOR    : {CENTER_DIST_FACTOR}")
print(f"  grid_cols             : {N_COLS}")
print(f"  grid_rows             : {n_rows}")
print(f"  panel_size            : {PANEL_W} x {PANEL_H}")

# -----------------------------------------------------------------------------
# 4. Create the figure grid
# -----------------------------------------------------------------------------
# Creates one subplot per image and later turns off any unused panels.
fig, axes = plt.subplots(
    n_rows,
    N_COLS,
    figsize=(N_COLS * PANEL_W, n_rows * PANEL_H),
    dpi=GRID_DPI,
    constrained_layout=True,
)

# Normalizes axes into a flat 1D array so iteration is simple.
axes = np.array(axes).reshape(-1)

# -----------------------------------------------------------------------------
# 5. Loop over every debug image
# -----------------------------------------------------------------------------
# Each image is processed independently so one failure does not stop the rest.
batch_rows = []

for plot_idx, (_, row) in enumerate(grid_df.iterrows()):
    ax = axes[plot_idx]

    # -------------------------------------------------------------------------
    # 5a. Extract metadata for the current image
    # -------------------------------------------------------------------------
    image_path = row["image_path"]

    image_id = row.get("image_id", None)
    if pd.isna(image_id):
        image_id = None

    file_name = row.get("file_name", None)
    if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
        file_name = os.path.basename(image_path)

    try:
        # ---------------------------------------------------------------------
        # 5b. Validate image path
        # ---------------------------------------------------------------------
        # Fails cleanly if the path is missing or invalid.
        if not isinstance(image_path, str) or len(image_path.strip()) == 0:
            raise ValueError(f"Invalid image_path: {image_path}")

        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image file not found: {image_path}")

        # ---------------------------------------------------------------------
        # 5c. Load the image as RGB
        # ---------------------------------------------------------------------
        # Converts to RGB if required and fully loads pixels into memory.
        with Image.open(image_path) as img:
            original_mode = img.mode

            if original_mode != "RGB":
                image = img.convert("RGB")
            else:
                image = img.copy()

            image.load()

        image_w, image_h = image.size

        # ---------------------------------------------------------------------
        # 5d. Build SAM3 inputs and run inference
        # ---------------------------------------------------------------------
        # Runs the single crossarm prompt on the current image.
        inputs = processor(
            images=image,
            text=PROMPT_TEXT,
            return_tensors="pt",
        ).to(RUN_DEVICE)

        # Uses processor-reported original sizes for coordinate-space mapping.
        target_sizes = inputs["original_sizes"].detach().cpu().tolist()

        with torch.no_grad():
            outputs = model(**inputs)

        # ---------------------------------------------------------------------
        # 5e. Post-process raw SAM3 outputs
        # ---------------------------------------------------------------------
        # Converts raw model outputs into masks / boxes / scores.
        results = processor.post_process_instance_segmentation(
            outputs,
            threshold=TEXT_THRESHOLD,
            mask_threshold=MASK_THRESHOLD,
            target_sizes=target_sizes,
        )

        result = results[0] if isinstance(results, list) and len(results) > 0 else {}

        masks = normalize_masks(result.get("masks", None))
        num_masks = int(masks.shape[0]) if masks.ndim == 3 else 0

        boxes = normalize_boxes(result.get("boxes", None), num_masks)
        scores = normalize_scores(result.get("scores", None), num_masks)

        # ---------------------------------------------------------------------
        # 5f. Build raw detections table
        # ---------------------------------------------------------------------
        # Creates a per-detection table for this image before filtering.
        if num_masks == 0:
            raw_detections_df = pd.DataFrame(
                columns=["orig_det_idx", "score", "x1", "y1", "x2", "y2"]
            )
            kept_after_containment_df = raw_detections_df.copy()
            removed_by_containment_df = raw_detections_df.copy()
        else:
            raw_detections_df = pd.DataFrame({
                "orig_det_idx": np.arange(num_masks, dtype=int),
                "score": scores.astype(float),
                "x1": boxes[:, 0].astype(float),
                "y1": boxes[:, 1].astype(float),
                "x2": boxes[:, 2].astype(float),
                "y2": boxes[:, 3].astype(float),
            })

            # Sorts raw detections by score descending for stable reading.
            raw_detections_df = raw_detections_df.sort_values(
                "score",
                ascending=False
            ).reset_index(drop=True)

            # -------------------------------------------------------------
            # First pass: containment suppression
            # -------------------------------------------------------------
            # Removes smaller detections mostly inside bigger ones.
            kept_after_containment_df, removed_by_containment_df = suppress_contained_shorter_detections(
                detections_df=raw_detections_df,
                containment_threshold=CONTAINMENT_THRESHOLD,
                min_area_ratio=MIN_AREA_RATIO,
                min_score_advantage=MIN_SCORE_ADVANTAGE,
            )

        # ---------------------------------------------------------------------
        # 5g. Second pass: keep only the main cluster
        # ---------------------------------------------------------------------
        # Removes detections too far from the main crossarm group.
        if len(kept_after_containment_df) == 0:
            final_kept_detections_df = kept_after_containment_df.copy()
            removed_by_cluster_df = kept_after_containment_df.copy()
            cluster_threshold_used = 0.0
        else:
            final_kept_detections_df, removed_by_cluster_df, cluster_threshold_used = keep_main_detection_cluster(
                detections_df=kept_after_containment_df,
                center_dist_factor=CENTER_DIST_FACTOR,
            )

        # ---------------------------------------------------------------------
        # 5h. Assign Xarm_1 / Xarm_2 / ... labels
        # ---------------------------------------------------------------------
        # Orders final detections top-to-bottom using box center coordinates.
        if len(final_kept_detections_df) > 0:
            final_kept_detections_df = final_kept_detections_df.copy()

            if "cx" not in final_kept_detections_df.columns:
                final_kept_detections_df["cx"] = (
                    final_kept_detections_df["x1"] + final_kept_detections_df["x2"]
                ) / 2.0

            if "cy" not in final_kept_detections_df.columns:
                final_kept_detections_df["cy"] = (
                    final_kept_detections_df["y1"] + final_kept_detections_df["y2"]
                ) / 2.0

            final_kept_detections_df = final_kept_detections_df.sort_values(
                ["cy", "cx"],
                ascending=[True, True]
            ).reset_index(drop=True)

            final_kept_detections_df["xarm_label"] = [
                f"Xarm_{i+1}" for i in range(len(final_kept_detections_df))
            ]

        # ---------------------------------------------------------------------
        # 5i. Plot the current image on its subplot
        # ---------------------------------------------------------------------
        # Draws base image first, then coloured masks, yellow boxes, and blue labels.
        ax.imshow(image)

        num_final_kept = len(final_kept_detections_df)

        if num_final_kept > 0:
            # -------------------------------------------------------------
            # Build a colormap for this image's kept detections
            # -------------------------------------------------------------
            # Each kept crossarm gets a different colour, matching the 6c style.
            try:
                cmap = plt.colormaps.get_cmap("tab10").resampled(max(num_final_kept, 1))
            except Exception:
                cmap = plt.cm.get_cmap("tab10", max(num_final_kept, 1))

            for det_plot_idx, det_row in final_kept_detections_df.iterrows():
                orig_idx = int(det_row["orig_det_idx"])

                mask_i = masks[orig_idx]
                box_i = boxes[orig_idx]
                score_i = float(det_row["score"])
                xarm_label = det_row["xarm_label"]

                # ---------------------------------------------------------
                # Multi-colour mask fill
                # ---------------------------------------------------------
                # Uses a different RGB colour for each kept crossarm.
                color_rgba = cmap(det_plot_idx)

                overlay = np.zeros((mask_i.shape[0], mask_i.shape[1], 4), dtype=np.float32)
                overlay[..., 0] = color_rgba[0]
                overlay[..., 1] = color_rgba[1]
                overlay[..., 2] = color_rgba[2]

                # Only masked pixels receive alpha; background stays transparent.
                overlay[..., 3] = mask_i.astype(np.float32) * MASK_ALPHA
                ax.imshow(overlay)

                # ---------------------------------------------------------
                # SAM3 predicted box (yellow)
                # ---------------------------------------------------------
                # Keeps the original SAM3 box styling for easy comparison.
                x1, y1, x2, y2 = [float(v) for v in box_i]

                rect = patches.Rectangle(
                    (x1, y1),
                    max(0.0, x2 - x1),
                    max(0.0, y2 - y1),
                    linewidth=1.8,
                    edgecolor="yellow",
                    facecolor="none",
                )
                ax.add_patch(rect)

                # ---------------------------------------------------------
                # Xarm label (fixed blue label box)
                # ---------------------------------------------------------
                # Keeps all labels the same blue, as per your preferred style.
                ax.text(
                    x1,
                    max(5, y1 - 5),
                    f"{xarm_label}: {score_i:.3f}",
                    color="white",
                    fontsize=8,
                    bbox=dict(facecolor=LABEL_BG, alpha=0.95, pad=1.2, edgecolor="none"),
                )

        # ---------------------------------------------------------------------
        # 5j. Set subplot title
        # ---------------------------------------------------------------------
        # Keeps titles short enough for a multi-image grid.
        ax.set_title(
            f"{file_name}\nRaw={num_masks} | Kept={num_final_kept}",
            fontsize=9,
            pad=8,
        )
        ax.axis("off")

        # ---------------------------------------------------------------------
        # 5k. Save summary row for this image
        # ---------------------------------------------------------------------
        # Stores a compact result summary for the batch table shown later.
        batch_rows.append({
            "debug_row_index": plot_idx,
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": original_mode,
            "width": image_w,
            "height": image_h,
            "raw_num_masks": int(num_masks),
            "kept_after_containment": int(len(kept_after_containment_df)),
            "removed_by_containment": int(len(removed_by_containment_df)),
            "final_kept": int(len(final_kept_detections_df)),
            "removed_by_cluster": int(len(removed_by_cluster_df)),
            "cluster_threshold_used": float(cluster_threshold_used),
            "xarm_labels": ", ".join(final_kept_detections_df["xarm_label"].tolist()) if len(final_kept_detections_df) > 0 else "",
            "run_status": "ok",
            "error_message": "",
        })

        # Frees the per-image object before moving to the next image.
        del image

    except Exception as e:
        # ---------------------------------------------------------------------
        # 5l. Graceful error display for the current image
        # ---------------------------------------------------------------------
        # Shows an error message inside the subplot instead of breaking the grid.
        ax.set_facecolor("white")
        ax.text(
            0.5, 0.5,
            f"FAILED\n{file_name}\n\n{type(e).__name__}\n{str(e)}",
            ha="center",
            va="center",
            fontsize=9,
            wrap=True,
            transform=ax.transAxes,
        )
        ax.set_title(f"{file_name}\nFAILED", fontsize=9, pad=8)
        ax.axis("off")

        batch_rows.append({
            "debug_row_index": plot_idx,
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": None,
            "width": np.nan,
            "height": np.nan,
            "raw_num_masks": np.nan,
            "kept_after_containment": np.nan,
            "removed_by_containment": np.nan,
            "final_kept": np.nan,
            "removed_by_cluster": np.nan,
            "cluster_threshold_used": np.nan,
            "xarm_labels": "",
            "run_status": "failed",
            "error_message": str(e),
        })

# -----------------------------------------------------------------------------
# 6. Turn off any unused axes
# -----------------------------------------------------------------------------
# Hides extra empty subplot panels when image count is not a multiple of 3.
for ax in axes[n_images:]:
    ax.axis("off")

# -----------------------------------------------------------------------------
# 7. Global figure title
# -----------------------------------------------------------------------------
# Adds one overall title for the whole presentation grid.
fig.suptitle(
    f"Debug Grid — Prompt: {PROMPT_TEXT}",
    fontsize=14,
)

plt.show()
plt.close()

# -----------------------------------------------------------------------------
# 8. Build and display batch summary table
# -----------------------------------------------------------------------------
# Creates a compact per-image summary table for the whole debug batch.
debug_batch_results_df = pd.DataFrame(batch_rows)

print("\nBatch summary table:")
try:
    display(debug_batch_results_df)
except NameError:
    print(debug_batch_results_df)

# -----------------------------------------------------------------------------
# 9. Final note
# -----------------------------------------------------------------------------
print("\nCell 6d completed successfully.")
print("This shows ALL images in debug_images_df using:")
print("- one prompt only")
print("- containment-based suppression")
print("- main-cluster false-positive filtering")
print("- different colours per final kept mask")
print("- yellow SAM3 boxes")
print("- blue Xarm_1 / Xarm_2 / ... labels")
print("- 3-column presentation grid")

In [ ]:
# =============================================================================
# CELL 6c — ONE PROMPT ON ONE DEBUG IMAGE
#            + CONTAINMENT SUPPRESSION
#            + MAIN-CLUSTER FALSE-POSITIVE FILTER
#            + UNIFORM PURPLE MASK DISPLAY
#            + Xarm_1 / Xarm_2 / ... LABELS
# =============================================================================
# EXPLANATION:
# This cell runs SAM3 on exactly ONE debug image using exactly ONE prompt:
#
#   "utility pole crossarm"
#
# It then applies two post-processing steps:
#
#   1) containment suppression
#      - if a shorter / smaller detection is mostly inside a larger crossarm
#        detection, remove the smaller one
#
#   2) main-cluster filtering
#      - if a detection is far away from the main crossarm group / cluster,
#        remove it as an isolated false positive
#
# Finally, it visualizes ONLY the final kept detections using:
# - the SAME purple colour for all kept masks
# - light transparency
# - NO outer border / outline
# - yellow SAM3 boxes
# - purple Xarm_1 / Xarm_2 / ... labels
#
# IMPORTANT:
# - the mask itself is NOT changed
# - this cell does NOT merge separated fragments across visible gaps
# - this is still a debugging / inspection cell
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
# os        : file/path checks
# numpy     : numeric array handling
# pandas    : tabular summaries for detections
# matplotlib: image, mask, and box visualization
# torch     : model inference
# PIL       : image loading / RGB conversion
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
# Confirms that the required objects from previous cells exist.
if "debug_images_df" not in globals():
    raise NameError(
        "debug_images_df not found.\n"
        "Please run Cell 6a first."
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError("debug_images_df exists but is not a pandas DataFrame.")

if debug_images_df.empty:
    raise ValueError("debug_images_df is empty.")

if "model" not in globals():
    raise NameError("model not found. Please run Cell 5b first.")

if "processor" not in globals():
    raise NameError("processor not found. Please run Cell 5b first.")

# -----------------------------------------------------------------------------
# 1. Choose exactly one debug image
# -----------------------------------------------------------------------------
# Keep this as a single-image debug cell.
# Change DEBUG_ROW_INDEX whenever you want to inspect another image.
DEBUG_ROW_INDEX = 7

if DEBUG_ROW_INDEX >= len(debug_images_df):
    raise IndexError(
        f"DEBUG_ROW_INDEX={DEBUG_ROW_INDEX} is out of range for "
        f"debug_images_df with {len(debug_images_df)} rows."
    )

# Pulls the chosen row from the debug subset.
row = debug_images_df.iloc[DEBUG_ROW_INDEX]

# -----------------------------------------------------------------------------
# 2. Extract metadata from the selected row
# -----------------------------------------------------------------------------
# Reads the core metadata needed for loading, prints, and plot titles.
image_path = row["image_path"]

# Safely fetches image_id if available.
image_id = row.get("image_id", None)
if pd.isna(image_id):
    image_id = None

# Safely fetches file_name if available; otherwise derives it from the path.
file_name = row.get("file_name", None)
if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
    file_name = os.path.basename(image_path)

# -----------------------------------------------------------------------------
# 3. Confirm the file exists before opening it
# -----------------------------------------------------------------------------
# Validates that the selected path is usable.
if not isinstance(image_path, str) or len(image_path.strip()) == 0:
    raise ValueError(f"Invalid image_path: {image_path}")

# Validates that the file really exists on disk.
if not os.path.exists(image_path):
    raise FileNotFoundError(
        "Selected image file does not exist on disk.\n"
        f"image_path: {image_path}"
    )

# -----------------------------------------------------------------------------
# 4. Load the image as RGB
# -----------------------------------------------------------------------------
# Opens the image and converts to RGB if needed.
# image.load() forces pixel data into memory before the file handle closes.
with Image.open(image_path) as img:
    original_mode = img.mode

    # Converts non-RGB images so SAM3 input format stays predictable.
    if original_mode != "RGB":
        print(
            f"WARNING: Image mode is '{original_mode}', not 'RGB'. "
            "Converting to RGB."
        )
        image = img.convert("RGB")
    else:
        image = img.copy()

    # Forces a full decode into memory before leaving the with-block.
    image.load()

# Stores image size for debug prints and any geometry reasoning.
image_w, image_h = image.size

# -----------------------------------------------------------------------------
# 5. Prompt, thresholds, and post-processing settings
# -----------------------------------------------------------------------------
# Uses exactly one prompt for this diagnostic step.
PROMPT_TEXT = "utility pole crossarm"

# Pulls thresholds from notebook globals if they already exist.
TEXT_THRESHOLD = float(globals().get("TEXT_SCORE_THRESHOLD", 0.30))
MASK_THRESHOLD = float(globals().get("MASK_THRESHOLD", 0.50))

# Reuses the notebook device if present; otherwise chooses a sensible fallback.
RUN_DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------------------------------------------------
# Containment suppression settings
# -----------------------------------------------------------------------------
# A smaller detection will be removed if:
# - at least CONTAINMENT_THRESHOLD fraction of its box is covered by a larger box
# - the larger box is at least MIN_AREA_RATIO times bigger
# - the larger box has at least the same (or better) score
CONTAINMENT_THRESHOLD = 0.80
MIN_AREA_RATIO = 1.20
MIN_SCORE_ADVANTAGE = 0.0

# -----------------------------------------------------------------------------
# Main-cluster filtering settings
# -----------------------------------------------------------------------------
# Detections that are too far from the main crossarm group are removed.
#
# CENTER_DIST_FACTOR:
# - pairwise center distance threshold = CENTER_DIST_FACTOR * median box diagonal
# - detections connected by this rule belong to the same cluster
# - we keep the strongest / largest connected component
CENTER_DIST_FACTOR = 2.75

print("Single-prompt SAM3 debug run:\n")
print(f"  DEBUG_ROW_INDEX       : {DEBUG_ROW_INDEX}")
print(f"  image_id              : {image_id}")
print(f"  file_name             : {file_name}")
print(f"  image_path            : {image_path}")
print(f"  width                 : {image_w}")
print(f"  height                : {image_h}")
print(f"  original_mode         : {original_mode}")
print(f"  final_mode            : {image.mode}")
print(f"  prompt                : {PROMPT_TEXT}")
print(f"  TEXT_THRESHOLD        : {TEXT_THRESHOLD}")
print(f"  MASK_THRESHOLD        : {MASK_THRESHOLD}")
print(f"  RUN_DEVICE            : {RUN_DEVICE}")
print(f"  CONTAINMENT_THRESHOLD : {CONTAINMENT_THRESHOLD}")
print(f"  MIN_AREA_RATIO        : {MIN_AREA_RATIO}")
print(f"  MIN_SCORE_ADVANTAGE   : {MIN_SCORE_ADVANTAGE}")
print(f"  CENTER_DIST_FACTOR    : {CENTER_DIST_FACTOR}")

# -----------------------------------------------------------------------------
# 6. Helper functions to normalize SAM3 outputs
# -----------------------------------------------------------------------------
# These helpers convert SAM3 outputs into consistent NumPy arrays so the rest
# of the cell can assume a stable format.
def to_numpy(x):
    """
    Convert tensor-like objects to NumPy arrays.

    Args:
        x:
            Tensor, list, scalar, or array-like value.

    Returns:
        np.ndarray
    """
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def normalize_masks(masks):
    """
    Normalize masks to shape (N, H, W), bool.

    Args:
        masks:
            Raw masks returned by SAM3 post-processing.

    Returns:
        np.ndarray
    """
    # No output at all -> safe empty fallback.
    if masks is None:
        return np.zeros((0, 0, 0), dtype=bool)

    # Convert to NumPy first for consistent handling.
    arr = to_numpy(masks)

    # Empty object -> safe empty fallback.
    if arr.size == 0:
        return np.zeros((0, 0, 0), dtype=bool)

    # Single-mask case: (H, W) -> (1, H, W)
    if arr.ndim == 2:
        arr = arr[None, ...]

    return arr.astype(bool)


def normalize_boxes(boxes, num_masks):
    """
    Normalize boxes to shape (N, 4).

    Args:
        boxes:
            Raw boxes returned by SAM3 post-processing.
        num_masks:
            Number of masks.

    Returns:
        np.ndarray
    """
    # Missing boxes -> safe zero fallback aligned to mask count.
    if boxes is None:
        return np.zeros((num_masks, 4), dtype=np.float32)

    arr = to_numpy(boxes).astype(np.float32)

    # Single-box case: [x1, y1, x2, y2] -> [[x1, y1, x2, y2]]
    if arr.ndim == 1 and arr.shape[0] == 4:
        arr = arr.reshape(1, 4)

    # Invalid shapes degrade safely to zeros.
    if arr.ndim != 2 or arr.shape[1] != 4:
        return np.zeros((num_masks, 4), dtype=np.float32)

    # Builds a fixed-size output aligned to mask count.
    out = np.zeros((num_masks, 4), dtype=np.float32)
    n = min(num_masks, len(arr))
    out[:n] = arr[:n]
    return out


def normalize_scores(scores, num_masks):
    """
    Normalize scores to shape (N,).

    Args:
        scores:
            Raw scores returned by SAM3 post-processing.
        num_masks:
            Number of masks.

    Returns:
        np.ndarray
    """
    # Missing scores -> safe zero fallback.
    if scores is None:
        return np.zeros((num_masks,), dtype=np.float32)

    arr = to_numpy(scores)

    # Scalar-score case.
    if arr.ndim == 0:
        if num_masks == 1:
            return np.array([float(arr)], dtype=np.float32)
        return np.zeros((num_masks,), dtype=np.float32)

    # Flattens into a simple 1D vector.
    arr = arr.astype(np.float32).reshape(-1)

    # Builds a fixed-size output aligned to mask count.
    out = np.zeros((num_masks,), dtype=np.float32)
    n = min(num_masks, len(arr))
    out[:n] = arr[:n]
    return out

# -----------------------------------------------------------------------------
# 6b. Helper functions for containment-based suppression
# -----------------------------------------------------------------------------
# Removes a shorter detection if it is mostly inside a larger one.
def box_area_xyxy(box_xyxy):
    """
    Compute area of an xyxy box.

    Args:
        box_xyxy:
            [x1, y1, x2, y2]

    Returns:
        float
    """
    x1, y1, x2, y2 = [float(v) for v in box_xyxy]
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def intersection_area_xyxy(box_a, box_b):
    """
    Compute intersection area between two xyxy boxes.

    Args:
        box_a:
            [x1, y1, x2, y2]
        box_b:
            [x1, y1, x2, y2]

    Returns:
        float
    """
    ax1, ay1, ax2, ay2 = [float(v) for v in box_a]
    bx1, by1, bx2, by2 = [float(v) for v in box_b]

    # Computes the overlap rectangle.
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    return max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)


def suppress_contained_shorter_detections(
    detections_df,
    containment_threshold=0.80,
    min_area_ratio=1.20,
    min_score_advantage=0.0,
):
    """
    Suppress smaller detections that are mostly contained inside a larger one.

    LOGIC:
    Detection j is suppressed if there exists detection i such that:
    - j is mostly inside i
    - i has at least the same (or better) score
    - i is meaningfully larger than j

    Args:
        detections_df:
            DataFrame with columns:
            - orig_det_idx
            - score
            - x1, y1, x2, y2

        containment_threshold:
            Fraction of the smaller box that must be covered by the larger box.

        min_area_ratio:
            Larger box area must be at least this ratio times the smaller box.

        min_score_advantage:
            Larger box score must exceed smaller box by at least this amount.

    Returns:
        tuple:
            kept_df, removed_df
    """
    if detections_df.empty:
        return detections_df.copy(), detections_df.iloc[0:0].copy()

    # Work on a clean copy so the original table stays untouched.
    df = detections_df.copy().reset_index(drop=True)

    # Precompute box areas once.
    df["box_area"] = df.apply(
        lambda r: box_area_xyxy([r["x1"], r["y1"], r["x2"], r["y2"]]),
        axis=1,
    )

    # Starts with every detection kept.
    keep_mask = np.ones(len(df), dtype=bool)
    removal_reason = [None] * len(df)

    # Compare each detection j against every other detection i.
    for j in range(len(df)):
        if not keep_mask[j]:
            continue

        area_j = float(df.loc[j, "box_area"])
        score_j = float(df.loc[j, "score"])
        box_j = [df.loc[j, "x1"], df.loc[j, "y1"], df.loc[j, "x2"], df.loc[j, "y2"]]

        # Invalid-area detections are removed immediately.
        if area_j <= 0:
            keep_mask[j] = False
            removal_reason[j] = "invalid_box_area"
            continue

        for i in range(len(df)):
            if i == j:
                continue

            area_i = float(df.loc[i, "box_area"])
            score_i = float(df.loc[i, "score"])
            box_i = [df.loc[i, "x1"], df.loc[i, "y1"], df.loc[i, "x2"], df.loc[i, "y2"]]

            # Candidate i must be meaningfully larger than j.
            if area_i < area_j * min_area_ratio:
                continue

            # Fraction of smaller box j covered by larger box i.
            inter_ij = intersection_area_xyxy(box_i, box_j)
            frac_small_inside_big = inter_ij / area_j if area_j > 0 else 0.0

            # Larger box must be at least as good in score.
            score_ok = (score_i >= score_j + min_score_advantage)

            # If both checks pass, remove j as redundant.
            if frac_small_inside_big >= containment_threshold and score_ok:
                keep_mask[j] = False
                removal_reason[j] = f"contained_in_det_{int(df.loc[i, 'orig_det_idx'])}"
                break

    # Split outputs into kept vs removed tables.
    kept_df = df[keep_mask].copy().reset_index(drop=True)
    removed_df = df[~keep_mask].copy().reset_index(drop=True)

    if len(removed_df) > 0:
        removed_df["removal_reason"] = [
            removal_reason[idx] for idx in np.where(~keep_mask)[0]
        ]

    return kept_df, removed_df

# -----------------------------------------------------------------------------
# 6c. Helper functions for main-cluster filtering
# -----------------------------------------------------------------------------
# Keeps only detections near the main crossarm group.
def compute_centers_and_scale(df):
    """
    Compute box centers and a characteristic size scale.

    Args:
        df:
            DataFrame with x1, y1, x2, y2.

    Returns:
        tuple:
            df_with_centers, median_diag
    """
    out = df.copy()

    # Computes center coordinates plus a diagonal length per box.
    out["cx"] = (out["x1"] + out["x2"]) / 2.0
    out["cy"] = (out["y1"] + out["y2"]) / 2.0
    out["w"] = (out["x2"] - out["x1"]).clip(lower=0.0)
    out["h"] = (out["y2"] - out["y1"]).clip(lower=0.0)
    out["diag"] = np.sqrt(out["w"] ** 2 + out["h"] ** 2)

    # Uses the median diagonal as a stable scale estimate.
    median_diag = float(np.median(out["diag"])) if len(out) > 0 else 0.0
    median_diag = max(median_diag, 1.0)

    return out, median_diag


def connected_components_from_center_distance(df, center_dist_factor=2.75):
    """
    Build connected components based on box-center distance.

    Two detections are connected if their center distance is less than:
        center_dist_factor * median_box_diagonal

    Args:
        df:
            DataFrame that already contains cx, cy, diag.
        center_dist_factor:
            Distance multiplier.

    Returns:
        tuple:
            components, center_dist_threshold
    """
    if len(df) == 0:
        return [], 0.0

    # Builds a center-distance threshold tied to typical box size.
    median_diag = float(np.median(df["diag"])) if len(df) > 0 else 1.0
    median_diag = max(median_diag, 1.0)
    center_dist_threshold = center_dist_factor * median_diag

    n = len(df)
    visited = np.zeros(n, dtype=bool)
    components = []

    # Extracts center coordinates for pairwise distance calculations.
    centers = df[["cx", "cy"]].to_numpy(dtype=float)

    # adjacency[i] stores detections close enough to i to be considered linked.
    adjacency = [[] for _ in range(n)]

    # Builds the adjacency graph using pairwise center distance.
    for i in range(n):
        for j in range(i + 1, n):
            dist_ij = np.linalg.norm(centers[i] - centers[j])

            if dist_ij <= center_dist_threshold:
                adjacency[i].append(j)
                adjacency[j].append(i)

    # Finds connected components using a DFS-style traversal.
    for start in range(n):
        if visited[start]:
            continue

        stack = [start]
        comp = []
        visited[start] = True

        while stack:
            node = stack.pop()
            comp.append(node)

            for nbr in adjacency[node]:
                if not visited[nbr]:
                    visited[nbr] = True
                    stack.append(nbr)

        components.append(sorted(comp))

    return components, center_dist_threshold


def keep_main_detection_cluster(detections_df, center_dist_factor=2.75):
    """
    Keep only detections belonging to the strongest connected cluster.

    CLUSTER SELECTION RULE:
    - prefer larger component size
    - break ties using higher total score

    Args:
        detections_df:
            DataFrame with orig_det_idx, score, x1, y1, x2, y2
        center_dist_factor:
            Center-distance multiplier.

    Returns:
        tuple:
            kept_df, removed_df, cluster_threshold
    """
    if detections_df.empty:
        return detections_df.copy(), detections_df.iloc[0:0].copy(), 0.0

    # Single detection case: nothing to cluster.
    if len(detections_df) == 1:
        df1, median_diag = compute_centers_and_scale(detections_df)
        return df1.reset_index(drop=True), df1.iloc[0:0].copy(), center_dist_factor * median_diag

    # Computes centers/scales and then builds connected components.
    df, median_diag = compute_centers_and_scale(detections_df)
    components, cluster_threshold = connected_components_from_center_distance(
        df,
        center_dist_factor=center_dist_factor,
    )

    best_component = None
    best_key = None

    # Pick the strongest component:
    # - larger component size first
    # - higher total score second
    for comp in components:
        comp_df = df.iloc[comp]
        comp_size = len(comp)
        comp_score_sum = float(comp_df["score"].sum())

        key = (comp_size, comp_score_sum)

        if best_key is None or key > best_key:
            best_key = key
            best_component = comp

    # Keep only the selected strongest component.
    keep_idx = set(best_component)
    kept_df = df.iloc[sorted(keep_idx)].copy().reset_index(drop=True)
    removed_df = df.drop(index=sorted(keep_idx)).copy().reset_index(drop=True)

    if len(removed_df) > 0:
        removed_df["removal_reason"] = "outside_main_cluster"

    return kept_df, removed_df, cluster_threshold

# -----------------------------------------------------------------------------
# 7. Run SAM3 for this one image + one prompt
# -----------------------------------------------------------------------------
# Builds model inputs from the image + prompt text.
inputs = processor(
    images=image,
    text=PROMPT_TEXT,
    return_tensors="pt",
).to(RUN_DEVICE)

# Uses original_sizes so SAM3 post-processing maps back to image pixel space.
target_sizes = inputs["original_sizes"].detach().cpu().tolist()

# Runs the model in inference mode only.
with torch.no_grad():
    outputs = model(**inputs)

# -----------------------------------------------------------------------------
# 8. Post-process the raw SAM3 outputs into masks / boxes / scores
# -----------------------------------------------------------------------------
# Converts raw model outputs into usable instance segmentation results.
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=TEXT_THRESHOLD,
    mask_threshold=MASK_THRESHOLD,
    target_sizes=target_sizes,
)

# Because this cell only runs one image, take the first result entry.
result = results[0] if isinstance(results, list) and len(results) > 0 else {}

# Normalizes masks / boxes / scores into stable NumPy arrays.
masks = normalize_masks(result.get("masks", None))
num_masks = int(masks.shape[0]) if masks.ndim == 3 else 0

boxes = normalize_boxes(result.get("boxes", None), num_masks)
scores = normalize_scores(result.get("scores", None), num_masks)

# -----------------------------------------------------------------------------
# 9. Build detections table and apply containment suppression
# -----------------------------------------------------------------------------
print("\nDetection summary:")
print(f"  raw_num_masks returned : {num_masks}")

if num_masks == 0:
    print("  No detections were returned for this prompt.")

    raw_detections_df = pd.DataFrame(
        columns=["orig_det_idx", "score", "x1", "y1", "x2", "y2"]
    )
    kept_after_containment_df = raw_detections_df.copy()
    removed_by_containment_df = raw_detections_df.copy()

else:
    # Constructs the raw per-detection table directly from SAM3 output.
    raw_detections_df = pd.DataFrame({
        "orig_det_idx": np.arange(num_masks, dtype=int),
        "score": scores.astype(float),
        "x1": boxes[:, 0].astype(float),
        "y1": boxes[:, 1].astype(float),
        "x2": boxes[:, 2].astype(float),
        "y2": boxes[:, 3].astype(float),
    })

    # Sorts detections by score descending for easier reading.
    raw_detections_df = raw_detections_df.sort_values(
        "score",
        ascending=False
    ).reset_index(drop=True)

    print("\nRaw detections table:")
    try:
        display(raw_detections_df)
    except NameError:
        print(raw_detections_df)

    # First cleanup pass:
    # removes smaller detections mostly contained in bigger ones.
    kept_after_containment_df, removed_by_containment_df = suppress_contained_shorter_detections(
        detections_df=raw_detections_df,
        containment_threshold=CONTAINMENT_THRESHOLD,
        min_area_ratio=MIN_AREA_RATIO,
        min_score_advantage=MIN_SCORE_ADVANTAGE,
    )

    print(f"\nKept after containment suppression   : {len(kept_after_containment_df)}")
    print(f"Removed by containment suppression   : {len(removed_by_containment_df)}")

    print("\nKept after containment suppression:")
    try:
        display(kept_after_containment_df)
    except NameError:
        print(kept_after_containment_df)

    if len(removed_by_containment_df) > 0:
        print("\nRemoved by containment suppression:")
        try:
            display(removed_by_containment_df)
        except NameError:
            print(removed_by_containment_df)

# -----------------------------------------------------------------------------
# 9b. Second pass: remove detections far away from the main crossarm cluster
# -----------------------------------------------------------------------------
# If nothing survived pass 1, keep outputs empty.
if len(kept_after_containment_df) == 0:
    final_kept_detections_df = kept_after_containment_df.copy()
    removed_by_cluster_df = kept_after_containment_df.copy()
    cluster_threshold_used = 0.0
else:
    # Otherwise, keep only detections in the strongest spatial cluster.
    final_kept_detections_df, removed_by_cluster_df, cluster_threshold_used = keep_main_detection_cluster(
        detections_df=kept_after_containment_df,
        center_dist_factor=CENTER_DIST_FACTOR,
    )

print(f"\nCluster distance threshold used : {cluster_threshold_used:.2f}")
print(f"Final kept detections           : {len(final_kept_detections_df)}")
print(f"Removed as isolated false-pos   : {len(removed_by_cluster_df)}")

print("\nFinal kept detections table:")
try:
    display(final_kept_detections_df)
except NameError:
    print(final_kept_detections_df)

if len(removed_by_cluster_df) > 0:
    print("\nRemoved by main-cluster filter:")
    try:
        display(removed_by_cluster_df)
    except NameError:
        print(removed_by_cluster_df)

# -----------------------------------------------------------------------------
# 9c. Assign ordered crossarm labels to FINAL kept detections
# -----------------------------------------------------------------------------
# Labels detections from top to bottom using the box vertical center.
if len(final_kept_detections_df) > 0:
    final_kept_detections_df = final_kept_detections_df.copy()

    # Creates center coordinates if they do not already exist.
    if "cx" not in final_kept_detections_df.columns:
        final_kept_detections_df["cx"] = (
            final_kept_detections_df["x1"] + final_kept_detections_df["x2"]
        ) / 2.0

    if "cy" not in final_kept_detections_df.columns:
        final_kept_detections_df["cy"] = (
            final_kept_detections_df["y1"] + final_kept_detections_df["y2"]
        ) / 2.0

    # Sort top-to-bottom, then left-to-right as a tie-breaker.
    final_kept_detections_df = final_kept_detections_df.sort_values(
        ["cy", "cx"],
        ascending=[True, True]
    ).reset_index(drop=True)

    # Assigns human-readable display labels.
    final_kept_detections_df["xarm_label"] = [
        f"Xarm_{i+1}" for i in range(len(final_kept_detections_df))
    ]

    print("\nFinal kept detections with Xarm labels:")
    try:
        display(
            final_kept_detections_df[
                ["xarm_label", "orig_det_idx", "score", "x1", "y1", "x2", "y2", "cx", "cy"]
            ]
        )
    except NameError:
        print(
            final_kept_detections_df[
                ["xarm_label", "orig_det_idx", "score", "x1", "y1", "x2", "y2", "cx", "cy"]
            ]
        )

# -----------------------------------------------------------------------------
# 10. Visualize only the FINAL kept detections on the image
# -----------------------------------------------------------------------------
# Shows:
# - the base image
# - every final kept mask in the same purple colour
# - yellow SAM3 boxes
# - purple Xarm labels
# - no mask outline / no border
plt.figure(figsize=(12, 9))
ax = plt.gca()

# Show the base image first so the masks and boxes overlay on top.
ax.imshow(image)

num_final_kept = len(final_kept_detections_df)

if num_final_kept > 0:
    # -------------------------------------------------------------------------
    # Uniform purple display settings
    # -------------------------------------------------------------------------
    # All masks use the SAME purple colour.
    purple_r = 0.50
    purple_g = 0.00
    purple_b = 0.50
    mask_alpha = 0.40

    # Plot final kept detections in the SAME order as final_kept_detections_df.
    for _, det_row in final_kept_detections_df.iterrows():
        orig_idx = int(det_row["orig_det_idx"])

        # Pulls the original mask/box/score for this kept detection.
        mask_i = masks[orig_idx]
        box_i = boxes[orig_idx]
        score_i = float(det_row["score"])
        xarm_label = det_row["xarm_label"]

        # -------------------------------------------------------------
        # Uniform purple mask fill
        # -------------------------------------------------------------
        # Creates an RGBA overlay where RGB = purple and alpha controls transparency.
        overlay = np.zeros((mask_i.shape[0], mask_i.shape[1], 4), dtype=np.float32)
        overlay[..., 0] = purple_r
        overlay[..., 1] = purple_g
        overlay[..., 2] = purple_b

        # Only masked pixels receive alpha; background stays transparent.
        overlay[..., 3] = mask_i.astype(np.float32) * mask_alpha
        ax.imshow(overlay)

        # -------------------------------------------------------------
        # Predicted box (yellow)
        # -------------------------------------------------------------
        # Draws the raw SAM3 box in yellow for visibility.
        x1, y1, x2, y2 = [float(v) for v in box_i]

        rect = patches.Rectangle(
            (x1, y1),
            max(0.0, x2 - x1),
            max(0.0, y2 - y1),
            linewidth=2.0,
            edgecolor="yellow",
            facecolor="none",
        )
        ax.add_patch(rect)

        # -------------------------------------------------------------
        # Detection label (fixed purple)
        # -------------------------------------------------------------
        # Uses one fixed purple label background for all Xarm labels.
        label_bg = "#800080"

        ax.text(
            x1,
            max(5, y1 - 5),
            f"{xarm_label}: {score_i:.3f}",
            color="white",
            fontsize=9,
            bbox=dict(facecolor=label_bg, alpha=0.95, pad=1.5, edgecolor="none"),
        )

# Sets plot title and removes axes for cleaner inspection.
ax.set_title(
    f"{file_name}\n"
    f"Prompt: {PROMPT_TEXT} | Raw detections: {num_masks} | Final kept: {num_final_kept}",
    fontsize=12,
    pad=12,
)
ax.axis("off")
plt.show()
plt.close()

# -----------------------------------------------------------------------------
# 11. Final note
# -----------------------------------------------------------------------------
print("\nCell 6c completed successfully.")
print("This shows ONE prompt on ONE image with:")
print("- containment-based suppression")
print("- main-cluster false-positive filtering")
print("- same purple colour for all final kept masks")
print("- yellow SAM3 boxes")
print("- purple Xarm_1 / Xarm_2 / ... labels")
print("- no mask border / no outline")

# -----------------------------------------------------------------------------
# 12. Optional cleanup
# -----------------------------------------------------------------------------
# Removes the large image object from memory now that the cell is done.
del image

In [ ]:
# =============================================================================
# CELL 6d — RUN ONE PROMPT ON ALL DEBUG IMAGES
#            + CONTAINMENT SUPPRESSION
#            + MAIN-CLUSTER FALSE-POSITIVE FILTER
#            + UNIFORM PURPLE MASK DISPLAY
#            + Xarm_1 / Xarm_2 / ... LABELS
#            + 3-COLUMN PRESENTATION GRID
# =============================================================================
# EXPLANATION:
# This cell applies the SAME single-prompt crossarm workflow from Cell 6c to
# ALL images in debug_images_df.
#
# WHAT IT DOES:
# - loops through every image in debug_images_df
# - runs SAM3 with the single prompt:
#       "utility pole crossarm"
# - applies the SAME two post-processing stages from Cell 6c:
#       1) containment suppression
#       2) main-cluster false-positive filtering
# - assigns Xarm_1 / Xarm_2 / ... labels per image
# - displays all debug images in one grid:
#       - 3 images per row
#       - fixed panel size
#       - presentation-friendly overall size
#
# VISUAL STYLE:
# - all final kept masks = same purple
# - light transparency
# - no border / no outline
# - SAM3 predicted boxes = yellow
# - Xarm labels = purple label box with white text
#
# IMPORTANT:
# - this cell does NOT save masks/crops
# - this cell is still a debug / QA visualization step
# - this cell REUSES helper functions already defined in Cell 6c
# =============================================================================

# -----------------------------------------------------------------------------
# Imports
# -----------------------------------------------------------------------------
# os        : file/path checks
# math      : grid row calculation
# numpy     : numeric helpers
# pandas    : summary tables
# matplotlib: grid plotting
# torch     : device fallback
# PIL       : image loading / RGB conversion
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
from PIL import Image

# -----------------------------------------------------------------------------
# 0. Safety checks
# -----------------------------------------------------------------------------
# Confirms the required objects from earlier cells already exist.
required_globals = [
    "debug_images_df",
    "model",
    "processor",
    "normalize_masks",
    "normalize_boxes",
    "normalize_scores",
    "suppress_contained_shorter_detections",
    "keep_main_detection_cluster",
]

missing_globals = [name for name in required_globals if name not in globals()]

if missing_globals:
    raise NameError(
        "Cell 6d requires objects/functions from earlier cells.\n"
        "Please run Cell 5b and the current Cell 6c first.\n"
        f"Missing globals: {missing_globals}"
    )

if not isinstance(debug_images_df, pd.DataFrame):
    raise TypeError("debug_images_df exists but is not a pandas DataFrame.")

if debug_images_df.empty:
    raise ValueError("debug_images_df is empty.")

# -----------------------------------------------------------------------------
# 1. Prompt, thresholds, and post-processing settings
# -----------------------------------------------------------------------------
# Uses the SAME single prompt and cleanup settings as Cell 6c.
PROMPT_TEXT = "utility pole crossarm"

TEXT_THRESHOLD = float(globals().get("TEXT_SCORE_THRESHOLD", 0.30))
MASK_THRESHOLD = float(globals().get("MASK_THRESHOLD", 0.50))

RUN_DEVICE = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")

# Containment suppression settings
CONTAINMENT_THRESHOLD = 0.80
MIN_AREA_RATIO = 1.20
MIN_SCORE_ADVANTAGE = 0.0

# Main-cluster filtering settings
CENTER_DIST_FACTOR = 2.75

# -----------------------------------------------------------------------------
# 2. Grid display settings
# -----------------------------------------------------------------------------
# Uses a fixed, presentation-friendly panel size and a 3-column layout.
N_COLS = 3
PANEL_W = 4.8
PANEL_H = 3.6
GRID_DPI = 120

# Same purple display style as current Cell 6c
PURPLE_R = 0.50
PURPLE_G = 0.00
PURPLE_B = 0.50
MASK_ALPHA = 0.40
LABEL_BG = "#800080"

# -----------------------------------------------------------------------------
# 3. Choose which images to show
# -----------------------------------------------------------------------------
# This cell uses the current debug subset directly.
grid_df = debug_images_df.copy().reset_index(drop=True)

n_images = len(grid_df)
n_rows = math.ceil(n_images / N_COLS)

print("Batch debug run over debug_images_df:\n")
print(f"  n_images              : {n_images}")
print(f"  prompt                : {PROMPT_TEXT}")
print(f"  TEXT_THRESHOLD        : {TEXT_THRESHOLD}")
print(f"  MASK_THRESHOLD        : {MASK_THRESHOLD}")
print(f"  RUN_DEVICE            : {RUN_DEVICE}")
print(f"  CONTAINMENT_THRESHOLD : {CONTAINMENT_THRESHOLD}")
print(f"  MIN_AREA_RATIO        : {MIN_AREA_RATIO}")
print(f"  MIN_SCORE_ADVANTAGE   : {MIN_SCORE_ADVANTAGE}")
print(f"  CENTER_DIST_FACTOR    : {CENTER_DIST_FACTOR}")
print(f"  grid_cols             : {N_COLS}")
print(f"  grid_rows             : {n_rows}")
print(f"  panel_size            : {PANEL_W} x {PANEL_H}")

# -----------------------------------------------------------------------------
# 4. Create the figure grid
# -----------------------------------------------------------------------------
# Creates one subplot per image and later turns off any unused panels.
fig, axes = plt.subplots(
    n_rows,
    N_COLS,
    figsize=(N_COLS * PANEL_W, n_rows * PANEL_H),
    dpi=GRID_DPI,
    constrained_layout=True,
)

# Normalizes axes into a flat 1D array so iteration is simple.
axes = np.array(axes).reshape(-1)

# -----------------------------------------------------------------------------
# 5. Loop over every debug image
# -----------------------------------------------------------------------------
# Each image is processed independently so one failure does not stop the rest.
batch_rows = []

for plot_idx, (_, row) in enumerate(grid_df.iterrows()):
    ax = axes[plot_idx]

    # -------------------------------------------------------------------------
    # 5a. Extract metadata for the current image
    # -------------------------------------------------------------------------
    image_path = row["image_path"]

    image_id = row.get("image_id", None)
    if pd.isna(image_id):
        image_id = None

    file_name = row.get("file_name", None)
    if pd.isna(file_name) or not isinstance(file_name, str) or len(file_name.strip()) == 0:
        file_name = os.path.basename(image_path)

    try:
        # ---------------------------------------------------------------------
        # 5b. Validate image path
        # ---------------------------------------------------------------------
        if not isinstance(image_path, str) or len(image_path.strip()) == 0:
            raise ValueError(f"Invalid image_path: {image_path}")

        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image file not found: {image_path}")

        # ---------------------------------------------------------------------
        # 5c. Load the image as RGB
        # ---------------------------------------------------------------------
        # Converts to RGB if required and fully loads pixels into memory.
        with Image.open(image_path) as img:
            original_mode = img.mode

            if original_mode != "RGB":
                image = img.convert("RGB")
            else:
                image = img.copy()

            image.load()

        image_w, image_h = image.size

        # ---------------------------------------------------------------------
        # 5d. Build SAM3 inputs and run inference
        # ---------------------------------------------------------------------
        inputs = processor(
            images=image,
            text=PROMPT_TEXT,
            return_tensors="pt",
        ).to(RUN_DEVICE)

        # Uses processor-reported original sizes for coordinate-space mapping.
        target_sizes = inputs["original_sizes"].detach().cpu().tolist()

        with torch.no_grad():
            outputs = model(**inputs)

        # ---------------------------------------------------------------------
        # 5e. Post-process raw SAM3 outputs
        # ---------------------------------------------------------------------
        results = processor.post_process_instance_segmentation(
            outputs,
            threshold=TEXT_THRESHOLD,
            mask_threshold=MASK_THRESHOLD,
            target_sizes=target_sizes,
        )

        result = results[0] if isinstance(results, list) and len(results) > 0 else {}

        masks = normalize_masks(result.get("masks", None))
        num_masks = int(masks.shape[0]) if masks.ndim == 3 else 0

        boxes = normalize_boxes(result.get("boxes", None), num_masks)
        scores = normalize_scores(result.get("scores", None), num_masks)

        # ---------------------------------------------------------------------
        # 5f. Build raw detections table
        # ---------------------------------------------------------------------
        if num_masks == 0:
            raw_detections_df = pd.DataFrame(
                columns=["orig_det_idx", "score", "x1", "y1", "x2", "y2"]
            )
            kept_after_containment_df = raw_detections_df.copy()
            removed_by_containment_df = raw_detections_df.copy()
        else:
            raw_detections_df = pd.DataFrame({
                "orig_det_idx": np.arange(num_masks, dtype=int),
                "score": scores.astype(float),
                "x1": boxes[:, 0].astype(float),
                "y1": boxes[:, 1].astype(float),
                "x2": boxes[:, 2].astype(float),
                "y2": boxes[:, 3].astype(float),
            })

            # Sorts raw detections by score descending for stable reading.
            raw_detections_df = raw_detections_df.sort_values(
                "score",
                ascending=False
            ).reset_index(drop=True)

            # -------------------------------------------------------------
            # First pass: containment suppression
            # -------------------------------------------------------------
            kept_after_containment_df, removed_by_containment_df = suppress_contained_shorter_detections(
                detections_df=raw_detections_df,
                containment_threshold=CONTAINMENT_THRESHOLD,
                min_area_ratio=MIN_AREA_RATIO,
                min_score_advantage=MIN_SCORE_ADVANTAGE,
            )

        # ---------------------------------------------------------------------
        # 5g. Second pass: keep only the main cluster
        # ---------------------------------------------------------------------
        if len(kept_after_containment_df) == 0:
            final_kept_detections_df = kept_after_containment_df.copy()
            removed_by_cluster_df = kept_after_containment_df.copy()
            cluster_threshold_used = 0.0
        else:
            final_kept_detections_df, removed_by_cluster_df, cluster_threshold_used = keep_main_detection_cluster(
                detections_df=kept_after_containment_df,
                center_dist_factor=CENTER_DIST_FACTOR,
            )

        # ---------------------------------------------------------------------
        # 5h. Assign Xarm_1 / Xarm_2 / ... labels
        # ---------------------------------------------------------------------
        # Orders final detections top-to-bottom using box center coordinates.
        if len(final_kept_detections_df) > 0:
            final_kept_detections_df = final_kept_detections_df.copy()

            if "cx" not in final_kept_detections_df.columns:
                final_kept_detections_df["cx"] = (
                    final_kept_detections_df["x1"] + final_kept_detections_df["x2"]
                ) / 2.0

            if "cy" not in final_kept_detections_df.columns:
                final_kept_detections_df["cy"] = (
                    final_kept_detections_df["y1"] + final_kept_detections_df["y2"]
                ) / 2.0

            final_kept_detections_df = final_kept_detections_df.sort_values(
                ["cy", "cx"],
                ascending=[True, True]
            ).reset_index(drop=True)

            final_kept_detections_df["xarm_label"] = [
                f"Xarm_{i+1}" for i in range(len(final_kept_detections_df))
            ]

        # ---------------------------------------------------------------------
        # 5i. Plot the current image on its subplot
        # ---------------------------------------------------------------------
        # Draws base image first, then purple masks, yellow boxes, and labels.
        ax.imshow(image)

        num_final_kept = len(final_kept_detections_df)

        if num_final_kept > 0:
            for _, det_row in final_kept_detections_df.iterrows():
                orig_idx = int(det_row["orig_det_idx"])

                mask_i = masks[orig_idx]
                box_i = boxes[orig_idx]
                score_i = float(det_row["score"])
                xarm_label = det_row["xarm_label"]

                # ---------------------------------------------------------
                # Uniform purple mask fill
                # ---------------------------------------------------------
                overlay = np.zeros((mask_i.shape[0], mask_i.shape[1], 4), dtype=np.float32)
                overlay[..., 0] = PURPLE_R
                overlay[..., 1] = PURPLE_G
                overlay[..., 2] = PURPLE_B

                # Only masked pixels receive alpha; background stays transparent.
                overlay[..., 3] = mask_i.astype(np.float32) * MASK_ALPHA
                ax.imshow(overlay)

                # ---------------------------------------------------------
                # SAM3 predicted box (yellow)
                # ---------------------------------------------------------
                x1, y1, x2, y2 = [float(v) for v in box_i]

                rect = patches.Rectangle(
                    (x1, y1),
                    max(0.0, x2 - x1),
                    max(0.0, y2 - y1),
                    linewidth=1.8,
                    edgecolor="yellow",
                    facecolor="none",
                )
                ax.add_patch(rect)

                # ---------------------------------------------------------
                # Xarm label (purple label box)
                # ---------------------------------------------------------
                ax.text(
                    x1,
                    max(5, y1 - 5),
                    f"{xarm_label}: {score_i:.3f}",
                    color="white",
                    fontsize=8,
                    bbox=dict(facecolor=LABEL_BG, alpha=0.95, pad=1.2, edgecolor="none"),
                )

        # ---------------------------------------------------------------------
        # 5j. Set subplot title
        # ---------------------------------------------------------------------
        # Keeps titles short enough for a multi-image grid.
        ax.set_title(
            f"{file_name}\nRaw={num_masks} | Kept={num_final_kept}",
            fontsize=9,
            pad=8,
        )
        ax.axis("off")

        # ---------------------------------------------------------------------
        # 5k. Save summary row for this image
        # ---------------------------------------------------------------------
        batch_rows.append({
            "debug_row_index": plot_idx,
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": original_mode,
            "width": image_w,
            "height": image_h,
            "raw_num_masks": int(num_masks),
            "kept_after_containment": int(len(kept_after_containment_df)),
            "removed_by_containment": int(len(removed_by_containment_df)),
            "final_kept": int(len(final_kept_detections_df)),
            "removed_by_cluster": int(len(removed_by_cluster_df)),
            "cluster_threshold_used": float(cluster_threshold_used),
            "xarm_labels": ", ".join(final_kept_detections_df["xarm_label"].tolist()) if len(final_kept_detections_df) > 0 else "",
            "run_status": "ok",
            "error_message": "",
        })

        # Frees the per-image object before moving to the next image.
        del image

    except Exception as e:
        # ---------------------------------------------------------------------
        # 5l. Graceful error display for the current image
        # ---------------------------------------------------------------------
        # Shows an error message inside the subplot instead of breaking the grid.
        ax.set_facecolor("white")
        ax.text(
            0.5, 0.5,
            f"FAILED\n{file_name}\n\n{type(e).__name__}\n{str(e)}",
            ha="center",
            va="center",
            fontsize=9,
            wrap=True,
            transform=ax.transAxes,
        )
        ax.set_title(f"{file_name}\nFAILED", fontsize=9, pad=8)
        ax.axis("off")

        batch_rows.append({
            "debug_row_index": plot_idx,
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "original_mode": None,
            "width": np.nan,
            "height": np.nan,
            "raw_num_masks": np.nan,
            "kept_after_containment": np.nan,
            "removed_by_containment": np.nan,
            "final_kept": np.nan,
            "removed_by_cluster": np.nan,
            "cluster_threshold_used": np.nan,
            "xarm_labels": "",
            "run_status": "failed",
            "error_message": str(e),
        })

# -----------------------------------------------------------------------------
# 6. Turn off any unused axes
# -----------------------------------------------------------------------------
# Hides extra empty subplot panels when image count is not a multiple of 3.
for ax in axes[n_images:]:
    ax.axis("off")

# -----------------------------------------------------------------------------
# 7. Global figure title
# -----------------------------------------------------------------------------
# Adds one overall title for the whole presentation grid.
fig.suptitle(
    f"Debug Grid — Prompt: {PROMPT_TEXT}",
    fontsize=14,
)

plt.show()
plt.close()

# -----------------------------------------------------------------------------
# 8. Build and display batch summary table
# -----------------------------------------------------------------------------
# Creates a compact per-image summary table for the whole debug batch.
debug_batch_results_df = pd.DataFrame(batch_rows)

print("\nBatch summary table:")
try:
    display(debug_batch_results_df)
except NameError:
    print(debug_batch_results_df)

# -----------------------------------------------------------------------------
# 9. Final note
# -----------------------------------------------------------------------------
print("\nCell 6d completed successfully.")
print("This shows ALL images in debug_images_df using:")
print("- one prompt only")
print("- containment-based suppression")
print("- main-cluster false-positive filtering")
print("- same purple colour for all final kept masks")
print("- yellow SAM3 boxes")
print("- purple Xarm_1 / Xarm_2 / ... labels")
print("- 3-column presentation grid")

In [ ]:
# =============================================================================
# CELL 9a — SAM3 TEXT MASKS FOR ALL UPLOADED IMAGES
# =============================================================================
# EXPLANATION:
# This cell runs HF SAM3 text-prompt inference on ALL images discovered in
# images_df from the uploaded ZIP.
#
# THIS VERSION IS FOR THE NEW CROSSARM PIPELINE:
# - no assets_df
# - no scene_split_df
# - no GT annotation boxes
# - no IoU calculations against labels
#
# WHAT THIS CELL DOES:
# 1) loops through all uploaded images
# 2) tries one or more crossarm text prompts per image
# 3) keeps the highest-scoring SAM3 candidate across all prompts
# 4) saves the chosen mask to disk
# 5) saves the padded crop image to disk
# 6) stores the SAM3 predicted box ("pred_box")
# 7) shows a small visual QA gallery
#
# IMPORTANT:
# - "annotation_box" is kept as None for compatibility only
# - the usable box in this pipeline is the SAM3 predicted box: pred_box
#
# OUTPUTS:
# - sam3_crossarm_mask_df   : one row per uploaded image, final best result
# - sam3_crossarm_prompt_df : one row per image x prompt summary
# =============================================================================

# -----------------------------------------------------------------------------
# 1. Imports
# -----------------------------------------------------------------------------
import os
import gc
import re
import json
import time
import math
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image

# -----------------------------------------------------------------------------
# 2. Configuration
# -----------------------------------------------------------------------------
# Prompt list:
# - if Cell 2 defined DEFAULT_CROSSARM_PROMPTS, use that
# - otherwise fall back to a sensible starter list
PROMPTS_TO_TRY = globals().get(
    "DEFAULT_CROSSARM_PROMPTS",
    [
        "crossarm",
        "utility pole crossarm",
        "electricity pole crossarm",
        "power pole crossarm",
        "distribution pole crossarm",
        "wooden crossarm",
    ]
)

if isinstance(PROMPTS_TO_TRY, str):
    PROMPTS_TO_TRY = [PROMPTS_TO_TRY]

# SAM3 thresholds
TEXT_THRESHOLD = float(globals().get("TEXT_SCORE_THRESHOLD", 0.25))
MASK_THRESHOLD = float(globals().get("MASK_THRESHOLD", 0.25))

# Fixed crop padding around the chosen SAM3 predicted box
PAD_PIXELS = 25

# Progress logging
PRINT_EVERY_N = 10

# Visual QA
MAX_VISUAL_QA = 12
MASK_ALPHA = 0.35
SHOW_PRED_BOX = True
SHOW_CROP_BOX = True

# Run-specific output folders
RUN_STAMP_LOCAL = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M%S")
SAM3_CROSSARM_RUN_NAME = f"sam3_crossarm_run_{RUN_STAMP_LOCAL}"

SAM3_CROSSARM_RUN_DIR = os.path.join(SILVER_PROMPT_RUNS, SAM3_CROSSARM_RUN_NAME)
SAM3_CROSSARM_MASK_DIR = os.path.join(SILVER_CROSSARM_MASKS, SAM3_CROSSARM_RUN_NAME)
SAM3_CROSSARM_CROP_DIR = os.path.join(SILVER_CROSSARM_CROPS, SAM3_CROSSARM_RUN_NAME)
SAM3_CROSSARM_DEBUG_DIR = os.path.join(SILVER_DEBUG, SAM3_CROSSARM_RUN_NAME)

os.makedirs(SAM3_CROSSARM_RUN_DIR, exist_ok=True)
os.makedirs(SAM3_CROSSARM_MASK_DIR, exist_ok=True)
os.makedirs(SAM3_CROSSARM_CROP_DIR, exist_ok=True)
os.makedirs(SAM3_CROSSARM_DEBUG_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# 3. Safety checks
# -----------------------------------------------------------------------------
assert "images_df" in globals(), "images_df not found. Run Cell 2b first."
assert "model" in globals(), "model not found. Run Cell 5b first."
assert "processor" in globals(), "processor not found. Run Cell 5b first."

required_img_cols = {"image_path"}
missing_img_cols = required_img_cols - set(images_df.columns)
assert not missing_img_cols, f"images_df missing columns: {missing_img_cols}"

# Use the globally prepared device if available
RUN_DEVICE = globals().get("DEVICE", next(model.parameters()).device)

# -----------------------------------------------------------------------------
# 4. Helpers
# -----------------------------------------------------------------------------
def load_cached_image(image_path):
    """
    Load one image from disk as RGB.

    Args:
        image_path:
            Absolute path to image.

    Returns:
        PIL.Image.Image:
            RGB image.
    """
    return Image.open(image_path).convert("RGB")


def sanitize_for_filename(text):
    """
    Convert text into a filesystem-safe fragment.

    Args:
        text:
            Any identifier or filename fragment.

    Returns:
        str:
            Safe filename fragment.
    """
    text = str(text).strip()
    text = re.sub(r"[^\w\-.]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def _to_numpy(x):
    """
    Convert tensor / list / scalar into numpy.

    Args:
        x:
            Tensor-like object.

    Returns:
        np.ndarray
    """
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def _normalize_masks(masks):
    """
    Normalize SAM3 masks into shape (N, H, W), bool.

    Args:
        masks:
            Raw SAM3 masks.

    Returns:
        np.ndarray
    """
    if masks is None:
        return np.zeros((0, 0, 0), dtype=bool)

    masks_np = _to_numpy(masks)

    if masks_np.size == 0:
        return np.zeros((0, 0, 0), dtype=bool)

    if masks_np.ndim == 2:
        masks_np = masks_np[None, ...]

    return masks_np.astype(bool)


def _normalize_boxes(boxes, num_masks):
    """
    Normalize boxes into shape (N, 4).

    Args:
        boxes:
            Raw SAM3 boxes.
        num_masks:
            Number of masks.

    Returns:
        np.ndarray
    """
    if boxes is None:
        return np.zeros((num_masks, 4), dtype=np.float32)

    boxes_np = _to_numpy(boxes).astype(np.float32)

    if boxes_np.ndim == 1 and boxes_np.shape[0] == 4:
        boxes_np = boxes_np.reshape(1, 4)

    if boxes_np.ndim != 2 or boxes_np.shape[1] != 4:
        return np.zeros((num_masks, 4), dtype=np.float32)

    fixed = np.zeros((num_masks, 4), dtype=np.float32)
    n = min(len(boxes_np), num_masks)
    fixed[:n] = boxes_np[:n]
    return fixed


def _normalize_scores(scores, num_masks):
    """
    Normalize scores into shape (N,).

    Args:
        scores:
            Raw SAM3 scores.
        num_masks:
            Number of masks.

    Returns:
        np.ndarray
    """
    if scores is None:
        return np.zeros((num_masks,), dtype=np.float32)

    scores_np = _to_numpy(scores)

    if scores_np.ndim == 0:
        if num_masks == 1:
            return np.array([float(scores_np)], dtype=np.float32)
        return np.zeros((num_masks,), dtype=np.float32)

    scores_np = scores_np.astype(np.float32).reshape(-1)

    fixed = np.zeros((num_masks,), dtype=np.float32)
    n = min(len(scores_np), num_masks)
    fixed[:n] = scores_np[:n]
    return fixed


def pad_xyxy_box(box_xyxy, image_w, image_h, pad_pixels=25):
    """
    Add fixed pixel padding around xyxy box and clamp to image bounds.

    Args:
        box_xyxy:
            [x1, y1, x2, y2]
        image_w:
            Image width.
        image_h:
            Image height.
        pad_pixels:
            Fixed padding.

    Returns:
        list or None
    """
    if box_xyxy is None:
        return None

    x1, y1, x2, y2 = [float(v) for v in box_xyxy]

    if x2 <= x1 or y2 <= y1:
        return None

    px1 = max(0.0, x1 - float(pad_pixels))
    py1 = max(0.0, y1 - float(pad_pixels))
    px2 = min(float(image_w), x2 + float(pad_pixels))
    py2 = min(float(image_h), y2 + float(pad_pixels))

    return [px1, py1, px2, py2]


def xyxy_to_xywh(box_xyxy):
    """
    Convert xyxy -> xywh.

    Args:
        box_xyxy:
            [x1, y1, x2, y2]

    Returns:
        tuple
    """
    if box_xyxy is None:
        return (np.nan, np.nan, np.nan, np.nan)

    x1, y1, x2, y2 = [float(v) for v in box_xyxy]
    return (x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1))


def save_best_mask_npz(mask_bool, image_id):
    """
    Save chosen best mask to compressed .npz.

    Args:
        mask_bool:
            Boolean mask.
        image_id:
            Image identifier.

    Returns:
        str or None
    """
    if mask_bool is None:
        return None

    safe_image = sanitize_for_filename(image_id)
    out_name = f"{safe_image}__best_mask.npz"
    out_path = os.path.join(SAM3_CROSSARM_MASK_DIR, out_name)

    np.savez_compressed(out_path, best_mask=mask_bool.astype(np.uint8))
    return out_path


def load_best_mask_npz(mask_path):
    """
    Reload saved mask from disk.

    Args:
        mask_path:
            Path to .npz mask file.

    Returns:
        np.ndarray or None
    """
    if mask_path is None or not os.path.exists(mask_path):
        return None

    with np.load(mask_path) as data:
        mask = data["best_mask"]

    return mask.astype(bool)


def save_crop_png(image_pil, crop_box, image_id):
    """
    Save padded crop as PNG.

    Args:
        image_pil:
            PIL RGB image.
        crop_box:
            [x1, y1, x2, y2]
        image_id:
            Image identifier.

    Returns:
        tuple:
            (crop_path, crop_h, crop_w)
    """
    if crop_box is None:
        return (None, np.nan, np.nan)

    x1, y1, x2, y2 = crop_box
    x1 = int(max(0, math.floor(x1)))
    y1 = int(max(0, math.floor(y1)))
    x2 = int(max(0, math.ceil(x2)))
    y2 = int(max(0, math.ceil(y2)))

    image_np = np.array(image_pil)
    crop_np = image_np[y1:y2, x1:x2]

    if crop_np.size == 0:
        return (None, np.nan, np.nan)

    safe_image = sanitize_for_filename(image_id)
    crop_path = os.path.join(SAM3_CROSSARM_CROP_DIR, f"{safe_image}__crop.png")
    Image.fromarray(crop_np).save(crop_path)

    return (crop_path, crop_np.shape[0], crop_np.shape[1])


def overlay_mask(ax, image_pil, mask_bool, alpha=0.35):
    """
    Draw image and green mask overlay.

    Args:
        ax:
            Matplotlib axis.
        image_pil:
            PIL RGB image.
        mask_bool:
            Boolean mask.
        alpha:
            Overlay transparency.
    """
    image_np = np.array(image_pil)
    ax.imshow(image_np)

    if mask_bool is not None and mask_bool.size > 0:
        overlay = np.zeros((mask_bool.shape[0], mask_bool.shape[1], 4), dtype=np.float32)
        overlay[..., 1] = 1.0
        overlay[..., 3] = mask_bool.astype(np.float32) * alpha
        ax.imshow(overlay)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis("off")


def draw_xyxy_box(ax, box_xyxy, color="yellow", label=None, linewidth=1.5):
    """
    Draw one xyxy box.

    Args:
        ax:
            Matplotlib axis.
        box_xyxy:
            [x1, y1, x2, y2]
        color:
            Edge color.
        label:
            Optional label.
        linewidth:
            Box thickness.
    """
    if box_xyxy is None:
        return

    x1, y1, x2, y2 = box_xyxy

    rect = patches.Rectangle(
        (x1, y1),
        max(0.0, x2 - x1),
        max(0.0, y2 - y1),
        linewidth=linewidth,
        edgecolor=color,
        facecolor="none",
    )
    ax.add_patch(rect)

    if label:
        ax.text(
            x1,
            max(5, y1 - 5),
            label,
            color="white",
            fontsize=8,
            bbox=dict(facecolor=color, alpha=0.75, pad=1.5, edgecolor="none"),
        )


def run_sam3_text_only(image, prompt_text):
    """
    Run HF SAM3 text-only prompting on one full image.

    Args:
        image:
            PIL RGB image.
        prompt_text:
            Text prompt.

    Returns:
        tuple:
            masks_np  : (N, H, W) bool
            boxes_np  : (N, 4)
            scores_np : (N,)
    """
    inputs = processor(
        images=image,
        text=prompt_text,
        return_tensors="pt",
    ).to(RUN_DEVICE)

    target_sizes = inputs["original_sizes"].detach().cpu().tolist()

    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=TEXT_THRESHOLD,
        mask_threshold=MASK_THRESHOLD,
        target_sizes=target_sizes,
    )

    result = results[0] if isinstance(results, list) and len(results) > 0 else {}

    masks = _normalize_masks(result.get("masks", None))
    num_masks = int(masks.shape[0]) if masks.ndim == 3 else 0

    boxes = _normalize_boxes(result.get("boxes", None), num_masks)
    scores = _normalize_scores(result.get("scores", None), num_masks)

    return masks, boxes, scores


def choose_best_candidate_across_prompts(image, prompts):
    """
    Run SAM3 for each prompt and keep the highest-scoring candidate overall.

    Args:
        image:
            PIL RGB image.
        prompts:
            List of text prompts.

    Returns:
        tuple:
            best_result:
                dict or None
            prompt_rows:
                list[dict]
    """
    best_result = None
    prompt_rows = []

    for prompt_text in prompts:
        masks, boxes, scores = run_sam3_text_only(image, prompt_text)
        num_masks = int(masks.shape[0]) if masks.ndim == 3 else 0

        if num_masks == 0:
            prompt_rows.append({
                "prompt_text": prompt_text,
                "num_masks": 0,
                "best_score_for_prompt": np.nan,
                "best_box_for_prompt": None,
                "best_local_index": np.nan,
                "prompt_status": "no_mask",
            })
            continue

        best_local_index = int(np.argmax(scores))
        best_score_for_prompt = float(scores[best_local_index])
        best_box_for_prompt = boxes[best_local_index].tolist()
        best_mask_for_prompt = masks[best_local_index].copy()

        prompt_rows.append({
            "prompt_text": prompt_text,
            "num_masks": num_masks,
            "best_score_for_prompt": best_score_for_prompt,
            "best_box_for_prompt": best_box_for_prompt,
            "best_local_index": best_local_index,
            "prompt_status": "done",
        })

        candidate = {
            "prompt_text": prompt_text,
            "num_masks": num_masks,
            "best_local_index": best_local_index,
            "best_score": best_score_for_prompt,
            "best_box": best_box_for_prompt,
            "best_mask": best_mask_for_prompt,
        }

        if (best_result is None) or (candidate["best_score"] > best_result["best_score"]):
            best_result = candidate

        del masks
        del boxes
        del scores
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return best_result, prompt_rows

# -----------------------------------------------------------------------------
# 5. Build deterministic image subset
# -----------------------------------------------------------------------------
run_images_df = images_df.copy()

if "file_name" not in run_images_df.columns:
    run_images_df["file_name"] = run_images_df["image_path"].map(lambda p: os.path.basename(p) if isinstance(p, str) else None)

if "stem" not in run_images_df.columns:
    run_images_df["stem"] = run_images_df["file_name"].map(lambda x: os.path.splitext(x)[0] if isinstance(x, str) else None)

run_images_df = run_images_df.sort_values(["file_name", "image_path"]).reset_index(drop=True)

print(f"Images to process : {len(run_images_df)}")
print(f"Prompts to try    : {len(PROMPTS_TO_TRY)}")
print("Prompt list:")
for p in PROMPTS_TO_TRY:
    print(f"  - {p}")

# -----------------------------------------------------------------------------
# 6. Run SAM3 on all uploaded images
# -----------------------------------------------------------------------------
sam3_crossarm_mask_rows = []
sam3_crossarm_prompt_rows = []

start_all = time.time()
n_total = len(run_images_df)

for i, (_, row) in enumerate(run_images_df.iterrows(), start=1):
    image_path = row["image_path"]
    file_name = row["file_name"] if pd.notna(row["file_name"]) else os.path.basename(image_path)
    stem = row["stem"] if pd.notna(row["stem"]) else Path(file_name).stem
    image_id = row["image_id"] if ("image_id" in row.index and pd.notna(row["image_id"])) else stem

    try:
        # ---------------------------------------------------------------------
        # 6.1 Missing image guard
        # ---------------------------------------------------------------------
        if not isinstance(image_path, str) or not os.path.exists(image_path):
            sam3_crossarm_mask_rows.append({
                "image_id": image_id,
                "file_name": file_name,
                "image_path": image_path,
                "run_status": "failed_missing_image",
                "selected_prompt": None,
                "num_masks_selected_prompt": 0,
                "best_mask_index": np.nan,
                "best_score": np.nan,
                "annotation_box": None,   # no GT annotation in this pipeline
                "pred_box": None,         # SAM3 predicted box
                "crop_box": None,
                "pred_x1": np.nan,
                "pred_y1": np.nan,
                "pred_x2": np.nan,
                "pred_y2": np.nan,
                "crop_x": np.nan,
                "crop_y": np.nan,
                "crop_w": np.nan,
                "crop_h": np.nan,
                "best_mask_path": None,
                "crop_image_path": None,
                "mask_h": np.nan,
                "mask_w": np.nan,
                "image_h": np.nan,
                "image_w": np.nan,
                "crop_image_h": np.nan,
                "crop_image_w": np.nan,
            })
            continue

        # ---------------------------------------------------------------------
        # 6.2 Load image
        # ---------------------------------------------------------------------
        image = load_cached_image(image_path)
        image_w, image_h = image.size

        # ---------------------------------------------------------------------
        # 6.3 Run all prompts and choose best overall
        # ---------------------------------------------------------------------
        best_result, prompt_rows = choose_best_candidate_across_prompts(
            image=image,
            prompts=PROMPTS_TO_TRY,
        )

        for pr in prompt_rows:
            sam3_crossarm_prompt_rows.append({
                "image_id": image_id,
                "file_name": file_name,
                "image_path": image_path,
                **pr,
            })

        # ---------------------------------------------------------------------
        # 6.4 No mask case
        # ---------------------------------------------------------------------
        if best_result is None:
            sam3_crossarm_mask_rows.append({
                "image_id": image_id,
                "file_name": file_name,
                "image_path": image_path,
                "run_status": "no_mask",
                "selected_prompt": None,
                "num_masks_selected_prompt": 0,
                "best_mask_index": np.nan,
                "best_score": np.nan,
                "annotation_box": None,   # no GT annotation in this pipeline
                "pred_box": None,
                "crop_box": None,
                "pred_x1": np.nan,
                "pred_y1": np.nan,
                "pred_x2": np.nan,
                "pred_y2": np.nan,
                "crop_x": np.nan,
                "crop_y": np.nan,
                "crop_w": np.nan,
                "crop_h": np.nan,
                "best_mask_path": None,
                "crop_image_path": None,
                "mask_h": np.nan,
                "mask_w": np.nan,
                "image_h": image_h,
                "image_w": image_w,
                "crop_image_h": np.nan,
                "crop_image_w": np.nan,
            })
            continue

        # ---------------------------------------------------------------------
        # 6.5 Final chosen SAM3 candidate
        # ---------------------------------------------------------------------
        pred_box = best_result["best_box"]
        crop_box = pad_xyxy_box(pred_box, image_w=image_w, image_h=image_h, pad_pixels=PAD_PIXELS)

        pred_x1, pred_y1, pred_x2, pred_y2 = [float(v) for v in pred_box]
        crop_x, crop_y, crop_w, crop_h = xyxy_to_xywh(crop_box)

        best_mask_path = save_best_mask_npz(best_result["best_mask"], image_id=image_id)
        crop_image_path, crop_image_h, crop_image_w = save_crop_png(image, crop_box, image_id=image_id)

        mask_h, mask_w = best_result["best_mask"].shape

        sam3_crossarm_mask_rows.append({
            "image_id": image_id,
            "file_name": file_name,
            "image_path": image_path,
            "run_status": "done",
            "selected_prompt": best_result["prompt_text"],
            "num_masks_selected_prompt": int(best_result["num_masks"]),
            "best_mask_index": int(best_result["best_local_index"]),
            "best_score": float(best_result["best_score"]),
            "annotation_box": None,   # no GT annotation in this pipeline
            "pred_box": pred_box,     # SAM3 predicted box
            "crop_box": crop_box,
            "pred_x1": pred_x1,
            "pred_y1": pred_y1,
            "pred_x2": pred_x2,
            "pred_y2": pred_y2,
            "crop_x": crop_x,
            "crop_y": crop_y,
            "crop_w": crop_w,
            "crop_h": crop_h,
            "best_mask_path": best_mask_path,
            "crop_image_path": crop_image_path,
            "mask_h": mask_h,
            "mask_w": mask_w,
            "image_h": image_h,
            "image_w": image_w,
            "crop_image_h": crop_image_h,
            "crop_image_w": crop_image_w,
        })

    except Exception as e:
        sam3_crossarm_mask_rows.append({
            "image_id": image_id if "image_id" in locals() else None,
            "file_name": file_name if "file_name" in locals() else None,
            "image_path": image_path if "image_path" in locals() else None,
            "run_status": "failed",
            "selected_prompt": None,
            "num_masks_selected_prompt": 0,
            "best_mask_index": np.nan,
            "best_score": np.nan,
            "annotation_box": None,
            "pred_box": None,
            "crop_box": None,
            "pred_x1": np.nan,
            "pred_y1": np.nan,
            "pred_x2": np.nan,
            "pred_y2": np.nan,
            "crop_x": np.nan,
            "crop_y": np.nan,
            "crop_w": np.nan,
            "crop_h": np.nan,
            "best_mask_path": None,
            "crop_image_path": None,
            "mask_h": np.nan,
            "mask_w": np.nan,
            "image_h": np.nan,
            "image_w": np.nan,
            "crop_image_h": np.nan,
            "crop_image_w": np.nan,
            "error_message": str(e),
        })

    # -------------------------------------------------------------------------
    # 6.6 Cleanup
    # -------------------------------------------------------------------------
    if "image" in locals():
        del image
    if "best_result" in locals():
        del best_result
    if "prompt_rows" in locals():
        del prompt_rows

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if i % PRINT_EVERY_N == 0 or i == n_total:
        print(f"[{i}/{n_total}] processed")

# Final output tables
sam3_crossarm_mask_df = pd.DataFrame(sam3_crossarm_mask_rows)
sam3_crossarm_prompt_df = pd.DataFrame(sam3_crossarm_prompt_rows)

# -----------------------------------------------------------------------------
# 7. Summary
# -----------------------------------------------------------------------------
print("\n" + "=" * 80)
print("CELL 9a — SAM3 CROSSARM MASK SUMMARY")
print("=" * 80)
print(f"Elapsed (sec): {time.time() - start_all:.2f}")
print(f"Images processed: {len(sam3_crossarm_mask_df)}")

print("\nRun status counts:")
print(sam3_crossarm_mask_df["run_status"].value_counts(dropna=False).to_string())

if "selected_prompt" in sam3_crossarm_mask_df.columns:
    print("\nSelected prompt counts:")
    print(sam3_crossarm_mask_df["selected_prompt"].value_counts(dropna=False).to_string())

valid_scores = sam3_crossarm_mask_df["best_score"].dropna()
if len(valid_scores) > 0:
    print("\nBest-score summary:")
    print(valid_scores.describe().round(4).to_string())

# -----------------------------------------------------------------------------
# 8. Visual QA gallery
# -----------------------------------------------------------------------------
qa_df = sam3_crossarm_mask_df[sam3_crossarm_mask_df["run_status"] == "done"].copy()
qa_df = qa_df.sort_values("best_score", ascending=False).head(MAX_VISUAL_QA).reset_index(drop=True)

if not qa_df.empty:
    n_show = len(qa_df)
    n_cols = 2
    n_rows = int(math.ceil(n_show / n_cols))

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(14, 5 * n_rows),
        dpi=110,
        constrained_layout=True
    )

    if n_rows == 1 and n_cols == 1:
        axes = np.array([axes])
    axes = np.array(axes).reshape(-1)

    for ax in axes[n_show:]:
        ax.axis("off")

    for ax, (_, row) in zip(axes, qa_df.iterrows()):
        image = load_cached_image(row["image_path"])
        best_mask = load_best_mask_npz(row["best_mask_path"])

        overlay_mask(ax, image, best_mask, alpha=MASK_ALPHA)

        if SHOW_PRED_BOX and isinstance(row["pred_box"], list):
            draw_xyxy_box(ax, row["pred_box"], color="yellow", label="sam3_box")

        if SHOW_CROP_BOX and isinstance(row["crop_box"], list):
            draw_xyxy_box(ax, row["crop_box"], color="magenta", label="crop_box", linewidth=1)

        ax.set_title(
            f"{row['file_name']}\n"
            f"prompt={row['selected_prompt']} | score={row['best_score']:.3f}",
            fontsize=10,
            pad=8,
        )

    fig.suptitle("SAM3 visual QA — uploaded images", fontsize=16)
    plt.show()

# -----------------------------------------------------------------------------
# 9. Preview tables
# -----------------------------------------------------------------------------
print("\nImage-level preview:")
display(
    sam3_crossarm_mask_df[
        [
            "image_id",
            "file_name",
            "run_status",
            "selected_prompt",
            "num_masks_selected_prompt",
            "best_mask_index",
            "best_score",
            "pred_box",
            "crop_box",
            "best_mask_path",
            "crop_image_path",
        ]
    ].head(20)
)

print("\nPrompt-level preview:")
display(
    sam3_crossarm_prompt_df[
        [
            "image_id",
            "file_name",
            "prompt_text",
            "prompt_status",
            "num_masks",
            "best_score_for_prompt",
            "best_box_for_prompt",
        ]
    ].head(30)
)

# -----------------------------------------------------------------------------
# 10. Save state
# -----------------------------------------------------------------------------
# Convert list/object box columns to JSON strings for safer parquet saving.
sam3_crossarm_mask_save_df = sam3_crossarm_mask_df.copy()
for col in ["annotation_box", "pred_box", "crop_box"]:
    if col in sam3_crossarm_mask_save_df.columns:
        sam3_crossarm_mask_save_df[col] = sam3_crossarm_mask_save_df[col].map(
            lambda x: json.dumps(x) if isinstance(x, list) else None
        )

sam3_crossarm_prompt_save_df = sam3_crossarm_prompt_df.copy()
for col in ["best_box_for_prompt"]:
    if col in sam3_crossarm_prompt_save_df.columns:
        sam3_crossarm_prompt_save_df[col] = sam3_crossarm_prompt_save_df[col].map(
            lambda x: json.dumps(x) if isinstance(x, list) else None
        )

globals()["sam3_crossarm_mask_df"] = sam3_crossarm_mask_df.copy()
globals()["sam3_crossarm_prompt_df"] = sam3_crossarm_prompt_df.copy()
globals()["sam3_crossarm_mask_save_df"] = sam3_crossarm_mask_save_df.copy()
globals()["sam3_crossarm_prompt_save_df"] = sam3_crossarm_prompt_save_df.copy()

if "save_state" in globals():
    save_state(
        df_names=[
            "images_df",
            "sam3_crossarm_mask_save_df",
            "sam3_crossarm_prompt_save_df",
        ],
        config_extra={
            "SAM3_CROSSARM_RUN_NAME": SAM3_CROSSARM_RUN_NAME,
            "SAM3_CROSSARM_PROMPTS": " | ".join(PROMPTS_TO_TRY),
            "SAM3_CROSSARM_TEXT_THRESHOLD": float(TEXT_THRESHOLD),
            "SAM3_CROSSARM_MASK_THRESHOLD": float(MASK_THRESHOLD),
            "SAM3_CROSSARM_PAD_PIXELS": int(PAD_PIXELS),
            "SAM3_CROSSARM_MASK_DIR": SAM3_CROSSARM_MASK_DIR,
            "SAM3_CROSSARM_CROP_DIR": SAM3_CROSSARM_CROP_DIR,
            "SAM3_CROSSARM_DEBUG_DIR": SAM3_CROSSARM_DEBUG_DIR,
            "SAM3_CROSSARM_IMAGES_PROCESSED": int(len(sam3_crossarm_mask_df)),
        }
    )